# Macaron food photo model (v9)

This notebook trains the small model that lets the Macaron app recognise a dish from a photo, on the phone, with no internet.

**What it does**

1. Downloads two public food photo datasets.
2. Fine-tunes a small pretrained network (MobileNetV4, medium) to tell the dishes apart.
3. Also learns the parts of a meal: the base (rice or noodle type) and the proteins (chicken, fish, egg, tofu...).
4. Saves three files for the app: `food.onnx` (the model), `labels.json` (the dish names) and `parts.json` (the part names).

**How to run it on Kaggle**

1. Go to kaggle.com, sign in, and choose **Create** then **New Notebook**.
2. **File** then **Import Notebook**, and upload this file.
3. In the right-hand panel, open **Session options**. Set **Accelerator** to **GPU T4 x2** and **Internet** to **On**.
4. Press **Run All** at the top. This version learns 1,017 dishes and collects photos for about 615 new ones, so expect several hours (my estimate is 5 to 8; Kaggle allows 12 per session). Keep the tab open or use **Save Version** then **Save & Run All** so it runs in the background.
5. When it finishes, open the **Output** section on the right. Download `food.onnx`, `labels.json` and `parts.json`.
6. Put the three files in the `model` folder of the app's repository.

The model learns 1,017 dish looks, which between them cover every food in the app (2,375 at the time of writing): shop versions and look-alikes share one look. The lists are in the next cell. Each entry links a dataset class to the exact dish name used in the app.

In [ ]:
!pip -q install -U timm datasets onnx onnxruntime open_clip_torch
!pip -q install ddgs


In [ ]:
import json, random, numpy as np, torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms as T
from datasets import load_dataset
from PIL import Image

SEED = 7
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on", DEVICE)

# dataset class -> exact dish name in the app's food list
MAP_MY = {
"fish_and_chips": "Fish and chips",
"fried_noodles": "Mee goreng mamak",
"fried_rice": "Nasi goreng kampung",
"hamburger": "Cheeseburger",
"kaya_toast": "Kaya toast set",
"laksa": "Curry laksa",
"mixed_rice": "Economy rice: 1 meat, 2 veg",
"nasi_lemak": "Nasi lemak biasa",
"popiah": "Popiah, fresh",
"roti_canai": "Roti canai kosong",
"satay": "Satay ayam"
}
MAP_F101 = {
"sushi": "Sushi plates",
"sashimi": "Salmon sashimi and rice",
"ramen": "Ramen, shoyu",
"bibimbap": "Bibimbap",
"pad_thai": "Pad thai",
"pizza": "Pizza",
"spaghetti_bolognese": "Spaghetti bolognese",
"spaghetti_carbonara": "Spaghetti carbonara",
"caesar_salad": "Chicken caesar salad",
"french_fries": "French fries",
"ice_cream": "Ice cream",
"chocolate_cake": "Chocolate cake",
"cheesecake": "Cheesecake",
"donuts": "Doughnut",
"hummus": "Hummus and falafel plate",
"falafel": "Hummus and falafel plate",
"edamame": "Edamame",
"miso_soup": "Miso soup",
"chicken_wings": "Fried chicken",
"dumplings": "Har gow",
"omelette": "Omelette with rice and cucumber",
"grilled_salmon": "Pan-seared salmon, rice and greens",
"gyoza": "Gyoza",
"takoyaki": "Takoyaki",
"pho": "Pho",
"spring_rolls": "Spring rolls, fried",
"chicken_curry": "Chicken curry with rice",
"peking_duck": "Roast duck rice",
"waffles": "Waffle with syrup",
"pancakes": "Pancakes with syrup",
"french_toast": "French toast",
"club_sandwich": "Club sandwich",
"steak": "Steak with vegetables",
"hot_dog": "Hot dog",
"tiramisu": "Tiramisu",
"lasagna": "Lasagna",
"macaroni_and_cheese": "Macaroni and cheese",
"garlic_bread": "Garlic bread",
"samosa": "Samosa",
"cup_cakes": "Cupcake",
"frozen_yogurt": "Frozen yogurt",
"tacos": "Tacos",
"macarons": "Macarons",
"fried_calamari": "Fried calamari",
"greek_salad": "Garden salad",
"eggs_benedict": "Eggs Benedict",
"apple_pie": "Apple pie"
}
MAP_IN = {
"chapati": "Chapati with dhal",
"idli": "Idli with sambar",
"masala_dosa": "Masala thosai",
"chai": "Teh tarik",
"samosa": "Samosa"
}
PER_CLASS = 500      # cap on training photos per class, to keep classes balanced
EPOCHS_HEAD, EPOCHS_ALL = 2, 14
# A stronger small network than MobileNetV3 (79.9% vs 75.8% on ImageNet), still fast enough on a phone.
ARCH = "mobilenetv4_conv_medium.e500_r256_in1k"
# Web photos are noisy (menus, logos, people). A CLIP model drops the ones that are not a photo of food.
CLEAN_WEB = True
BATCH = 64

# ---- v9: what the photo is made of ----
# Next to the dish, the model also says which base (rice or noodle type) and which proteins it sees,
# so the app can still help when the exact dish is unknown (an economy rice plate, a bowl of plain bee hoon).
PART_NAMES = ["white rice", "fried rice", "coconut rice", "chicken rice", "spiced rice", "brown rice", "porridge", "yellow noodles", "egg noodles", "bee hoon", "kway teow", "flour noodles", "ramen", "instant noodles", "glass noodles", "mee sua", "pasta", "bread", "flatbread", "chicken", "pork", "beef", "mutton", "duck", "fish", "prawn", "squid", "egg", "tofu", "fish ball", "anchovies", "vegetables"]
TAGS_OF = {
"6-inch Veggie Delite with egg": [
"egg"
],
"6-inch chicken teriyaki sub": [
"bread",
"chicken"
],
"6-inch roasted chicken sub": [
"bread",
"chicken"
],
"6-inch tuna sub": [
"bread",
"fish"
],
"7-Eleven Slurpee, regular": [],
"7-Eleven chicken pau": [
"bread",
"chicken"
],
"7-Eleven chicken slice sandwich": [
"bread",
"chicken"
],
"7-Eleven egg mayo sandwich": [
"bread",
"egg"
],
"7-Eleven fried rice": [
"fried rice"
],
"7-Eleven mashed potato with gravy": [],
"7-Eleven nasi lemak": [
"coconut rice",
"anchovies"
],
"7-Eleven onigiri, chicken": [
"chicken"
],
"7-Eleven onigiri, tuna mayo": [
"fish"
],
"7-Eleven tuna sandwich": [
"bread",
"fish"
],
"ABC soup with chicken and rice": [
"white rice",
"chicken"
],
"ABC soup with pork ribs": [
"pork"
],
"Aglio olio with prawns": [
"pasta",
"prawn"
],
"Air katira": [],
"Air mata kucing": [],
"Air-fryer tofu with broccoli and brown rice": [
"brown rice",
"tofu",
"vegetables"
],
"Ais kacang (ABC)": [],
"Ajitama (ramen egg)": [
"ramen",
"egg"
],
"Al Halabi hummus with bread": [
"bread"
],
"All Berry Bang smoothie": [],
"Almond croissant": [
"bread"
],
"Almond paste": [],
"Amazonas Latin sharing plates, per person": [],
"Americano": [],
"Anlene Gold 5X": [],
"Apam balik": [
"flatbread"
],
"Apple": [],
"Apple pie": [],
"Army stew (budae jjigae)": [
"white rice"
],
"Asam laksa": [
"bee hoon"
],
"Asam pedas fish, 1 piece": [
"fish"
],
"Avocado": [],
"Avocado toast with egg": [
"bread",
"egg"
],
"Ayam Goreng McD": [
"chicken"
],
"Ayam goreng berempah": [
"chicken"
],
"Ayam goreng berempah, air fryer": [
"chicken"
],
"Ayam goreng kunyit": [
"chicken"
],
"Ayam masak kicap with rice and sayur": [
"white rice",
"chicken",
"vegetables"
],
"Ayam masak merah with rice": [
"white rice",
"chicken"
],
"Ayam percik with rice": [
"white rice",
"chicken"
],
"BBQ chicken wings": [
"chicken"
],
"BBQ pork ribs": [
"pork"
],
"BLT sandwich": [
"bread"
],
"Bagel with cream cheese": [
"bread"
],
"Bagel, plain": [
"bread"
],
"Bak chang": [
"white rice"
],
"Bak chor mee": [
"egg noodles",
"pork"
],
"Bak kut teh with rice": [
"white rice",
"pork"
],
"Baked cheese pasta": [
"pasta"
],
"Baked cheese tart": [],
"Baked chicken thigh with potatoes and salad": [
"chicken",
"vegetables"
],
"Baked pork chop rice": [
"white rice",
"pork"
],
"Baked salmon with sweet potato and greens": [
"fish",
"vegetables"
],
"Banana": [],
"Banana bread": [
"bread"
],
"Banana leaf rice (Kanna Curry House)": [
"white rice",
"chicken",
"vegetables"
],
"Banana leaf rice (Sri Nirwana Maju)": [
"white rice",
"fish",
"vegetables"
],
"Banana leaf rice with chicken": [
"white rice",
"chicken"
],
"Banana leaf rice, vegetarian": [
"white rice",
"vegetables"
],
"Banana pudding": [],
"Banh mi": [
"bread"
],
"Banoffee pie": [],
"Barley drink, sweetened": [],
"Barley, fu chuk and gingko": [],
"Basmati rice, plain": [
"white rice"
],
"Baumkuchen": [],
"Bean sprouts stir-fry": [
"vegetables"
],
"Beancurd skin roll, fried": [
"tofu"
],
"Beancurd skin roll, steamed": [
"tofu"
],
"Beard Papa original cream puff": [],
"Bee Cheng Hiang bak kwa, pork": [
"pork"
],
"Bee hoon goreng": [
"fried rice",
"bee hoon",
"chicken",
"egg",
"vegetables"
],
"Bee hoon soup / Mee hoon soup": [
"bee hoon",
"fish ball",
"vegetables"
],
"Bee hoon soup with fish balls and greens": [
"bee hoon",
"fish ball",
"vegetables"
],
"Beef ball noodle soup": [
"bee hoon",
"beef"
],
"Beef ball noodles, dry": [
"beef"
],
"Beef burger with fries": [
"bread",
"beef"
],
"Beef noodle soup": [
"egg noodles",
"beef"
],
"Beef noodles (Soong Kee)": [
"egg noodles",
"beef"
],
"Bibimbap": [
"white rice"
],
"Big Mac": [
"bread"
],
"Big bao (dai bao)": [
"bread",
"chicken",
"egg"
],
"Big breakfast": [
"bread",
"pork",
"egg"
],
"Bitter gourd soup with lean pork": [
"pork"
],
"Bitter gourd with egg": [
"egg"
],
"Black bean fish": [
"fish"
],
"Black bean soup with chicken": [
"chicken"
],
"Black pepper chicken": [
"chicken"
],
"Black pepper pork": [
"pork"
],
"Black sesame paste": [],
"Blanched greens with oyster sauce": [
"vegetables"
],
"Blondie": [],
"Blueberries": [],
"Blueberry bagel with cream cheese": [
"bread"
],
"Blueberry muffin": [],
"Boat noodles": [],
"Boiled egg": [
"egg"
],
"Braised cabbage with glass noodles": [
"glass noodles",
"vegetables"
],
"Braised chicken feet with mushrooms": [
"chicken"
],
"Braised chicken with potatoes": [
"chicken"
],
"Braised chicken with potatoes and rice": [
"white rice",
"chicken"
],
"Braised mushrooms with tofu": [
"tofu"
],
"Braised pork belly (tau yu bak)": [
"pork"
],
"Braised pork with yam": [
"pork"
],
"Braised tofu (economy rice)": [
"white rice",
"tofu"
],
"Brinjal with sambal": [
"vegetables"
],
"Briyani rice": [
"spiced rice"
],
"Broccoli and cauliflower soup": [
"vegetables"
],
"Brown rice": [
"brown rice"
],
"Brown rice (economy rice)": [
"white rice",
"brown rice"
],
"Brown sugar pearl fresh milk": [],
"Brownie": [],
"Bubble milk tea": [],
"Bubur Ayam McD": [
"porridge",
"chicken"
],
"Bubur cha cha": [],
"Bubur lambuk": [
"porridge",
"mutton"
],
"Bubur pulut hitam": [],
"Build-your-own salad bowl": [
"chicken",
"egg",
"vegetables"
],
"Buldak carbonara ramen": [
"ramen",
"instant noodles",
"pasta"
],
"Buldak hot chicken ramen": [
"ramen",
"instant noodles",
"chicken"
],
"Bulgogi with rice": [
"white rice",
"beef"
],
"Bun bo Hue": [
"bee hoon",
"bread"
],
"Burger (myBurgerLab)": [
"bread"
],
"Burnt cheesecake": [],
"Butter cake": [],
"Butter chicken with naan": [
"flatbread",
"chicken"
],
"Butter croissant": [
"bread"
],
"Butter garlic prawns with rice": [
"white rice",
"prawn"
],
"Butter prawns": [
"prawn"
],
"Butter toast": [
"bread"
],
"Cadbury Dairy Milk, whole bar": [],
"Caesar salad, no chicken": [
"chicken",
"vegetables"
],
"Calbee Jagabee": [],
"California roll": [],
"Canele": [],
"Cantonese fried noodles (yee mee)": [
"egg noodles",
"pork",
"prawn",
"egg",
"vegetables"
],
"Caramel frappé": [],
"Carrot cake with cream cheese": [],
"Castella cake": [],
"Cauliflower fried rice with egg and chicken": [
"fried rice",
"chicken",
"egg"
],
"Cauliflower rice": [
"white rice"
],
"Cempedak": [],
"Cendol": [],
"Cendol cake": [],
"Century egg": [
"egg"
],
"Century egg and pork porridge": [
"porridge",
"pork",
"egg"
],
"Cereal prawns": [
"prawn"
],
"Chai boey (mustard greens stew)": [
"vegetables"
],
"Cham ais": [
"pork"
],
"Chapati with dhal": [
"flatbread"
],
"Char kway teow / Kway teow goreng": [
"bee hoon",
"kway teow",
"prawn",
"egg",
"vegetables"
],
"Char siu bao": [
"bread",
"pork"
],
"Char siu cheong fun": [
"pork"
],
"Char siu rice": [
"chicken rice",
"pork"
],
"Char siu sou": [
"pork"
],
"Chawanmushi": [],
"Chayote and carrot soup with chicken": [
"chicken"
],
"Chee cheong fun (SS2 morning market)": [],
"Chee cheong fun, plain": [],
"Cheese bun": [
"bread"
],
"Cheese foam tea": [],
"Cheese naan": [
"flatbread"
],
"Cheese ramyeon": [
"ramen"
],
"Cheeseburger": [
"bread"
],
"Cheesecake": [],
"Cheezy wedges, large": [],
"Cherries": [],
"Cherry tomatoes": [],
"Chicken and vegetable clear soup with konjac rice": [
"white rice",
"chicken",
"vegetables"
],
"Chicken and vegetable pasta": [
"pasta",
"chicken",
"vegetables"
],
"Chicken and vegetable soup with bread": [
"bread",
"chicken",
"vegetables"
],
"Chicken bao": [
"bread",
"chicken"
],
"Chicken breast salad bowl": [
"chicken",
"vegetables"
],
"Chicken breast salad wrap": [
"flatbread",
"chicken",
"vegetables"
],
"Chicken burger, crispy": [
"bread",
"chicken"
],
"Chicken caesar salad": [
"chicken",
"vegetables"
],
"Chicken chop (Steven's Corner)": [
"chicken",
"egg"
],
"Chicken chop with black pepper sauce": [
"chicken"
],
"Chicken curry with potatoes and rice": [
"white rice",
"chicken"
],
"Chicken curry with rice": [
"white rice",
"chicken"
],
"Chicken feet in black bean sauce": [
"chicken"
],
"Chicken floss bun": [
"bread",
"chicken"
],
"Chicken karaage": [
"chicken"
],
"Chicken katsu curry rice": [
"white rice",
"chicken"
],
"Chicken kebab plate with salad": [
"chicken",
"vegetables"
],
"Chicken noodle soup": [
"chicken",
"vegetables"
],
"Chicken noodles, dry": [
"egg noodles",
"chicken"
],
"Chicken nuggets": [
"chicken"
],
"Chicken pie": [
"chicken"
],
"Chicken pita": [
"chicken"
],
"Chicken porridge": [
"porridge",
"chicken"
],
"Chicken porridge with ginger": [
"porridge",
"chicken"
],
"Chicken quesadilla": [
"flatbread",
"chicken"
],
"Chicken rice, roasted": [
"chicken rice",
"chicken"
],
"Chicken rice, steamed": [
"chicken rice",
"chicken"
],
"Chicken sandwich, grilled": [
"bread",
"chicken"
],
"Chicken sausage roll": [
"chicken",
"pork"
],
"Chicken shawarma wrap": [
"flatbread",
"chicken"
],
"Chicken siew mai": [
"chicken"
],
"Chicken tenders": [
"chicken"
],
"Chicken teriyaki don": [
"white rice",
"chicken"
],
"Chicken wing skewer": [
"chicken"
],
"Chicken wings, fried": [
"chicken"
],
"Chickpea and cucumber salad": [
"vegetables"
],
"Chilli pan mee": [
"flour noodles"
],
"Chinese cabbage and fish ball soup": [
"fish ball",
"vegetables"
],
"Chinese pear": [],
"Chipsmore Original": [],
"Chirashi don": [
"white rice"
],
"Chobani Fit, strawberry": [],
"Chocolate bar": [],
"Chocolate bingsu": [],
"Chocolate cake": [],
"Chocolate chip cookie": [],
"Chocolate eclair": [],
"Chocolate milkshake": [],
"Chocolate muffin": [],
"Chocolate sundae": [],
"Chocolate éclair": [],
"Churros with chocolate dip": [],
"Ciku (sapodilla)": [],
"Cinnabon Classic Roll": [],
"Cinnamon roll": [],
"Claypot chicken rice": [
"chicken rice",
"chicken"
],
"Claypot loh shu fun": [
"bee hoon",
"kway teow",
"pork",
"egg"
],
"Claypot yee mee": [
"egg noodles"
],
"Cloud 9 rice bowl": [
"white rice"
],
"Club sandwich": [
"bread"
],
"Coconut shake": [],
"Cold soba": [],
"Coleslaw": [],
"Coney dog": [],
"Corn and carrot soup with chicken": [
"chicken"
],
"Corn cup": [],
"Cottage cheese toast with tomato": [
"bread"
],
"Cream puff": [],
"Creamy mushroom chicken pasta": [
"pasta",
"chicken"
],
"Crispy chicken with rice": [
"white rice",
"chicken"
],
"Croffle": [
"bread"
],
"Cromboloni": [
"bread"
],
"Crookie": [
"bread"
],
"Crêpe with banana and chocolate": [],
"Cup noodles": [
"instant noodles"
],
"Cupcake": [],
"Curry chicken (economy rice)": [
"white rice",
"chicken"
],
"Curry chicken bun": [
"bread",
"chicken"
],
"Curry fish (economy rice)": [
"white rice",
"fish"
],
"Curry gravy, add-on": [],
"Curry laksa": [
"bee hoon"
],
"Curry mee": [
"yellow noodles",
"bee hoon",
"chicken",
"tofu",
"vegetables"
],
"Curry puff": [],
"Curry squid": [
"squid"
],
"Curry vegetables (sayur lodeh)": [
"vegetables"
],
"Cuura tasting dish, per person": [],
"D Empire Chinese set, per person": [
"white rice",
"vegetables"
],
"Dak-kkochi (chicken skewer)": [
"chicken"
],
"Dan dan noodles": [
"yellow noodles",
"pork"
],
"Dates": [],
"De Irrfan's laksa Johor": [
"bee hoon"
],
"Dhal curry": [],
"Dhal curry with vegetables and rice": [
"white rice",
"vegetables"
],
"Dim sum breakfast": [
"prawn"
],
"Ding Dang": [],
"Dorayaki": [],
"Doritos Nacho Cheese": [],
"Double beef burger": [
"bread",
"beef"
],
"Double cheeseburger": [
"bread"
],
"Double-skin milk pudding": [],
"Doughnut": [],
"Dragon fruit, red": [],
"Dragon fruit, white": [],
"Dried mango": [],
"Dubai chewy cookie": [],
"Dubai chocolate bar": [],
"Dubai chocolate strawberry cup": [],
"Duck noodle soup": [
"duck"
],
"Duck noodles, dry": [
"egg noodles",
"duck"
],
"Duku langsat": [],
"Dumplings in soup with greens": [
"vegetables"
],
"Durian cendol": [],
"Durian crepe": [],
"Economy bee hoon": [
"bee hoon",
"egg"
],
"Economy rice (Home Kitchen)": [
"white rice"
],
"Economy rice: 1 meat, 2 veg": [
"white rice"
],
"Economy rice: steamed fish, tofu, greens": [
"white rice",
"fish",
"tofu",
"vegetables"
],
"Edamame": [],
"Egg McMuffin": [
"egg"
],
"Egg and cheese bagel": [
"bread",
"egg"
],
"Egg and tomato stir-fry with rice": [
"white rice",
"egg"
],
"Egg foo yong (omelette)": [
"egg"
],
"Egg fried rice with vegetables": [
"fried rice",
"egg",
"vegetables"
],
"Egg mayo sandwich": [
"bread",
"egg"
],
"Egg sandwich": [
"bread",
"egg"
],
"Egg tart": [
"egg"
],
"Egg white omelette with spinach and mushrooms": [
"egg",
"vegetables"
],
"Eggs Benedict": [
"egg"
],
"Einspänner cream coffee": [],
"Ensure Gold": [],
"Eomuk (fish cake skewer)": [
"fish ball"
],
"Extra chashu": [
"pork"
],
"FamilyMart chicken katsu bento": [
"white rice",
"chicken"
],
"FamilyMart chicken katsu sando": [
"bread",
"chicken"
],
"FamilyMart egg mayo sando": [
"bread",
"egg"
],
"FamilyMart mac and cheese": [
"pasta"
],
"FamilyMart nasi lemak": [
"coconut rice",
"anchovies"
],
"FamilyMart oden, chikuwa": [
"fish ball"
],
"FamilyMart oden, crab stick": [
"fish ball"
],
"FamilyMart oden, egg": [
"egg",
"fish ball"
],
"FamilyMart oden, fish ball stick": [
"fish ball"
],
"FamilyMart oden, fish cake": [
"fish ball"
],
"FamilyMart oden, radish": [
"fish ball"
],
"FamilyMart oden, tofu puff": [
"tofu",
"fish ball"
],
"FamilyMart oden, udon noodles": [
"flour noodles",
"fish ball"
],
"FamilyMart onigiri, chicken teriyaki": [
"chicken"
],
"FamilyMart onigiri, salmon": [
"fish"
],
"FamilyMart onigiri, spicy tuna": [
"fish"
],
"FamilyMart onigiri, tuna mayo": [
"fish"
],
"FamilyMart tuna mayo sando": [
"bread",
"fish"
],
"Fettuccine carbonara with mushroom": [
"pasta"
],
"Fig, fresh": [],
"Filet-O-Fish": [
"bread",
"fish"
],
"Financier": [],
"Fish and chips": [
"fish"
],
"Fish ball noodle soup": [
"fish ball",
"vegetables"
],
"Fish burger": [
"bread",
"fish"
],
"Fish curry with rice": [
"white rice",
"fish"
],
"Fish head mee hoon, clear soup": [
"bee hoon",
"fish"
],
"Fish head noodle, milky soup": [
"bee hoon",
"fish",
"vegetables"
],
"Fish porridge": [
"porridge",
"fish"
],
"Fish slice noodle soup": [
"bee hoon",
"fish"
],
"Fish slice soup with tofu and tomato": [
"white rice",
"fish",
"tofu"
],
"Flavoured yogurt cup": [],
"French beans with minced pork": [
"pork"
],
"French fries": [],
"French fries, large": [],
"French toast": [
"bread"
],
"Fresh Delights fruit cup": [],
"Fresh coconut water": [],
"Fresh orange juice, no sugar": [],
"Fried calamari": [
"squid"
],
"Fried chicken": [
"chicken"
],
"Fried chicken wing (economy rice)": [
"white rice",
"chicken"
],
"Fried chicken with curry rice (Lim Fried Chicken)": [
"white rice",
"chicken"
],
"Fried ebi roll": [
"prawn"
],
"Fried egg": [
"egg"
],
"Fried fish cake": [
"fish ball"
],
"Fried fish fillet": [
"fish"
],
"Fried mantou with condensed milk": [],
"Fried pork cutlet (economy rice)": [
"white rice",
"pork"
],
"Fried prawn dumplings with mayo": [
"prawn"
],
"Fried tofu with sweet chilli": [
"tofu"
],
"Fried wantan": [],
"Frozen yogurt": [],
"Fruit Danish": [],
"Fruit sando": [
"bread"
],
"Fruit tart": [],
"Fruit tea, less sugar": [],
"Garden salad": [
"vegetables"
],
"Garlic bread": [
"bread"
],
"Garlic fried rice": [
"fried rice"
],
"Garlic naan": [
"flatbread"
],
"Garlic toast": [
"bread"
],
"Garrett popcorn, Chicago mix": [],
"Ginger and onion pork": [
"pork"
],
"Ginger and spring onion chicken": [
"chicken"
],
"Glass noodle stir-fry": [
"glass noodles",
"egg",
"vegetables"
],
"Granola bar": [],
"Grapefruit": [],
"Grapes, green seedless": [],
"Grapes, purple seedless": [],
"Grapes, red": [],
"Greek salad": [
"vegetables"
],
"Greek yogurt bowl with berries and granola": [],
"Greek yogurt with fruit and oats": [],
"Green apple": [],
"Green bean soup": [],
"Green curry chicken with rice": [
"white rice",
"chicken"
],
"Green papaya fish soup": [
"fish"
],
"Grilled cheese sandwich": [
"bread"
],
"Grilled chicken breast": [
"chicken"
],
"Grilled chicken burger (GCB)": [
"bread",
"chicken"
],
"Grilled chicken chop with salad": [
"chicken",
"vegetables"
],
"Grilled chicken salad": [
"chicken",
"vegetables"
],
"Grilled chicken wings (BBQ)": [
"chicken"
],
"Grilled chicken wrap": [
"flatbread",
"chicken"
],
"Grilled corn skewer": [],
"Grilled fish with salad and sweet potato": [
"fish",
"vegetables"
],
"Grilled fish with vegetables": [
"fish",
"vegetables"
],
"Grilled lean beef with asparagus and mash": [
"beef"
],
"Grilled mantou skewer": [],
"Guava": [],
"Gui ling gao herbal jelly": [],
"Gummy bears": [],
"Gym Junkie protein smoothie": [],
"Gyoza": [],
"Gyudon": [
"flour noodles",
"beef"
],
"Gyudon, mini with salad": [
"flour noodles",
"beef",
"vegetables"
],
"Hacks": [],
"Haidilao sauce bowl, sesame and garlic": [],
"Hailam mee (Yut Kee)": [
"yellow noodles"
],
"Hainan chicken rice (Hometown Hainan Coffee)": [
"chicken rice",
"chicken"
],
"Hainan toast with kaya, and eggs": [
"bread",
"egg"
],
"Hainanese chicken chop": [
"chicken"
],
"Half-boiled eggs": [
"egg"
],
"Half-boiled eggs with wholemeal toast": [
"bread",
"egg"
],
"Ham and cheese sandwich": [
"bread",
"pork"
],
"Ham sui kok": [
"pork"
],
"Har gow": [
"prawn"
],
"Haw flakes": [],
"Hello Panda Strawberry": [],
"Herbal chicken soup": [
"chicken"
],
"Hershey's Kisses": [],
"Hokkien mee (KL)": [
"yellow noodles"
],
"Honey butter biscuit": [],
"Honeydew": [],
"Honeydew sago": [],
"Horlicks ais": [],
"Hot dog": [
"bread"
],
"Hot plate tofu": [
"tofu"
],
"Hot pot, typical meal": [],
"Hummus and falafel plate": [
"vegetables"
],
"Ice cream": [],
"Ice cream cone, packaged": [],
"Iced chocolate": [],
"Iced lemon tea, packet": [],
"Idli with sambar": [],
"Ikan bakar with rice and ulam": [
"white rice",
"fish"
],
"Ikan pari bakar with rice": [
"white rice",
"fish"
],
"Inari sushi": [
"fish"
],
"Indomie mi goreng": [
"instant noodles"
],
"Injeolmi bingsu": [],
"Inside Scoop, double scoop in a waffle cone": [],
"Inside Scoop, single scoop": [],
"Instant noodles, curry (Maggi)": [
"instant noodles"
],
"Ipoh hor fun": [
"kway teow"
],
"Jackfruit (nangka)": [],
"Jagung cheese": [],
"Jajangmyeon": [],
"Jambu air (rose apple)": [],
"Japanese curry rice, beef": [
"white rice",
"beef"
],
"Japanese curry rice, chicken": [
"white rice",
"chicken"
],
"Japanese rice": [
"white rice"
],
"Japanese set (Hana Dining)": [
"fish"
],
"Japchae": [
"glass noodles"
],
"Jjapaguri (ram-don)": [
"white rice"
],
"Julie's Cheese Sandwich": [
"bread"
],
"Julie's Peanut Butter Sandwich": [
"bread"
],
"KFC Hot & Spicy, breast": [],
"KFC Original Recipe, breast": [],
"KFC Original Recipe, drumstick": [
"chicken"
],
"KFC Original Recipe, thigh": [],
"KFC Original Recipe, wing": [
"chicken"
],
"KFC nuggets": [],
"KK Mart nasi lemak bungkus": [
"coconut rice",
"anchovies"
],
"Kacang putih mix": [],
"Kangkung belacan": [
"vegetables"
],
"Kangkung belacan with tofu and rice": [
"white rice",
"tofu",
"vegetables"
],
"Kappa maki (cucumber)": [],
"Karaage don (Hungry Don)": [
"white rice",
"chicken"
],
"Kari kepala ikan": [
"white rice",
"fish"
],
"Kaya bun": [
"bread"
],
"Kaya puff": [
"bread"
],
"Kaya toast set": [
"bread",
"egg"
],
"Kek batik": [],
"Kek lapis Sarawak": [],
"Kek lapis legit": [],
"Keropok lekor": [],
"Kim Gary macaroni soup with ham": [
"pasta",
"pork"
],
"Kimbap": [],
"Kimchi fried rice with egg": [
"fried rice",
"egg",
"vegetables"
],
"Kimchi fried rice with egg, home-style": [
"fried rice",
"egg",
"vegetables"
],
"Kimchi jjigae with rice": [
"white rice",
"vegetables"
],
"Kimchi ramyeon": [
"ramen",
"vegetables"
],
"Kimchi tofu stew, light": [
"tofu",
"vegetables"
],
"King oyster mushroom skewer": [],
"Kita vegetarian rice bowl": [
"white rice"
],
"Kiwi": [],
"Koala's March Strawberry": [],
"Kofta kebab, 1 skewer": [
"mutton"
],
"Kofta plate with rice": [
"white rice"
],
"Kolo mee / Kon lo mee": [
"egg noodles",
"pork",
"egg"
],
"Konjac noodle soup with prawns and greens": [
"prawn",
"vegetables"
],
"Kopi C": [],
"Kopi ais": [],
"Korean BBQ, typical meal": [
"white rice",
"pork"
],
"Korean army stew (budae jjigae) at home": [],
"Korean fried chicken, soy garlic": [
"chicken"
],
"Korean fried chicken, yangnyeom": [
"chicken"
],
"Korean garlic cream cheese bun": [
"bread"
],
"Kuih, 2 pieces": [],
"Kunafa": [],
"Kung pao chicken": [
"chicken"
],
"Kung pao chicken with rice": [
"white rice",
"chicken"
],
"Kway teow soup": [
"bee hoon",
"kway teow",
"pork",
"fish ball",
"vegetables"
],
"Lady's fingers with sambal": [],
"Laksam": [
"bee hoon"
],
"Lamb chop with fries": [
"mutton"
],
"Lamb skewers (BBQ)": [
"mutton"
],
"Lanzhou beef lamian": [
"yellow noodles",
"beef"
],
"Las Meninas tapas, per person": [],
"Lasagna": [
"pasta"
],
"Latte": [],
"Latte, iced": [],
"Lay's Classic": [],
"Lay's Hot & Spicy": [],
"Lay's Nori Seaweed": [],
"Lay's Sour Cream & Onion": [],
"Lemon chicken": [
"chicken"
],
"Lemon tart": [],
"Leng chee kang": [],
"Lentil soup with pita": [],
"Lettuce with oyster sauce": [
"vegetables"
],
"Lettuce wraps with minced chicken": [
"chicken",
"vegetables"
],
"Lexus Chocolate Cream Sandwich": [
"bread"
],
"Limau ais kurang manis": [],
"Lindt Lindor": [],
"Lo bak go, pan-fried radish cake": [],
"Lok lok": [],
"Long beans with egg": [
"egg",
"vegetables"
],
"Longan": [],
"Longan and sea coconut": [],
"Lontong": [],
"Lor mai kai": [],
"Lor mee / Loh mee": [
"yellow noodles"
],
"Lotus paste bun": [
"bread"
],
"Lotus root and peanut soup with pork ribs": [
"pork"
],
"Lotus root soup, vegetarian": [],
"Low-fat milk": [],
"Luffa and egg soup": [
"egg"
],
"Luncheon meat slices": [
"pork"
],
"Lychee": [],
"Ma lai gao": [],
"Mac and cheese with chicken": [
"pasta",
"chicken"
],
"Macaroni and cheese": [
"pasta"
],
"Macarons": [],
"Madeleine": [],
"Maggi goreng": [
"instant noodles"
],
"Mala noodles": [
"yellow noodles",
"vegetables"
],
"Mala skewers (chuan chuan)": [],
"Mandarin orange": [],
"Mango": [],
"Mango Magic smoothie": [],
"Mango bingsu": [],
"Mango crepe": [],
"Mango lassi": [],
"Mango pudding": [],
"Mango sago pomelo": [],
"Mango sticky rice": [],
"Mangosteen": [],
"Mapo tofu": [
"tofu"
],
"Maritozzo cream bun": [
"bread"
],
"Marmite chicken": [
"chicken"
],
"Mars": [],
"Masala King chicken tikka masala": [
"flatbread",
"chicken"
],
"Masala tea": [],
"Masala thosai": [
"flatbread"
],
"Massaman curry with rice": [
"white rice"
],
"Matcha latte, iced": [],
"Matcha tiramisu": [],
"McChicken": [
"bread",
"chicken"
],
"McFlurry with Oreo": [],
"Mee bandung": [
"yellow noodles"
],
"Mee goreng mamak": [
"yellow noodles"
],
"Mee goreng, home-style": [
"yellow noodles"
],
"Mee hoon kueh soup": [
"flour noodles",
"pork",
"anchovies",
"vegetables"
],
"Mee jawa": [
"yellow noodles",
"prawn",
"egg",
"tofu",
"vegetables"
],
"Mee kicap": [
"yellow noodles",
"chicken",
"vegetables"
],
"Mee rebus": [
"yellow noodles"
],
"Mee siam": [
"yellow noodles",
"bee hoon"
],
"Mee soto": [
"yellow noodles",
"chicken",
"vegetables"
],
"Mee sua soup with chicken": [
"bee hoon",
"mee sua",
"chicken",
"egg"
],
"Mentos": [],
"Mexican coffee bun": [
"bread"
],
"Mi xian in soup with meatballs": [],
"Mille crepe, original": [],
"Mille crêpe": [],
"Milo ais": [],
"Milo, packet drink": [],
"Minced pork soup": [
"pork",
"vegetables"
],
"Miso salmon and tofu soup with rice": [
"white rice",
"fish",
"tofu"
],
"Miso soup": [],
"Miso soup with tofu, rice and egg": [
"white rice",
"egg",
"tofu"
],
"Mixed berries": [],
"Mixed cut fruit": [],
"Mixed nuts": [],
"Mixed vegetable minestrone": [
"vegetables"
],
"Mixue ice cream cone": [],
"Mixue lemonade": [],
"Mocha, iced": [],
"Mochi": [],
"Mochi doughnut": [],
"Molten lava cake with ice cream": [],
"Mont Blanc": [],
"Moo ping with sticky rice": [],
"Mookata (Thai BBQ), typical meal": [],
"Mooncake, lotus paste": [],
"Muah chee": [],
"Munchy's Oat Krunch Strawberry & Blackcurrant": [],
"Murtabak ayam": [
"flatbread",
"chicken"
],
"Musang King durian": [],
"Mushroom Swiss burger": [
"bread"
],
"Mushroom chicken stew": [
"chicken"
],
"Mushroom clear soup": [],
"Mushroom soup with garlic bread": [
"bread"
],
"Mutton curry with rice": [
"white rice",
"mutton"
],
"Nasi Lemak McD": [
"coconut rice",
"anchovies"
],
"Nasi ayam penyet": [
"chicken rice",
"chicken"
],
"Nasi briyani ayam": [
"spiced rice",
"chicken"
],
"Nasi briyani, rice only": [
"spiced rice"
],
"Nasi campur: ayam bakar, sayur, ulam": [
"white rice",
"chicken",
"vegetables"
],
"Nasi dagang": [
"white rice",
"fish"
],
"Nasi goreng USA": [
"fried rice"
],
"Nasi goreng ayam": [
"fried rice",
"chicken"
],
"Nasi goreng belacan": [
"fried rice"
],
"Nasi goreng cina": [
"fried rice"
],
"Nasi goreng daging merah": [
"fried rice",
"beef"
],
"Nasi goreng ikan masin": [
"fried rice",
"fish"
],
"Nasi goreng kampung": [
"fried rice"
],
"Nasi goreng pattaya": [
"fried rice"
],
"Nasi goreng seafood": [
"fried rice"
],
"Nasi goreng tom yum": [
"fried rice"
],
"Nasi goreng with fried egg": [
"fried rice",
"egg"
],
"Nasi kandar": [
"white rice",
"chicken"
],
"Nasi kerabu with ayam percik": [
"white rice",
"chicken"
],
"Nasi lemak ayam goreng": [
"coconut rice",
"chicken",
"anchovies"
],
"Nasi lemak biasa": [
"coconut rice",
"egg",
"anchovies"
],
"Nasi lemak kerang": [
"coconut rice",
"anchovies"
],
"Nasi lemak rendang ayam": [
"coconut rice",
"chicken",
"anchovies"
],
"Nasi lemak rice": [
"coconut rice",
"anchovies"
],
"Nasi lemak sambal sotong": [
"coconut rice",
"squid",
"anchovies"
],
"Nasi lemak telur mata": [
"coconut rice",
"egg",
"anchovies"
],
"Nasi lemak telur rebus, kurang nasi": [
"coconut rice",
"egg",
"anchovies"
],
"Nasi mandi ayam": [
"spiced rice",
"chicken"
],
"Nasi mandi ayam, half rice": [
"spiced rice",
"chicken",
"vegetables"
],
"Nasi minyak": [
"spiced rice"
],
"Nasi tomato": [
"spiced rice"
],
"Neslo ais": [],
"Oat porridge, savoury": [
"porridge",
"egg"
],
"Oats with milk and banana": [],
"Oden": [
"fish ball"
],
"Oil rice (chicken rice)": [
"chicken rice",
"chicken"
],
"Okonomiyaki": [],
"Old cucumber soup with pork ribs": [
"pork"
],
"Omelette with onion and chilli (telur dadar)": [
"egg"
],
"Omelette with rice and cucumber": [
"white rice",
"egg"
],
"Omelette, plain": [
"egg"
],
"Ondeh-ondeh cake": [],
"Onigiri and boiled egg": [
"fish",
"egg"
],
"Orange": [],
"Original Recipe chicken, 2 pieces": [
"chicken"
],
"Original chicken, 2 pieces": [
"chicken"
],
"Otak-otak": [],
"Overnight oats with Greek yogurt and berries, light": [],
"Overnight oats with chia and banana": [],
"Pad kra pao chicken, home-style": [
"white rice",
"chicken"
],
"Pad kra pao with rice and egg": [
"white rice",
"egg"
],
"Pad thai": [
"bee hoon"
],
"Pain au chocolat": [],
"Pan mee, soup": [
"flour noodles"
],
"Pan-fried pork dumplings": [
"pork"
],
"Pan-seared salmon, rice and greens": [
"white rice",
"fish",
"vegetables"
],
"Pancakes with syrup": [],
"Pandan chiffon cake": [],
"Pandan gula melaka cake": [],
"Pandan layer cake": [],
"Papaya": [],
"Paper thosai": [
"flatbread"
],
"Pavlova with cream and fruit": [],
"Peach": [],
"Peanut butter banana toast": [
"bread"
],
"Peanut butter toast": [
"bread"
],
"Peanut paste": [],
"Pear": [],
"Pecan pie": [],
"Penne arrabbiata with chicken": [
"pasta",
"chicken"
],
"Peony roast meat rice": [
"white rice"
],
"Persimmon": [],
"Pho": [
"egg noodles",
"bee hoon",
"beef"
],
"Pho, chicken": [
"bee hoon",
"chicken"
],
"Pineapple": [],
"Pineapple bun with butter": [
"bread"
],
"Pineapple fried rice": [
"fried rice"
],
"Pineapple tarts": [],
"Pisang emas": [],
"Pisang goreng": [],
"Pistachio latte": [],
"Pizza": [
"flatbread"
],
"Pizza, BBQ chicken": [
"flatbread",
"chicken"
],
"Pizza, Hawaiian chicken": [
"flatbread",
"chicken"
],
"Pizza, cheese stuffed crust": [
"flatbread"
],
"Pizza, margherita": [
"flatbread"
],
"Pizza, meat lovers": [
"flatbread"
],
"Pizza, pepperoni": [
"flatbread"
],
"Pizza, seafood (tuna or prawn)": [
"flatbread",
"fish",
"prawn"
],
"Pizza, thin crust": [
"flatbread"
],
"Pizza, veggie": [
"flatbread"
],
"Plain fried rice": [
"fried rice"
],
"Plain naan": [
"flatbread"
],
"Plain porridge with side dishes (Teochew)": [
"porridge"
],
"Plum": [],
"Poached chicken rice (Chicken Rice Hub)": [
"chicken rice",
"chicken"
],
"Poached eggs": [
"egg"
],
"Poke bowl, salmon": [
"fish"
],
"Polo bun with butter": [
"bread"
],
"Pomegranate": [],
"Pomelo": [],
"Popcorn, caramel": [],
"Popcorn, salted": [],
"Popiah, 1 roll": [],
"Popiah, fresh": [],
"Popo Fish Muruku": [
"fish"
],
"Pork noodle soup": [
"bee hoon",
"kway teow",
"pork",
"vegetables"
],
"Pork noodle soup, dry": [
"pork"
],
"Pork noodle soup, lean pork only": [
"bee hoon",
"pork",
"vegetables"
],
"Pork porridge": [
"porridge",
"pork"
],
"Portuguese egg tart": [
"egg"
],
"Potato chips": [],
"Potato with minced pork": [
"pork"
],
"Prawn and egg white fried cauliflower rice, light": [
"white rice",
"prawn",
"egg"
],
"Prawn cheong fun": [
"prawn"
],
"Prawn mee": [
"yellow noodles",
"prawn"
],
"Prawn skewer": [
"prawn"
],
"Pretzel, cinnamon sugar": [],
"Pretzel, original": [],
"Prunes": [],
"Pumpkin soup": [],
"Puri with potato masala": [],
"Quarter chicken with 2 sides": [
"chicken",
"vegetables"
],
"Quarter chicken with 3 sides": [
"chicken",
"vegetables"
],
"Quiche": [],
"Quinoa chicken bowl": [
"chicken"
],
"Raisins": [],
"Rambutan": [],
"Ramen, chicken paitan": [
"ramen",
"chicken"
],
"Ramen, miso": [
"ramen"
],
"Ramen, shio": [
"ramen"
],
"Ramen, shoyu": [
"ramen"
],
"Ramen, spicy miso": [
"ramen"
],
"Ramen, tantanmen": [
"ramen"
],
"Ramen, tonkotsu": [
"ramen",
"pork"
],
"Ramly burger, ayam special": [
"flatbread",
"chicken",
"egg"
],
"Ramly burger, biasa": [
"bread"
],
"Ramly burger, special": [
"flatbread",
"egg"
],
"Ramyeon with egg": [
"ramen",
"egg"
],
"Raspberries": [],
"Rava thosai": [
"flatbread"
],
"Ravioli in tomato sauce": [],
"Red bean bun": [
"bread"
],
"Red bean soup": [],
"Red curry chicken with rice": [
"white rice",
"chicken"
],
"Red velvet cake": [],
"Rendang daging with rice": [
"white rice",
"beef"
],
"Ritz crackers": [],
"Roast chicken, quarter": [
"chicken"
],
"Roast duck rice": [
"chicken rice",
"duck"
],
"Roasted chicken rice (Choon Yien)": [
"chicken rice",
"chicken"
],
"Rock melon": [],
"Rojak buah": [],
"Root beer float": [],
"Roti bawang": [
"flatbread"
],
"Roti bom": [
"flatbread"
],
"Roti canai kosong": [
"flatbread"
],
"Roti cheese": [
"flatbread"
],
"Roti john": [
"flatbread"
],
"Roti pisang": [
"flatbread"
],
"Roti planta": [
"flatbread"
],
"Roti sardin": [
"flatbread"
],
"Roti telur": [
"flatbread",
"egg"
],
"Roti telur bawang": [
"flatbread",
"egg"
],
"Roti tisu": [
"flatbread"
],
"Sago gula melaka": [],
"Salmon don": [
"white rice",
"fish"
],
"Salmon maki": [
"fish"
],
"Salmon mentai aburi": [
"fish"
],
"Salmon nigiri": [
"fish"
],
"Salmon poke bowl": [
"brown rice",
"fish"
],
"Salmon sashimi": [
"fish"
],
"Salmon sashimi and rice": [
"white rice",
"fish"
],
"Salmon teriyaki bento": [
"white rice",
"fish"
],
"Salmon teriyaki with rice and broccoli": [
"white rice",
"fish",
"vegetables"
],
"Salt bread (shio pan)": [
"bread"
],
"Salted egg": [
"egg"
],
"Salted egg chicken rice": [
"chicken rice",
"chicken",
"egg"
],
"Salted egg custard bun (lau sar bao)": [
"bread",
"egg"
],
"Salted egg pasta": [
"pasta",
"egg"
],
"Salted vegetable and tofu soup": [
"tofu",
"vegetables"
],
"Salted vegetable duck soup": [
"duck",
"vegetables"
],
"Sambal egg": [
"egg"
],
"Sambal fish": [
"fish"
],
"Sambal kangkung": [
"vegetables"
],
"Sambal prawns": [
"prawn"
],
"Samgyeopsal set with rice": [
"white rice",
"pork",
"vegetables"
],
"Samgyetang": [
"white rice",
"chicken"
],
"Samosa": [],
"Sang har mee": [
"egg noodles",
"prawn",
"egg"
],
"Sarawak laksa": [
"bee hoon"
],
"Sardine sandwich": [
"bread"
],
"Satay ayam": [
"white rice",
"chicken"
],
"Sate lilit": [
"fish"
],
"Sate padang": [],
"Sausage McMuffin with egg": [
"pork",
"egg"
],
"Sausage bun": [
"bread",
"pork"
],
"Sausage skewer": [
"pork"
],
"Scone with jam and cream": [],
"Scrambled eggs": [
"egg"
],
"Scrambled eggs on wholemeal toast": [
"bread",
"egg"
],
"Seafood marinara pasta": [
"pasta"
],
"Seafood pajeon": [],
"Seafood ramyeon (jjamppong style)": [
"ramen"
],
"Seaweed and tofu soup": [
"tofu"
],
"Seaweed snack": [],
"Sesame balls (jin deui)": [],
"Sheng jian bao": [],
"Shin Ramyun": [],
"Shin Ramyun with egg and cheese": [
"egg"
],
"Shish tawook, 1 skewer": [
"chicken"
],
"Shucked fresh oysters": [],
"Siew mai": [],
"Siew yuk rice": [
"white rice"
],
"Sirap bandung": [],
"Smoked salmon and cream cheese bagel": [
"bread",
"fish"
],
"Smoothie bowl": [],
"Snow fungus with longan": [],
"Soba with chicken and edamame": [
"chicken"
],
"Soda limau asam boi": [],
"Soft drink": [],
"Soft drink, zero sugar": [],
"Soft serve cone": [],
"Som tam": [],
"Soto ayam": [
"white rice",
"chicken"
],
"Sotteok sotteok": [
"pork"
],
"Soufflé pancakes": [],
"Sour fish soup (suan cai fish)": [
"fish"
],
"Sour fish soup with rice": [
"white rice",
"fish"
],
"Soursop": [],
"Soy garlic chicken rice box": [
"chicken rice",
"chicken"
],
"Soy sauce chicken (economy rice)": [
"white rice",
"chicken"
],
"Soy sauce fried fish": [
"fish"
],
"Soya cincau": [],
"Spaghetti aglio olio": [
"pasta"
],
"Spaghetti aglio olio with chicken": [
"pasta",
"chicken"
],
"Spaghetti aglio olio with prawns": [
"pasta",
"prawn"
],
"Spaghetti alfredo": [
"pasta"
],
"Spaghetti arrabbiata": [
"pasta"
],
"Spaghetti bolognese": [
"pasta"
],
"Spaghetti carbonara": [
"pasta"
],
"Spaghetti meatballs": [
"pasta"
],
"Spaghetti pesto with chicken": [
"pasta",
"chicken"
],
"Spaghetti pomodoro": [
"pasta"
],
"Spaghetti vongole": [
"pasta"
],
"Spicy Chicken McDeluxe": [
"bread",
"chicken"
],
"Spicy salmon roll": [
"fish"
],
"Spinach and egg drop soup with rice": [
"white rice",
"egg",
"vegetables"
],
"Spinach and tofu clear soup": [
"tofu",
"vegetables"
],
"Spring rolls, fried": [],
"Squid skewer": [
"squid"
],
"Starfruit": [],
"Steak with vegetables": [
"beef",
"vegetables"
],
"Steamed Hainan bread with kaya": [
"bread"
],
"Steamed chicken breast with ginger, brown rice and choy sum": [
"brown rice",
"chicken",
"vegetables"
],
"Steamed chicken with ginger and wolfberries": [
"chicken"
],
"Steamed chicken, rice and broccoli": [
"white rice",
"chicken",
"vegetables"
],
"Steamed corn": [],
"Steamed egg custard": [
"egg"
],
"Steamed egg with minced chicken, rice": [
"white rice",
"chicken",
"egg"
],
"Steamed egg with minced pork and rice": [
"white rice",
"pork",
"egg"
],
"Steamed fish (economy rice)": [
"white rice",
"fish"
],
"Steamed fish balls": [
"fish ball"
],
"Steamed fish with rice and greens": [
"white rice",
"fish",
"vegetables"
],
"Steamed fish with tofu and spring onion, no rice": [
"white rice",
"fish",
"tofu"
],
"Steamed minced pork patty": [
"pork"
],
"Steamed minced pork patty with rice": [
"white rice",
"pork",
"vegetables"
],
"Steamed minced pork with preserved vegetable": [
"pork",
"vegetables"
],
"Steamed minced pork with salted egg": [
"pork",
"egg"
],
"Steamed minced pork with salted fish": [
"pork",
"fish"
],
"Steamed pork ribs with black bean": [
"pork"
],
"Steamed siakap with ginger and soy, rice and greens": [
"white rice",
"fish",
"vegetables"
],
"Sticky rice": [],
"Stir-fried broccoli": [
"vegetables"
],
"Stir-fried cabbage": [
"vegetables"
],
"Stir-fried choy sum": [
"vegetables"
],
"Stir-fried greens": [
"vegetables"
],
"Stir-fried kai lan with garlic": [],
"Stir-fried long beans": [
"vegetables"
],
"Stir-fried spinach (bayam)": [
"chicken",
"vegetables"
],
"Stir-fried sweet potato leaves": [],
"Strawberries": [],
"Strawberry matcha latte": [],
"Strawberry shortcake": [],
"Stuffed tofu and brinjal": [
"tofu",
"vegetables"
],
"Sugarcane juice": [],
"Sui kow in soup": [],
"Sundubu jjigae with rice": [
"white rice"
],
"Sup ayam with rice": [
"white rice",
"chicken"
],
"Sushi plates": [
"fish"
],
"Sweet and sour chicken": [
"chicken"
],
"Sweet and sour chicken with rice": [
"white rice",
"chicken"
],
"Sweet and sour fish": [
"fish"
],
"Sweet and sour pork": [
"pork"
],
"Sweet and sour pork with rice": [
"white rice",
"pork"
],
"Sweet potato ginger soup": [],
"Swiss roll": [],
"Tacos": [],
"Taiyaki": [],
"Takoyaki": [],
"Tamago nigiri": [
"egg"
],
"Tandoori chicken with naan": [
"flatbread",
"chicken"
],
"Tang yuan in ginger syrup": [],
"Tanghulu": [],
"Taro milk tea with pearls": [],
"Taro sago": [],
"Tau fu fah, less syrup": [],
"Teh ais": [],
"Teh tarik": [],
"Telur bungkus": [
"egg"
],
"Tempeh and vegetable sambal with rice": [
"white rice",
"tofu",
"vegetables"
],
"Tempeh, fried": [
"tofu"
],
"Tempura udon": [
"flour noodles"
],
"Texas Chicken, 2 pieces": [
"chicken"
],
"Thai basil chicken rice with egg": [
"chicken rice",
"chicken",
"egg"
],
"Thai fried rice with chicken": [
"fried rice",
"chicken"
],
"Thai milk tea": [],
"Thosai telur": [
"flatbread",
"egg"
],
"Thosai with dhal and chutney": [
"flatbread"
],
"Three-colour steamed egg": [
"egg"
],
"Tic Tac": [],
"Tim Tam Chewy Caramel": [],
"Tim Tam Double Coat": [],
"Tim Tam Original": [],
"Tim Tam White": [],
"Tiramisu": [],
"Tofu and minced chicken with long beans": [
"chicken",
"tofu",
"vegetables"
],
"Tofu and vegetable stir-fry with rice": [
"white rice",
"tofu",
"vegetables"
],
"Tofu and vegetable stir-fry, no rice": [
"white rice",
"tofu",
"vegetables"
],
"Tofu, braised": [
"tofu"
],
"Tom yum noodles": [],
"Tom yum pasta": [
"pasta"
],
"Tom yum soup with rice": [
"white rice"
],
"Tomato and egg soup": [
"egg"
],
"Tomato egg noodle soup": [
"egg"
],
"Tomato egg stir-fry": [
"egg"
],
"Tonkatsu set": [
"white rice",
"chicken",
"vegetables"
],
"Truffle mushroom pasta": [
"pasta"
],
"Tsukemen (dipping ramen)": [
"ramen"
],
"Tteok-kkochi (rice cake skewer)": [],
"Tteokbokki": [],
"Tuna and egg salad wrap": [
"flatbread",
"fish",
"egg",
"vegetables"
],
"Tuna mayo bun": [
"bread",
"fish"
],
"Tuna nigiri": [
"fish"
],
"Tuna puff": [
"fish"
],
"Tuna salad": [
"fish",
"vegetables"
],
"Tuna sandwich": [
"bread",
"fish"
],
"Turmeric rice (nasi kunyit)": [
"spiced rice"
],
"Twix": [],
"Udon in soup": [
"flour noodles",
"egg"
],
"Udon, beef (niku)": [
"flour noodles",
"beef"
],
"Udon, curry": [
"flour noodles"
],
"Udon, kitsune": [
"flour noodles"
],
"Ulam with sambal and grilled chicken, light": [
"chicken"
],
"Ulam with sambal on the side": [],
"Unagi don": [
"white rice",
"fish"
],
"Unagi nigiri": [
"fish"
],
"Unsweetened soy milk": [],
"Vadai": [],
"Vegetable curry with rice": [
"white rice",
"vegetables"
],
"Vegetable dumplings, steamed": [
"vegetables"
],
"Veggie burger": [
"bread"
],
"Vietnamese fresh spring rolls": [],
"Vietnamese prawn rice paper rolls": [
"prawn"
],
"Vinegar pork trotter": [
"pork"
],
"Vitagen": [],
"Waffle with ice cream": [],
"Waffle with syrup": [],
"Walnut paste": [],
"Want Want Senbei": [],
"Want Want Shelly Senbei": [],
"Wantan mee (PJ Old Town)": [
"egg noodles"
],
"Wantan mee, dry": [
"egg noodles"
],
"Wantan mee, soup": [
"egg noodles",
"pork",
"egg",
"vegetables"
],
"Wat tan hor": [
"kway teow"
],
"Watercress soup with lean pork": [
"pork"
],
"Watermelon": [],
"Watermelon juice, no sugar": [],
"Whipped potato with gravy": [],
"White Rabbit candy": [],
"White bread": [
"bread"
],
"White radish soup with pork ribs": [
"pork"
],
"White rice": [
"white rice"
],
"White rice (economy rice)": [
"white rice"
],
"Whopper": [
"bread"
],
"Winter melon soup with dried scallops": [],
"Xiao long bao": [],
"Yaki udon": [
"flour noodles"
],
"Yakiniku, typical meal": [
"white rice"
],
"Yakisoba": [
"yellow noodles"
],
"Yakitori": [],
"Yakult Ace": [],
"Yakult green tea": [],
"Yam puff (wu kok)": [],
"Yan Yan Strawberry": [],
"Yang chow fried rice": [
"fried rice"
],
"Yong tau foo (PJ Old Town)": [
"tofu"
],
"Yong tau foo, soup": [
"bee hoon",
"tofu"
],
"Yoursnackies snack pack": [],
"Zha jiang mian": [
"yellow noodles",
"pork"
],
"Zinger burger": [
"bread"
],
"Zucchini noodles with prawns and tomato": [
"prawn"
],
"llaollao Sanum": [],
"llaollao, medium tub with 3 toppings": [],
"llaollao, small tub with 1 topping": [],
"myNEWS Maru bento, chicken": [
"white rice",
"chicken"
],
"myNEWS Maru onigiri, tuna mayo": [
"fish"
],
"myNEWS Maru sandwich, egg mayo": [
"bread",
"egg"
]
}
# new dishes: search phrase -> exact food name in the app
V9_EXTRA = {
"chobani greek yogurt cup": "Flavoured yogurt cup",
"greek yogurt tub with fruit": "Flavoured yogurt cup",
"flavoured yogurt cup supermarket": "Flavoured yogurt cup",
"onigiri rice ball nori": "FamilyMart onigiri, tuna mayo",
"familymart onigiri": "FamilyMart onigiri, tuna mayo",
"7 eleven onigiri": "FamilyMart onigiri, tuna mayo",
"familymart oden": "FamilyMart oden, fish cake",
"oden skewers in broth": "FamilyMart oden, fish cake",
"japanese oden bowl": "FamilyMart oden, fish cake",
"familymart sofuto soft serve": "Soft serve cone",
"soft serve ice cream cone": "Soft serve cone",
"convenience store egg mayo sandwich": "7-Eleven egg mayo sandwich",
"triangle sandwich pack": "7-Eleven egg mayo sandwich",
"tamago sando": "7-Eleven egg mayo sandwich",
"familymart bento box": "FamilyMart chicken katsu bento",
"convenience store bento": "FamilyMart chicken katsu bento",
"chicken katsu bento box": "FamilyMart chicken katsu bento",
"7 eleven slurpee cup": "7-Eleven Slurpee, regular",
"slurpee drink": "7-Eleven Slurpee, regular",
"gardenia white bread slices": "White bread",
"sliced white bread loaf": "White bread",
"milo packet drink": "Milo, packet drink",
"chocolate milk drink carton straw": "Milo, packet drink",
"dutch lady milk packet": "Milo, packet drink",
"nasi lemak ayam goreng": "Nasi lemak ayam goreng",
"nasi lemak with fried chicken": "Nasi lemak ayam goreng",
"nasi lemak ayam berempah": "Nasi lemak ayam goreng",
"grilled ribeye steak plate": "Steak with vegetables",
"sirloin steak with fries": "Steak with vegetables",
"steak dinner plate": "Steak with vegetables",
"vitagen bottle": "Vitagen",
"vitagen cultured milk drink malaysia": "Vitagen",
"yakult bottle": "Yakult Ace",
"yakult cultured milk drink": "Yakult Ace",
"dubai chewy cookie": "Dubai chewy cookie",
"dubai cookie pistachio kataifi": "Dubai chewy cookie",
"plain white rice bowl": "White rice (economy rice)",
"steamed white rice plate": "White rice (economy rice)",
"brown rice bowl": "Brown rice (economy rice)",
"hainanese chicken rice oil rice plain": "Oil rice (chicken rice)",
"nasi lemak rice only coconut rice": "Nasi lemak rice",
"basmati rice bowl": "Basmati rice, plain",
"nasi briyani rice": "Briyani rice",
"nasi kunyit turmeric rice": "Turmeric rice (nasi kunyit)",
"plain egg fried rice": "Plain fried rice",
"economy rice fried chicken wing": "Fried chicken wing (economy rice)",
"fried chicken wings plate": "Fried chicken wing (economy rice)",
"malaysian curry chicken pieces": "Curry chicken (economy rice)",
"sweet and sour pork": "Sweet and sour pork",
"sweet and sour chicken": "Sweet and sour chicken",
"marmite chicken": "Marmite chicken",
"kung pao chicken": "Kung pao chicken",
"black pepper chicken stir fry": "Black pepper chicken",
"lemon chicken chinese": "Lemon chicken",
"braised pork belly soy sauce tau yu bak": "Braised pork belly (tau yu bak)",
"fried fish fillet": "Fried fish fillet",
"steamed fish soy sauce ginger": "Steamed fish (economy rice)",
"sambal fish malaysia": "Sambal fish",
"sambal prawns": "Sambal prawns",
"luncheon meat slices fried": "Luncheon meat slices",
"egg foo yong": "Egg foo yong (omelette)",
"sambal egg telur": "Sambal egg",
"braised tofu chinese": "Braised tofu (economy rice)",
"mapo tofu": "Mapo tofu",
"kangkung belacan": "Kangkung belacan",
"stir fried cabbage chinese": "Stir-fried cabbage",
"stir fried long beans egg": "Long beans with egg",
"stir fried bean sprouts": "Bean sprouts stir-fry",
"stir fried broccoli": "Stir-fried broccoli",
"bitter gourd with egg": "Bitter gourd with egg",
"stir fried choy sum": "Stir-fried choy sum",
"pork noodle soup malaysia": "Pork noodle soup",
"zhu za fen pork noodles": "Pork noodle soup",
"wantan mee soup": "Wantan mee, soup",
"wonton noodle soup malaysia": "Wantan mee, soup",
"beef ball noodle soup": "Beef ball noodle soup",
"beef ball noodles malaysia": "Beef ball noodle soup",
"beef noodle soup malaysia": "Beef noodle soup",
"braised beef noodle soup": "Beef noodle soup",
"fish ball noodle soup": "Fish ball noodle soup",
"fishball noodles malaysia": "Fish ball noodle soup",
"fish head noodle milky soup": "Fish head noodle, milky soup",
"fish head bee hoon milky": "Fish head noodle, milky soup",
"kway teow soup": "Kway teow soup",
"kuey teow th'ng": "Kway teow soup",
"duck noodle soup": "Duck noodle soup",
"braised duck noodles": "Duck noodle soup",
"ipoh chicken hor fun": "Chicken noodle soup",
"chicken noodle soup malaysia": "Chicken noodle soup",
"mee sua soup": "Mee sua soup with chicken",
"misua soup chicken": "Mee sua soup with chicken",
"sang har mee": "Sang har mee",
"freshwater prawn noodles crispy": "Sang har mee",
"cantonese fried yee mee": "Cantonese fried noodles (yee mee)",
"cantonese fried noodles egg gravy": "Cantonese fried noodles (yee mee)",
"wat tan hor": "Wat tan hor",
"kway teow egg gravy": "Wat tan hor",
"curry mee malaysia": "Curry mee",
"curry noodles penang": "Curry mee",
"mee soto": "Mee soto",
"soto mee malaysia": "Mee soto",
"mee hoon kueh": "Mee hoon kueh soup",
"pan mee hand torn noodles soup": "Mee hoon kueh soup",
"mee jawa": "Mee jawa",
"mee rebus sweet potato gravy": "Mee jawa",
"mee bandung muar": "Mee bandung",
"mee bandung": "Mee bandung",
"claypot loh shu fun": "Claypot loh shu fun",
"claypot silver needle noodles": "Claypot loh shu fun",
"claypot yee mee": "Claypot yee mee",
"claypot noodles egg": "Claypot yee mee",
"kon lo mee": "Kolo mee / Kon lo mee",
"kolo mee char siu": "Kolo mee / Kon lo mee",
"kampua mee": "Kolo mee / Kon lo mee",
"kampua noodles sibu": "Kolo mee / Kon lo mee",
"hakka mee": "Kolo mee / Kon lo mee",
"hakka noodles minced pork": "Kolo mee / Kon lo mee",
"bak chor mee": "Bak chor mee",
"minced meat noodles singapore": "Bak chor mee",
"mee kicap": "Mee kicap",
"soy sauce noodles malaysia": "Mee kicap",
"kway teow goreng": "Char kway teow / Kway teow goreng",
"fried kuey teow malay style": "Char kway teow / Kway teow goreng",
"bee hoon goreng": "Bee hoon goreng",
"fried rice vermicelli malaysia": "Bee hoon goreng",
"bee hoon soup": "Bee hoon soup / Mee hoon soup",
"rice vermicelli soup fish ball": "Bee hoon soup / Mee hoon soup",
"maggi goreng": "Maggi goreng",
"mee maggi goreng mamak": "Maggi goreng",
"tom yum noodles": "Tom yum noodles",
"tom yam mee": "Tom yum noodles",
"dan dan noodles": "Dan dan noodles",
"dandan mian": "Dan dan noodles",
"zha jiang mian": "Zha jiang mian",
"jjajangmyeon black bean noodles": "Zha jiang mian",
"lanzhou beef noodles": "Lanzhou beef lamian",
"lanzhou lamian": "Lanzhou beef lamian",
"mala noodles": "Mala noodles",
"sichuan spicy noodles": "Mala noodles",
"japchae": "Japchae",
"korean glass noodles stir fry": "Japchae",
"yakisoba": "Yakisoba",
"japanese fried noodles yakisoba": "Yakisoba",
"laksa johor": "De Irrfan's laksa Johor",
"laksa johor spaghetti": "De Irrfan's laksa Johor",
"cup noodles": "Cup noodles",
"instant cup noodles cooked": "Cup noodles",
"nasi lemak telur rebus kurang nasi": "Nasi lemak telur rebus, kurang nasi",
"half-boiled eggs with wholemeal toast": "Half-boiled eggs with wholemeal toast",
"bubur ayam mcd malaysia": "Bubur Ayam McD",
"egg mcmuffin food": "Egg McMuffin",
"oats with milk and banana": "Oats with milk and banana",
"greek yogurt with fruit and oats": "Greek yogurt with fruit and oats",
"scrambled eggs on wholemeal toast": "Scrambled eggs on wholemeal toast",
"peanut butter banana toast": "Peanut butter banana toast",
"egg sandwich food": "Egg sandwich",
"onigiri and boiled egg": "Onigiri and boiled egg",
"egg mayo sandwich": "Egg mayo sandwich",
"hainan toast with kaya and eggs": "Hainan toast with kaya, and eggs",
"steamed hainan bread with kaya": "Steamed Hainan bread with kaya",
"chicken rice roasted chinese dish": "Chicken rice, roasted",
"stir-fried spinach bayam": "Stir-fried spinach (bayam)",
"lettuce with oyster sauce": "Lettuce with oyster sauce",
"brinjal with sambal malaysia": "Brinjal with sambal",
"curry vegetables sayur lodeh": "Curry vegetables (sayur lodeh)",
"curry gravy add-on": "Curry gravy, add-on",
"economy rice steamed fish tofu greens": "Economy rice: steamed fish, tofu, greens",
"nasi campur ayam bakar sayur ulam": "Nasi campur: ayam bakar, sayur, ulam",
"nasi goreng with fried egg": "Nasi goreng with fried egg",
"banana leaf rice vegetarian": "Banana leaf rice, vegetarian",
"laksam food malaysia": "Laksam",
"fish head mee hoon clear soup": "Fish head mee hoon, clear soup",
"grilled fish with vegetables": "Grilled fish with vegetables",
"spaghetti aglio olio with chicken": "Spaghetti aglio olio with chicken",
"quarter chicken with 2 sides": "Quarter chicken with 2 sides",
"chicken pita food": "Chicken pita",
"quarter chicken with 3 sides": "Quarter chicken with 3 sides",
"6-inch roasted chicken sub": "6-inch roasted chicken sub",
"6-inch chicken teriyaki sub": "6-inch chicken teriyaki sub",
"6-inch tuna sub": "6-inch tuna sub",
"6-inch veggie delite with egg": "6-inch Veggie Delite with egg",
"filet-o-fish food": "Filet-O-Fish",
"mcchicken food": "McChicken",
"grilled chicken burger gcb": "Grilled chicken burger (GCB)",
"ayam goreng mcd": "Ayam Goreng McD",
"original recipe chicken 2 pieces": "Original Recipe chicken, 2 pieces",
"zinger burger food": "Zinger burger",
"crispy chicken with rice": "Crispy chicken with rice",
"original chicken 2 pieces": "Original chicken, 2 pieces",
"salmon poke bowl": "Salmon poke bowl",
"build-your-own salad bowl": "Build-your-own salad bowl",
"salmon teriyaki bento": "Salmon teriyaki bento",
"gyudon food": "Gyudon",
"gyudon mini with salad": "Gyudon, mini with salad",
"soy garlic chicken rice box": "Soy garlic chicken rice box",
"kimbap food": "Kimbap",
"thai basil chicken rice with egg": "Thai basil chicken rice with egg",
"pineapple fried rice": "Pineapple fried rice",
"chicken shawarma wrap": "Chicken shawarma wrap",
"nasi mandi ayam": "Nasi mandi ayam",
"nasi mandi ayam half rice": "Nasi mandi ayam, half rice",
"chicken kebab plate with salad": "Chicken kebab plate with salad",
"lentil soup with pita": "Lentil soup with pita",
"kofta plate with rice": "Kofta plate with rice",
"mi xian in soup with meatballs": "Mi xian in soup with meatballs",
"roti john food malaysia": "Roti john",
"steamed chicken rice and broccoli": "Steamed chicken, rice and broccoli",
"egg fried rice with vegetables": "Egg fried rice with vegetables",
"tofu and vegetable stir-fry with rice": "Tofu and vegetable stir-fry with rice",
"pork noodle soup lean pork only": "Pork noodle soup, lean pork only",
"pork noodle soup dry chinese dish": "Pork noodle soup, dry",
"beef ball noodles dry chinese dish": "Beef ball noodles, dry",
"fish slice noodle soup chinese dish": "Fish slice noodle soup",
"duck noodles dry chinese dish": "Duck noodles, dry",
"chicken noodles dry chinese dish": "Chicken noodles, dry",
"glass noodle stir-fry chinese dish": "Glass noodle stir-fry",
"tomato egg noodle soup chinese dish": "Tomato egg noodle soup",
"samgyeopsal set with rice": "Samgyeopsal set with rice",
"kimchi fried rice with egg": "Kimchi fried rice with egg",
"ramyeon with egg": "Ramyeon with egg",
"korean fried chicken yangnyeom": "Korean fried chicken, yangnyeom",
"mookata thai bbq": "Mookata (Thai BBQ), typical meal",
"yakiniku food": "Yakiniku, typical meal",
"lamb skewers bbq chinese dish": "Lamb skewers (BBQ)",
"grilled chicken wings bbq": "Grilled chicken wings (BBQ)",
"ikan pari bakar with rice": "Ikan pari bakar with rice",
"lok lok food chinese dish": "Lok lok",
"spicy chicken mcdeluxe": "Spicy Chicken McDeluxe",
"double cheeseburger food": "Double cheeseburger",
"nasi lemak mcd malaysia": "Nasi Lemak McD",
"french fries food": "French fries, large",
"sausage mcmuffin with egg": "Sausage McMuffin with egg",
"original recipe breast": "KFC Original Recipe, breast",
"original recipe drumstick": "KFC Original Recipe, drumstick",
"original recipe wing": "KFC Original Recipe, wing",
"hot spicy breast": "KFC Hot & Spicy, breast",
"cheezy wedges food": "Cheezy wedges, large",
"nuggets food": "KFC nuggets",
"whipped potato with gravy": "Whipped potato with gravy",
"coleslaw food": "Coleslaw",
"whopper food": "Whopper",
"texas chicken 2 pieces": "Texas Chicken, 2 pieces",
"honey butter biscuit": "Honey butter biscuit",
"coney dog food": "Coney dog",
"root beer float": "Root beer float",
"ramly burger special malaysia": "Ramly burger, special",
"salmon don food": "Salmon don",
"tempura udon food": "Tempura udon",
"salmon nigiri food": "Salmon nigiri",
"thai fried rice with chicken": "Thai fried rice with chicken",
"moo ping with sticky rice": "Moo ping with sticky rice",
"chicken chop with black pepper sauce": "Chicken chop with black pepper sauce",
"mushroom soup with garlic bread": "Mushroom soup with garlic bread",
"beef burger with fries": "Beef burger with fries",
"seafood marinara pasta": "Seafood marinara pasta",
"creamy mushroom chicken pasta": "Creamy mushroom chicken pasta",
"grilled chicken wrap": "Grilled chicken wrap",
"smoothie bowl dessert": "Smoothie bowl",
"salted egg chicken rice chinese dish": "Salted egg chicken rice",
"chilli pan mee chinese dish": "Chilli pan mee",
"economy bee hoon chinese dish": "Economy bee hoon",
"ayam percik with rice": "Ayam percik with rice",
"nasi goreng pattaya malaysia": "Nasi goreng pattaya",
"nasi goreng usa malaysia": "Nasi goreng USA",
"roti tisu dessert": "Roti tisu",
"roti planta food": "Roti planta",
"cheese naan food": "Cheese naan",
"puri with potato masala": "Puri with potato masala",
"ramen miso food": "Ramen, miso",
"ramen shio food": "Ramen, shio",
"ramen spicy miso": "Ramen, spicy miso",
"ramen tantanmen food": "Ramen, tantanmen",
"ramen chicken paitan": "Ramen, chicken paitan",
"tsukemen dipping ramen": "Tsukemen (dipping ramen)",
"extra chashu food": "Extra chashu",
"ajitama ramen egg": "Ajitama (ramen egg)",
"shin ramyun food": "Shin Ramyun",
"shin ramyun with egg and cheese": "Shin Ramyun with egg and cheese",
"buldak hot chicken ramen": "Buldak hot chicken ramen",
"buldak carbonara ramen": "Buldak carbonara ramen",
"jjapaguri ram-don food": "Jjapaguri (ram-don)",
"kimchi ramyeon food": "Kimchi ramyeon",
"seafood ramyeon jjamppong style": "Seafood ramyeon (jjamppong style)",
"cheese ramyeon food": "Cheese ramyeon",
"instant noodles curry maggi": "Instant noodles, curry (Maggi)",
"indomie mi goreng malaysia": "Indomie mi goreng",
"spaghetti aglio olio": "Spaghetti aglio olio",
"spaghetti aglio olio with prawns": "Spaghetti aglio olio with prawns",
"spaghetti pomodoro food": "Spaghetti pomodoro",
"spaghetti arrabbiata food": "Spaghetti arrabbiata",
"spaghetti pesto with chicken": "Spaghetti pesto with chicken",
"spaghetti alfredo food": "Spaghetti alfredo",
"spaghetti vongole food": "Spaghetti vongole",
"spaghetti meatballs food": "Spaghetti meatballs",
"truffle mushroom pasta": "Truffle mushroom pasta",
"salted egg pasta": "Salted egg pasta",
"tom yum pasta": "Tom yum pasta",
"penne arrabbiata with chicken": "Penne arrabbiata with chicken",
"baked cheese pasta": "Baked cheese pasta",
"mac and cheese with chicken": "Mac and cheese with chicken",
"fettuccine carbonara with mushroom": "Fettuccine carbonara with mushroom",
"ravioli in tomato sauce": "Ravioli in tomato sauce",
"pizza margherita food": "Pizza, margherita",
"pizza pepperoni food": "Pizza, pepperoni",
"pizza hawaiian chicken": "Pizza, Hawaiian chicken",
"pizza bbq chicken": "Pizza, BBQ chicken",
"pizza meat lovers": "Pizza, meat lovers",
"pizza veggie food": "Pizza, veggie",
"pizza seafood tuna or prawn": "Pizza, seafood (tuna or prawn)",
"pizza thin crust": "Pizza, thin crust",
"pizza cheese stuffed crust": "Pizza, cheese stuffed crust",
"nasi goreng cina chinese dish": "Nasi goreng cina",
"nasi goreng ikan masin chinese dish": "Nasi goreng ikan masin",
"nasi goreng tom yum": "Nasi goreng tom yum",
"nasi goreng ayam malaysia": "Nasi goreng ayam",
"nasi goreng seafood malaysia": "Nasi goreng seafood",
"nasi goreng belacan malaysia": "Nasi goreng belacan",
"nasi goreng daging merah": "Nasi goreng daging merah",
"yang chow fried rice chinese dish": "Yang chow fried rice",
"garlic fried rice": "Garlic fried rice",
"nasi lemak rendang ayam": "Nasi lemak rendang ayam",
"nasi lemak sambal sotong": "Nasi lemak sambal sotong",
"nasi lemak telur mata": "Nasi lemak telur mata",
"nasi lemak kerang malaysia": "Nasi lemak kerang",
"roti bawang food": "Roti bawang",
"roti telur bawang": "Roti telur bawang",
"roti pisang dessert": "Roti pisang",
"roti sardin food": "Roti sardin",
"roti bom dessert": "Roti bom",
"roti cheese food": "Roti cheese",
"thosai telur food": "Thosai telur",
"rava thosai food": "Rava thosai",
"paper thosai food": "Paper thosai",
"garlic naan food": "Garlic naan",
"plain naan food": "Plain naan",
"chicken burger crispy": "Chicken burger, crispy",
"fish burger food": "Fish burger",
"double beef burger": "Double beef burger",
"mushroom swiss burger": "Mushroom Swiss burger",
"ramly burger biasa malaysia": "Ramly burger, biasa",
"ramly burger ayam special": "Ramly burger, ayam special",
"veggie burger food": "Veggie burger",
"tuna nigiri food": "Tuna nigiri",
"tamago nigiri food": "Tamago nigiri",
"unagi nigiri food": "Unagi nigiri",
"inari sushi food": "Inari sushi",
"salmon maki food": "Salmon maki",
"kappa maki cucumber": "Kappa maki (cucumber)",
"spicy salmon roll": "Spicy salmon roll",
"fried ebi roll": "Fried ebi roll",
"salmon mentai aburi": "Salmon mentai aburi",
"salmon sashimi food": "Salmon sashimi",
"chirashi don food": "Chirashi don",
"cheese foam tea": "Cheese foam tea",
"pho chicken food": "Pho, chicken",
"bun bo hue": "Bun bo Hue",
"banh mi food": "Banh mi",
"vietnamese fresh spring rolls": "Vietnamese fresh spring rolls",
"udon kitsune food": "Udon, kitsune",
"udon beef niku": "Udon, beef (niku)",
"udon curry food": "Udon, curry",
"yaki udon food": "Yaki udon",
"fish curry with rice": "Fish curry with rice",
"mutton curry with rice": "Mutton curry with rice",
"vegetable curry with rice": "Vegetable curry with rice",
"japanese curry rice chicken": "Japanese curry rice, chicken",
"japanese curry rice beef": "Japanese curry rice, beef",
"massaman curry with rice": "Massaman curry with rice",
"red curry chicken with rice": "Red curry chicken with rice",
"kari kepala ikan": "Kari kepala ikan",
"fish porridge food chinese dish": "Fish porridge",
"pork porridge food chinese dish": "Pork porridge",
"plain porridge with side dishes teochew": "Plain porridge with side dishes (Teochew)",
"bubur lambuk food malaysia": "Bubur lambuk",
"oat porridge savoury": "Oat porridge, savoury",
"scrambled eggs food": "Scrambled eggs",
"omelette plain food": "Omelette, plain",
"omelette with onion and chilli telur dadar": "Omelette with onion and chilli (telur dadar)",
"poached eggs food": "Poached eggs",
"half-boiled eggs food": "Half-boiled eggs",
"telur bungkus food malaysia": "Telur bungkus",
"salted egg food chinese dish": "Salted egg",
"century egg food chinese dish": "Century egg",
"tuna sandwich food": "Tuna sandwich",
"chicken sandwich grilled": "Chicken sandwich, grilled",
"ham and cheese sandwich": "Ham and cheese sandwich",
"grilled cheese sandwich": "Grilled cheese sandwich",
"blt sandwich food": "BLT sandwich",
"caesar salad no chicken": "Caesar salad, no chicken",
"greek salad food": "Greek salad",
"grilled chicken salad": "Grilled chicken salad",
"tuna salad food": "Tuna salad",
"poke bowl salmon": "Poke bowl, salmon",
"ayam goreng berempah malaysia": "Ayam goreng berempah",
"ayam goreng kunyit malaysia": "Ayam goreng kunyit",
"chicken tenders food": "Chicken tenders",
"chicken wings fried": "Chicken wings, fried",
"roast chicken quarter": "Roast chicken, quarter",
"nasi minyak food malaysia": "Nasi minyak",
"nasi tomato food malaysia": "Nasi tomato",
"nasi briyani rice only": "Nasi briyani, rice only",
"sticky rice food": "Sticky rice",
"japanese rice food": "Japanese rice",
"cauliflower rice food": "Cauliflower rice",
"butter toast food": "Butter toast",
"peanut butter toast": "Peanut butter toast",
"garlic toast food": "Garlic toast",
"cadbury dairy milk whole bar": "Cadbury Dairy Milk, whole bar",
"tim tam original": "Tim Tam Original",
"tim tam double coat": "Tim Tam Double Coat",
"tim tam chewy caramel": "Tim Tam Chewy Caramel",
"tim tam white": "Tim Tam White",
"twix dessert": "Twix",
"mars dessert": "Mars",
"hershey's kisses dessert": "Hershey's Kisses",
"lindt lindor dessert": "Lindt Lindor",
"hello panda strawberry": "Hello Panda Strawberry",
"yan yan strawberry": "Yan Yan Strawberry",
"koala's march strawberry": "Koala's March Strawberry",
"chipsmore original dessert": "Chipsmore Original",
"julie's peanut butter sandwich": "Julie's Peanut Butter Sandwich",
"julie's cheese sandwich": "Julie's Cheese Sandwich",
"lexus chocolate cream sandwich": "Lexus Chocolate Cream Sandwich",
"munchy's oat krunch strawberry blackcurrant": "Munchy's Oat Krunch Strawberry & Blackcurrant",
"lay's classic food": "Lay's Classic",
"lay's sour cream onion": "Lay's Sour Cream & Onion",
"lay's nori seaweed": "Lay's Nori Seaweed",
"lay's hot spicy": "Lay's Hot & Spicy",
"doritos nacho cheese": "Doritos Nacho Cheese",
"calbee jagabee food": "Calbee Jagabee",
"want want senbei": "Want Want Senbei",
"want want shelly senbei": "Want Want Shelly Senbei",
"ding dang dessert": "Ding Dang",
"mentos dessert": "Mentos",
"hacks dessert": "Hacks",
"tic tac dessert": "Tic Tac",
"haw flakes dessert": "Haw flakes",
"white rabbit candy": "White Rabbit candy",
"gummy bears dessert": "Gummy bears",
"ritz crackers food": "Ritz crackers",
"popo fish muruku": "Popo Fish Muruku",
"sauce bowl sesame and garlic": "Haidilao sauce bowl, sesame and garlic",
"macaroni soup with ham chinese dish": "Kim Gary macaroni soup with ham",
"bak kwa pork chinese dish": "Bee Cheng Hiang bak kwa, pork",
"kacang putih mix": "Kacang putih mix",
"hummus with bread": "Al Halabi hummus with bread",
"chinese set food": "D Empire Chinese set, per person",
"rice bowl food chinese dish": "Cloud 9 rice bowl",
"tapas food": "Las Meninas tapas, per person",
"latin sharing plates": "Amazonas Latin sharing plates, per person",
"tasting dish food": "Cuura tasting dish, per person",
"snack pack food": "Yoursnackies snack pack",
"fruit cup dessert": "Fresh Delights fruit cup",
"fresh oysters food": "Shucked fresh oysters",
"chicken tikka masala": "Masala King chicken tikka masala",
"vegetarian rice bowl": "Kita vegetarian rice bowl",
"roast meat rice chinese dish": "Peony roast meat rice",
"ayam masak merah with rice": "Ayam masak merah with rice",
"sambal kangkung food malaysia": "Sambal kangkung",
"steamed siakap with ginger and soy rice and greens": "Steamed siakap with ginger and soy, rice and greens",
"chicken curry with potatoes and rice": "Chicken curry with potatoes and rice",
"dhal curry with vegetables and rice": "Dhal curry with vegetables and rice",
"mee goreng food malaysia": "Mee goreng, home-style",
"chicken porridge with ginger chinese dish": "Chicken porridge with ginger",
"bee hoon soup with fish balls and greens": "Bee hoon soup with fish balls and greens",
"kangkung belacan with tofu and rice": "Kangkung belacan with tofu and rice",
"sweet and sour chicken with rice": "Sweet and sour chicken with rice",
"pad kra pao chicken": "Pad kra pao chicken, home-style",
"salmon teriyaki with rice and broccoli": "Salmon teriyaki with rice and broccoli",
"kimchi fried rice with egg 2": "Kimchi fried rice with egg, home-style",
"korean army stew budae jjigae at home": "Korean army stew (budae jjigae) at home",
"aglio olio with prawns": "Aglio olio with prawns",
"chicken and vegetable soup with bread": "Chicken and vegetable soup with bread",
"abc soup with pork ribs": "ABC soup with pork ribs",
"steamed egg with minced pork and rice": "Steamed egg with minced pork and rice",
"tofu and minced chicken with long beans": "Tofu and minced chicken with long beans",
"overnight oats with chia and banana": "Overnight oats with chia and banana",
"greek yogurt bowl with berries and granola": "Greek yogurt bowl with berries and granola",
"egg and tomato stir-fry with rice": "Egg and tomato stir-fry with rice",
"chicken breast salad wrap": "Chicken breast salad wrap",
"baked salmon with sweet potato and greens": "Baked salmon with sweet potato and greens",
"tempeh and vegetable sambal with rice": "Tempeh and vegetable sambal with rice",
"miso salmon and tofu soup with rice": "Miso salmon and tofu soup with rice",
"butter garlic prawns with rice": "Butter garlic prawns with rice",
"dumplings in soup with greens": "Dumplings in soup with greens",
"quinoa chicken bowl": "Quinoa chicken bowl",
"ayam goreng berempah air fryer": "Ayam goreng berempah, air fryer",
"spinach and egg drop soup with rice": "Spinach and egg drop soup with rice",
"lotus root and peanut soup with pork ribs": "Lotus root and peanut soup with pork ribs",
"watercress soup with lean pork": "Watercress soup with lean pork",
"old cucumber soup with pork ribs": "Old cucumber soup with pork ribs",
"winter melon soup with dried scallops": "Winter melon soup with dried scallops",
"green papaya fish soup chinese dish": "Green papaya fish soup",
"lotus root soup vegetarian chinese dish": "Lotus root soup, vegetarian",
"corn and carrot soup with chicken": "Corn and carrot soup with chicken",
"white radish soup with pork ribs": "White radish soup with pork ribs",
"black bean soup with chicken": "Black bean soup with chicken",
"seaweed and tofu soup chinese dish": "Seaweed and tofu soup",
"luffa and egg soup chinese dish": "Luffa and egg soup",
"bitter gourd soup with lean pork": "Bitter gourd soup with lean pork",
"chinese cabbage and fish ball soup": "Chinese cabbage and fish ball soup",
"tomato and egg soup chinese dish": "Tomato and egg soup",
"chayote and carrot soup with chicken": "Chayote and carrot soup with chicken",
"pumpkin soup food": "Pumpkin soup",
"mixed vegetable minestrone": "Mixed vegetable minestrone",
"spinach and tofu clear soup": "Spinach and tofu clear soup",
"mushroom clear soup chinese dish": "Mushroom clear soup",
"broccoli and cauliflower soup": "Broccoli and cauliflower soup",
"steamed chicken breast with ginger brown rice and choy sum": "Steamed chicken breast with ginger, brown rice and choy sum",
"konjac noodle soup with prawns and greens": "Konjac noodle soup with prawns and greens",
"cauliflower fried rice with egg and chicken": "Cauliflower fried rice with egg and chicken",
"egg white omelette with spinach and mushrooms": "Egg white omelette with spinach and mushrooms",
"grilled fish with salad and sweet potato": "Grilled fish with salad and sweet potato",
"chicken breast salad bowl": "Chicken breast salad bowl",
"tofu and vegetable stir-fry no rice": "Tofu and vegetable stir-fry, no rice",
"soba with chicken and edamame": "Soba with chicken and edamame",
"lettuce wraps with minced chicken": "Lettuce wraps with minced chicken",
"steamed fish with tofu and spring onion no rice": "Steamed fish with tofu and spring onion, no rice",
"overnight oats with greek yogurt and berries": "Overnight oats with Greek yogurt and berries, light",
"cottage cheese toast with tomato": "Cottage cheese toast with tomato",
"vietnamese prawn rice paper rolls": "Vietnamese prawn rice paper rolls",
"zucchini noodles with prawns and tomato": "Zucchini noodles with prawns and tomato",
"chicken and vegetable clear soup with konjac rice": "Chicken and vegetable clear soup with konjac rice",
"grilled lean beef with asparagus and mash": "Grilled lean beef with asparagus and mash",
"kimchi tofu stew": "Kimchi tofu stew, light",
"chickpea and cucumber salad": "Chickpea and cucumber salad",
"ulam with sambal and grilled chicken": "Ulam with sambal and grilled chicken, light",
"air-fryer tofu with broccoli and brown rice": "Air-fryer tofu with broccoli and brown rice",
"prawn and egg white fried cauliflower rice": "Prawn and egg white fried cauliflower rice, light",
"abc soup with chicken and rice": "ABC soup with chicken and rice",
"steamed egg with minced chicken rice": "Steamed egg with minced chicken, rice",
"tuna and egg salad wrap": "Tuna and egg salad wrap",
"chicken and vegetable pasta": "Chicken and vegetable pasta",
"miso soup with tofu rice and egg": "Miso soup with tofu, rice and egg",
"sardine sandwich food malaysia": "Sardine sandwich",
"banana dessert": "Banana",
"papaya dessert": "Papaya",
"apple dessert": "Apple",
"guava food": "Guava",
"watermelon dessert": "Watermelon",
"orange dessert": "Orange",
"mixed cut fruit": "Mixed cut fruit",
"blanched greens with oyster sauce": "Blanched greens with oyster sauce",
"ulam with sambal on the side": "Ulam with sambal on the side",
"corn cup food": "Corn cup",
"chawanmushi food": "Chawanmushi",
"unsweetened soy milk": "Unsweetened soy milk",
"low-fat milk drink": "Low-fat milk",
"mixed nuts food": "Mixed nuts",
"steamed corn food": "Steamed corn",
"popiah 1 roll chinese dish": "Popiah, 1 roll",
"oden food": "Oden",
"kaya bun dessert chinese dish": "Kaya bun",
"red bean bun chinese dish": "Red bean bun",
"limau ais kurang manis": "Limau ais kurang manis",
"sirap bandung drink": "Sirap bandung",
"fresh coconut water": "Fresh coconut water",
"fresh orange juice no sugar": "Fresh orange juice, no sugar",
"brown rice food": "Brown rice",
"grilled chicken breast": "Grilled chicken breast",
"economy rice food chinese dish": "Economy rice (Home Kitchen)",
"chicken chop food": "Chicken chop (Steven's Corner)",
"banana leaf rice 2": "Banana leaf rice (Kanna Curry House)",
"roasted chicken rice chinese dish": "Roasted chicken rice (Choon Yien)",
"fried chicken with curry rice": "Fried chicken with curry rice (Lim Fried Chicken)",
"hainan chicken rice chinese dish": "Hainan chicken rice (Hometown Hainan Coffee)",
"wantan mee food chinese dish": "Wantan mee (PJ Old Town)",
"yong tau foo chinese dish": "Yong tau foo (PJ Old Town)",
"chee cheong fun chinese dish": "Chee cheong fun (SS2 morning market)",
"hailam mee food chinese dish": "Hailam mee (Yut Kee)",
"beef noodles food chinese dish": "Beef noodles (Soong Kee)",
"banana leaf rice 3": "Banana leaf rice (Sri Nirwana Maju)",
"poached chicken rice chinese dish": "Poached chicken rice (Chicken Rice Hub)",
"karaage don food": "Karaage don (Hungry Don)",
"food court meal chinese dish": "Economy rice: 1 meat, 2 veg",
"burger food": "Burger (myBurgerLab)",
"japanese set food": "Japanese set (Hana Dining)",
"ensure gold drink": "Ensure Gold",
"anlene gold 5x": "Anlene Gold 5X",
"chobani fit strawberry": "Chobani Fit, strawberry",
"onigiri salmon food": "FamilyMart onigiri, salmon",
"onigiri chicken teriyaki": "FamilyMart onigiri, chicken teriyaki",
"onigiri spicy tuna": "FamilyMart onigiri, spicy tuna",
"oden fish ball stick": "FamilyMart oden, fish ball stick",
"oden radish food": "FamilyMart oden, radish",
"oden egg food": "FamilyMart oden, egg",
"oden tofu puff": "FamilyMart oden, tofu puff",
"oden crab stick": "FamilyMart oden, crab stick",
"oden chikuwa food": "FamilyMart oden, chikuwa",
"oden udon noodles": "FamilyMart oden, udon noodles",
"egg mayo sando": "FamilyMart egg mayo sando",
"tuna mayo sando": "FamilyMart tuna mayo sando",
"chicken katsu sando": "FamilyMart chicken katsu sando",
"nasi lemak food malaysia": "FamilyMart nasi lemak",
"mac and cheese": "FamilyMart mac and cheese",
"onigiri tuna mayo": "7-Eleven onigiri, tuna mayo",
"onigiri chicken food": "7-Eleven onigiri, chicken",
"tuna sandwich food 2": "7-Eleven tuna sandwich",
"chicken slice sandwich": "7-Eleven chicken slice sandwich",
"nasi lemak food malaysia 2": "7-Eleven nasi lemak",
"fried rice food": "7-Eleven fried rice",
"mashed potato with gravy": "7-Eleven mashed potato with gravy",
"chicken pau food chinese dish": "7-Eleven chicken pau",
"maru onigiri tuna mayo": "myNEWS Maru onigiri, tuna mayo",
"maru sandwich egg mayo": "myNEWS Maru sandwich, egg mayo",
"maru bento chicken": "myNEWS Maru bento, chicken",
"nasi lemak bungkus malaysia": "KK Mart nasi lemak bungkus",
"iced lemon tea packet": "Iced lemon tea, packet",
"soft drink zero sugar": "Soft drink, zero sugar",
"cham ais drink": "Cham ais",
"neslo ais drink": "Neslo ais",
"horlicks ais drink": "Horlicks ais",
"barley drink sweetened": "Barley drink, sweetened",
"soya cincau drink": "Soya cincau",
"air mata kucing": "Air mata kucing",
"masala tea drink": "Masala tea",
"fruit tea less sugar": "Fruit tea, less sugar",
"yakult green tea": "Yakult green tea",
"gym junkie protein smoothie": "Gym Junkie protein smoothie",
"llaollao tub with 1 topping": "llaollao, small tub with 1 topping",
"sanum dessert": "llaollao Sanum",
"sago gula melaka malaysia": "Sago gula melaka",
"muah chee dessert chinese dish": "Muah chee",
"kaya puff dessert chinese dish": "Kaya puff",
"chicken floss bun chinese dish": "Chicken floss bun",
"seaweed snack food": "Seaweed snack",
"chicken siew mai chinese dish": "Chicken siew mai",
"steamed fish balls chinese dish": "Steamed fish balls",
"vegetable dumplings steamed chinese dish": "Vegetable dumplings, steamed",
"beancurd skin roll steamed chinese dish": "Beancurd skin roll, steamed",
"beancurd skin roll fried chinese dish": "Beancurd skin roll, fried",
"char siu cheong fun chinese dish": "Char siu cheong fun",
"ham sui kok chinese dish": "Ham sui kok",
"fried prawn dumplings with mayo": "Fried prawn dumplings with mayo",
"sui kow in soup chinese dish": "Sui kow in soup",
"lotus paste bun chinese dish": "Lotus paste bun",
"chicken bao food chinese dish": "Chicken bao",
"big bao dai bao chinese dish": "Big bao (dai bao)",
"stuffed tofu and brinjal chinese dish": "Stuffed tofu and brinjal",
"butter cake dessert": "Butter cake",
"tuna puff food": "Tuna puff",
"sausage bun food": "Sausage bun",
"tuna mayo bun": "Tuna mayo bun",
"curry chicken bun": "Curry chicken bun",
"cheese bun food": "Cheese bun",
"pretzel cinnamon sugar": "Pretzel, cinnamon sugar",
"longan and sea coconut chinese dish": "Longan and sea coconut",
"lemonade drink": "Mixue lemonade",
"soda limau asam boi": "Soda limau asam boi",
"air katira drink malaysia": "Air katira",
"strawberry shortcake dessert": "Strawberry shortcake",
"durian crepe dessert": "Durian crepe",
"mango crepe dessert": "Mango crepe",
"dorayaki dessert": "Dorayaki",
"castella cake dessert": "Castella cake",
"baumkuchen dessert": "Baumkuchen",
"kek batik dessert malaysia": "Kek batik",
"kek lapis sarawak malaysia": "Kek lapis Sarawak",
"kek lapis legit malaysia": "Kek lapis legit",
"pandan layer cake malaysia": "Pandan layer cake",
"ondeh-ondeh cake dessert malaysia": "Ondeh-ondeh cake",
"cendol cake dessert malaysia": "Cendol cake",
"chocolate eclair dessert": "Chocolate eclair",
"canele dessert": "Canele",
"madeleine dessert": "Madeleine",
"financier dessert": "Financier",
"maritozzo cream bun": "Maritozzo cream bun",
"baked cheese tart": "Baked cheese tart",
"banoffee pie dessert": "Banoffee pie",
"pecan pie dessert": "Pecan pie",
"pavlova with cream and fruit": "Pavlova with cream and fruit",
"mont blanc dessert": "Mont Blanc",
"blondie dessert": "Blondie",
"braised chicken with potatoes chinese dish": "Braised chicken with potatoes",
"braised chicken with potatoes and rice": "Braised chicken with potatoes and rice",
"steamed minced pork patty chinese dish": "Steamed minced pork patty",
"steamed minced pork with salted fish": "Steamed minced pork with salted fish",
"steamed minced pork with salted egg": "Steamed minced pork with salted egg",
"steamed minced pork with preserved vegetable": "Steamed minced pork with preserved vegetable",
"steamed minced pork patty with rice": "Steamed minced pork patty with rice",
"steamed egg custard chinese dish": "Steamed egg custard",
"three-colour steamed egg chinese dish": "Three-colour steamed egg",
"tomato egg stir-fry chinese dish": "Tomato egg stir-fry",
"sour fish soup suan cai fish": "Sour fish soup (suan cai fish)",
"sour fish soup with rice": "Sour fish soup with rice",
"salted vegetable and tofu soup": "Salted vegetable and tofu soup",
"salted vegetable duck soup chinese dish": "Salted vegetable duck soup",
"herbal chicken soup chinese dish": "Herbal chicken soup",
"chai boey mustard greens stew": "Chai boey (mustard greens stew)",
"vinegar pork trotter chinese dish": "Vinegar pork trotter",
"mushroom chicken stew chinese dish": "Mushroom chicken stew",
"soy sauce chicken economy rice": "Soy sauce chicken (economy rice)",
"ginger and spring onion chicken": "Ginger and spring onion chicken",
"steamed chicken with ginger and wolfberries": "Steamed chicken with ginger and wolfberries",
"ginger and onion pork chinese dish": "Ginger and onion pork",
"black pepper pork chinese dish": "Black pepper pork",
"braised pork with yam chinese dish": "Braised pork with yam",
"fried pork cutlet economy rice": "Fried pork cutlet (economy rice)",
"braised chicken feet with mushrooms": "Braised chicken feet with mushrooms",
"potato with minced pork chinese dish": "Potato with minced pork",
"french beans with minced pork": "French beans with minced pork",
"braised mushrooms with tofu chinese dish": "Braised mushrooms with tofu",
"hot plate tofu chinese dish": "Hot plate tofu",
"fried tofu with sweet chilli": "Fried tofu with sweet chilli",
"braised cabbage with glass noodles": "Braised cabbage with glass noodles",
"stir-fried long beans chinese dish": "Stir-fried long beans",
"stir-fried kai lan with garlic": "Stir-fried kai lan with garlic",
"stir-fried sweet potato leaves chinese dish": "Stir-fried sweet potato leaves",
"lady's fingers with sambal": "Lady's fingers with sambal",
"curry fish economy rice chinese dish": "Curry fish (economy rice)",
"sweet and sour fish chinese dish": "Sweet and sour fish",
"black bean fish chinese dish": "Black bean fish",
"soy sauce fried fish chinese dish": "Soy sauce fried fish",
"asam pedas fish malaysia": "Asam pedas fish, 1 piece",
"cereal prawns food chinese dish": "Cereal prawns",
"curry squid food chinese dish": "Curry squid",
"mille crepe original": "Mille crepe, original",
"ice cream cone packaged": "Ice cream cone, packaged",
"grapes purple seedless": "Grapes, purple seedless",
"grapes green seedless": "Grapes, green seedless",
"grapes red fruit": "Grapes, red",
"strawberries fruit": "Strawberries",
"blueberries fruit": "Blueberries",
"raspberries fruit": "Raspberries",
"mixed berries fruit": "Mixed berries",
"cherries fruit": "Cherries",
"mango fruit": "Mango",
"pineapple fruit": "Pineapple",
"honeydew fruit": "Honeydew",
"rock melon fruit": "Rock melon",
"dragon fruit red": "Dragon fruit, red",
"dragon fruit white": "Dragon fruit, white",
"kiwi fruit": "Kiwi",
"pear fruit": "Pear",
"chinese pear fruit": "Chinese pear",
"green apple fruit": "Green apple",
"peach fruit": "Peach",
"plum fruit": "Plum",
"persimmon fruit": "Persimmon",
"pomegranate fruit": "Pomegranate",
"mandarin orange fruit": "Mandarin orange",
"pomelo fruit": "Pomelo",
"grapefruit fruit": "Grapefruit",
"rambutan fruit": "Rambutan",
"longan fruit": "Longan",
"lychee fruit": "Lychee",
"mangosteen fruit": "Mangosteen",
"duku langsat fruit": "Duku langsat",
"jackfruit nangka fruit": "Jackfruit (nangka)",
"cempedak fruit": "Cempedak",
"ciku sapodilla fruit": "Ciku (sapodilla)",
"starfruit fruit": "Starfruit",
"jambu air rose apple": "Jambu air (rose apple)",
"soursop fruit": "Soursop",
"avocado fruit": "Avocado",
"pisang emas fruit": "Pisang emas",
"fig fresh fruit": "Fig, fresh",
"cherry tomatoes fruit": "Cherry tomatoes",
"dates fruit": "Dates",
"raisins fruit": "Raisins",
"prunes fruit": "Prunes",
"dried mango fruit": "Dried mango",
"chicken wing skewer street food": "Chicken wing skewer",
"squid skewer street food": "Squid skewer",
"prawn skewer street food": "Prawn skewer",
"sausage skewer street food": "Sausage skewer",
"grilled mantou skewer street food": "Grilled mantou skewer",
"grilled corn skewer street food": "Grilled corn skewer",
"king oyster mushroom skewer street food": "King oyster mushroom skewer",
"sate padang malaysia street food": "Sate padang",
"sate lilit malaysia street food": "Sate lilit",
"dak-kkochi chicken skewer street food": "Dak-kkochi (chicken skewer)",
"tteok-kkochi rice cake skewer street food": "Tteok-kkochi (rice cake skewer)",
"sotteok sotteok street food": "Sotteok sotteok",
"eomuk fish cake skewer street food": "Eomuk (fish cake skewer)",
"shish tawook 1 skewer street food": "Shish tawook, 1 skewer",
"kofta kebab 1 skewer street food": "Kofta kebab, 1 skewer",
"mala skewers chuan chuan street food": "Mala skewers (chuan chuan)"
}
V9_PER_QUERY = 45
# photos that teach only the parts (one base or one protein by itself), not a dish
PART_ONLY = {
"plain yellow noodles bowl": [
"yellow noodles"
],
"cooked yellow noodles mee": [
"yellow noodles"
],
"plain egg noodles wonton noodles": [
"egg noodles"
],
"thin egg noodles cooked": [
"egg noodles"
],
"plain bee hoon rice vermicelli cooked": [
"bee hoon"
],
"rice vermicelli bowl": [
"bee hoon"
],
"plain kway teow flat rice noodles": [
"kway teow"
],
"hor fun flat noodles": [
"kway teow"
],
"plain udon noodles": [
"flour noodles"
],
"plain ramen noodles": [
"ramen"
],
"instant noodles cooked bowl": [
"instant noodles"
],
"glass noodles tang hoon cooked": [
"glass noodles"
],
"mee sua vermicelli": [
"mee sua"
],
"plain spaghetti": [
"pasta"
],
"rice porridge plain congee": [
"porridge"
],
"fried chicken pieces plate": [
"chicken"
],
"steamed chicken slices": [
"chicken"
],
"char siu slices": [
"pork"
],
"roast pork siu yuk": [
"pork"
],
"black pepper beef stir fry": [
"beef"
],
"beef rendang": [
"beef"
],
"mutton curry": [
"mutton"
],
"roast duck slices": [
"duck"
],
"fried ikan kembung": [
"fish"
],
"butter prawns": [
"prawn"
],
"fried calamari": [
"squid"
],
"sambal sotong": [
"squid"
],
"fried egg sunny side up": [
"egg"
],
"half boiled eggs": [
"egg"
],
"plain omelette": [
"egg"
],
"fried tofu": [
"tofu"
],
"fish balls bowl": [
"fish ball"
],
"fried fish cake slices": [
"fish ball"
],
"fried ikan bilis peanuts": [
"anchovies"
],
"economy rice plate chicken vegetables": [
"white rice",
"chicken",
"vegetables"
],
"economy rice fish tofu vegetables": [
"white rice",
"fish",
"tofu",
"vegetables"
],
"economy rice egg vegetables": [
"white rice",
"egg",
"vegetables"
],
"nasi campur fried chicken egg": [
"white rice",
"chicken",
"egg"
],
"chap fan pork vegetables": [
"white rice",
"pork",
"vegetables"
]
}
PARTS_WEIGHT = 0.5


## Load the photos

Both datasets come from the Hugging Face Hub. Food-101 is large (about 5 GB), so this step takes a few minutes.

In [ ]:
my = load_dataset("lowyisan/malaysian_food_images")
f101 = load_dataset("ethz/food101")

my_names = my["train"].features["label"].names
f_names = f101["train"].features["label"].names

try:
    india = load_dataset("rajistics/indian_food_images")
    in_names = india["train"].features["label"].names
except Exception as e:                                  # optional extra data: carry on without it
    india = None; MAP_IN = {}; print("Skipping the Indian food dataset:", e)


# ---- extra Malaysian hawker and home dishes from Wikimedia Commons ----
# Photos on Wikimedia Commons are published under licences that allow reuse, and the site offers an API for this.
# Search results are noisy (not every photo shows the dish), so expect lower accuracy for these classes.
import os, re, io, time, requests
COMMONS = {
"ayam masak kicap": "Ayam masak kicap with rice and sayur",
"kari ayam": "Chicken curry with rice",
"steamed fish chinese": "Steamed fish with rice and greens",
"kangkung belacan stir fried": "Stir-fried greens",
"dal curry": "Dhal curry",
"roast chicken potatoes home": "Baked chicken thigh with potatoes and salad",
"braised tofu": "Tofu, braised",
"tempeh goreng": "Tempeh, fried",
"char kway teow": "Char kway teow / Kway teow goreng",
"bak kut teh": "Bak kut teh with rice",
"hainanese chicken rice": "Chicken rice, steamed",
"nasi briyani": "Nasi briyani ayam",
"dosa": "Thosai with dhal and chutney",
"chapati": "Chapati with dhal",
"idli": "Idli with sambar",
"wonton noodles": "Wantan mee, dry",
"asam laksa": "Asam laksa",
"hokkien mee kuala lumpur": "Hokkien mee (KL)",
"mee rebus": "Mee rebus",
"lontong": "Lontong",
"nasi kandar": "Nasi kandar",
"murtabak": "Murtabak ayam",
"cendol": "Cendol",
"ais kacang": "Ais kacang (ABC)",
"apam balik": "Apam balik",
"pisang goreng": "Pisang goreng",
"curry puff": "Curry puff",
"teh tarik": "Teh tarik",
"rojak buah": "Rojak buah",
"yong tau foo": "Yong tau foo, soup",
"claypot chicken rice": "Claypot chicken rice",
"chee cheong fun": "Chee cheong fun, plain",
"prawn mee": "Prawn mee",
"pan mee": "Pan mee, soup",
"nasi dagang": "Nasi dagang",
"soto ayam": "Soto ayam",
"ikan bakar": "Ikan bakar with rice and ulam",
"tandoori chicken": "Tandoori chicken with naan",
"kuih": "Kuih, 2 pieces",
"roti telur": "Roti telur",
"mee siam": "Mee siam",
"dim sum": "Dim sum breakfast",
"char siu bao": "Char siu bao",
"siu mai": "Siew mai",
"lo mai gai": "Lor mai kai",
"ayam penyet": "Nasi ayam penyet",
"fried egg": "Fried egg",
"boiled egg": "Boiled egg",
"steamed white rice bowl": "White rice",
"douhua tofu pudding": "Tau fu fah, less syrup",
"keropok lekor": "Keropok lekor",
"sup ayam": "Sup ayam with rice",
"nasi lemak": "Nasi lemak biasa",
"roti canai": "Roti canai kosong",
"satay": "Satay ayam",
"yakult bottle": "Yakult Ace",
"vitagen cultured milk": "Vitagen",
"brown sugar boba milk": "Brown sugar pearl fresh milk",
"bubble tea": "Bubble milk tea",
"taro milk tea": "Taro milk tea with pearls",
"iced matcha latte": "Matcha latte, iced",
"chocolate milkshake": "Chocolate milkshake",
"mango lassi": "Mango lassi",
"mango smoothie": "Mango Magic smoothie",
"berry smoothie": "All Berry Bang smoothie",
"sugarcane juice": "Sugarcane juice",
"kopi ais iced coffee": "Kopi ais",
"teh ais iced milk tea": "Teh ais",
"caramel frappuccino": "Caramel frappé",
"iced chocolate drink": "Iced chocolate",
"milo ais": "Milo ais",
"watermelon juice": "Watermelon juice, no sugar",
"caffe latte": "Latte",
"americano coffee": "Americano",
"cola can": "Soft drink",
"frozen yogurt toppings": "llaollao, medium tub with 3 toppings",
"ice cream scoop cup": "Inside Scoop, single scoop",
"ice cream cone scoops": "Inside Scoop, double scoop in a waffle cone",
"waffle with ice cream": "Waffle with ice cream",
"souffle pancake": "Soufflé pancakes",
"mango bingsu": "Mango bingsu",
"injeolmi bingsu": "Injeolmi bingsu",
"chocolate bingsu": "Chocolate bingsu",
"mango pomelo sago": "Mango sago pomelo",
"red bean soup dessert": "Red bean soup",
"bubur cha cha": "Bubur cha cha",
"mango sticky rice": "Mango sticky rice",
"daifuku mochi": "Mochi",
"chocolate brownie": "Brownie",
"chocolate chip cookie": "Chocolate chip cookie",
"churros": "Churros with chocolate dip",
"banana chocolate crepe": "Crêpe with banana and chocolate",
"molten chocolate cake": "Molten lava cake with ice cream",
"mcflurry": "McFlurry with Oreo",
"chocolate sundae": "Chocolate sundae",
"tiramisu": "Tiramisu",
"cheesecake slice": "Cheesecake",
"chocolate cake slice": "Chocolate cake",
"macarons": "Macarons",
"glazed doughnut": "Doughnut",
"popcorn": "Popcorn, salted",
"potato chips": "Potato chips",
"chocolate bar": "Chocolate bar",
"granola bar": "Granola bar",
"soft serve ice cream": "Soft serve cone",
"xiaolongbao": "Xiao long bao",
"har gow": "Har gow",
"steamed spare ribs dim sum": "Steamed pork ribs with black bean",
"chicken feet dim sum": "Chicken feet in black bean sauce",
"shrimp rice noodle roll": "Prawn cheong fun",
"turnip cake dim sum": "Lo bak go, pan-fried radish cake",
"taro dumpling wu gok": "Yam puff (wu kok)",
"fried wonton": "Fried wantan",
"shengjian mantou": "Sheng jian bao",
"liu sha bao custard bun": "Salted egg custard bun (lau sar bao)",
"ma lai go sponge cake": "Ma lai gao",
"jian dui sesame ball": "Sesame balls (jin deui)",
"char siu sou pastry": "Char siu sou",
"mango pudding": "Mango pudding",
"century egg congee": "Century egg and pork porridge",
"zongzi": "Bak chang",
"egg tart": "Egg tart",
"fried spring rolls": "Spring rolls, fried",
"gyoza": "Gyoza",
"fried mantou condensed milk": "Fried mantou with condensed milk",
"croissant": "Butter croissant",
"almond croissant": "Almond croissant",
"pain au chocolat": "Pain au chocolat",
"danish pastry": "Fruit Danish",
"cinnamon roll": "Cinnamon roll",
"blueberry muffin": "Blueberry muffin",
"chocolate muffin": "Chocolate muffin",
"banana bread": "Banana bread",
"scone jam cream": "Scone with jam and cream",
"cream puff": "Cream puff",
"chocolate eclair": "Chocolate éclair",
"swiss roll cake": "Swiss roll",
"pandan chiffon cake": "Pandan chiffon cake",
"red velvet cake": "Red velvet cake",
"carrot cake slice": "Carrot cake with cream cheese",
"basque burnt cheesecake": "Burnt cheesecake",
"mille crepe cake": "Mille crêpe",
"fruit tart": "Fruit tart",
"lemon tart": "Lemon tart",
"pastel de nata": "Portuguese egg tart",
"pineapple tart": "Pineapple tarts",
"mooncake": "Mooncake, lotus paste",
"taiyaki": "Taiyaki",
"sausage roll": "Chicken sausage roll",
"chicken pie": "Chicken pie",
"quiche": "Quiche",
"soft pretzel": "Pretzel, original",
"bagel cream cheese": "Bagel with cream cheese",
"pineapple bun bolo bao": "Polo bun with butter",
"coffee bun rotiboy": "Mexican coffee bun",
"sliced white bread": "White bread",
"cupcake": "Cupcake",
"apple pie": "Apple pie",
"waffle syrup": "Waffle with syrup",
"pancakes syrup": "Pancakes with syrup",
"green bean soup dessert": "Green bean soup",
"black sesame soup": "Black sesame paste",
"peanut soup dessert chinese": "Peanut paste",
"walnut soup dessert chinese": "Walnut paste",
"almond tea dessert chinese": "Almond paste",
"sweet potato soup ginger dessert": "Sweet potato ginger soup",
"tangyuan": "Tang yuan in ginger syrup",
"snow fungus soup": "Snow fungus with longan",
"barley ginkgo dessert soup": "Barley, fu chuk and gingko",
"bubur pulut hitam": "Bubur pulut hitam",
"honeydew sago": "Honeydew sago",
"leng chee kang": "Leng chee kang",
"guilinggao": "Gui ling gao herbal jelly",
"double skin milk": "Double-skin milk pudding",
"taro sago dessert": "Taro sago",
"bagel": "Bagel, plain",
"lox bagel": "Smoked salmon and cream cheese bagel",
"egg cheese bagel sandwich": "Egg and cheese bagel",
"blueberry bagel": "Blueberry bagel with cream cheese",
"shio pan salt bread": "Salt bread (shio pan)",
"korean cream cheese garlic bread": "Korean garlic cream cheese bun",
"dubai chocolate": "Dubai chocolate bar",
"dubai chewy cookie": "Dubai chewy cookie",
"dubai chocolate strawberries cup": "Dubai chocolate strawberry cup",
"kunafa": "Kunafa",
"cromboloni": "Cromboloni",
"crookie croissant cookie": "Crookie",
"croffle": "Croffle",
"mochi donut pon de ring": "Mochi doughnut",
"matcha tiramisu": "Matcha tiramisu",
"banana pudding": "Banana pudding",
"tanghulu": "Tanghulu",
"corn cheese cup street food": "Jagung cheese",
"fruit sando": "Fruit sando",
"durian flesh": "Musang King durian",
"durian cendol": "Durian cendol",
"pandan gula melaka cake": "Pandan gula melaka cake",
"caramel cheese popcorn": "Garrett popcorn, Chicago mix",
"soft serve cone": "Mixue ice cream cone",
"strawberry matcha latte": "Strawberry matcha latte",
"pistachio latte": "Pistachio latte",
"einspanner coffee": "Einspänner cream coffee",
"coconut shake": "Coconut shake",
"fish soup tofu tomato": "Fish slice soup with tofu and tomato",
"minced pork soup": "Minced pork soup",
"fried fish cake": "Fried fish cake",
"quesadilla": "Chicken quesadilla",
"samgyeopsal korean bbq grill": "Korean BBQ, typical meal",
"bulgogi": "Bulgogi with rice",
"kimchi jjigae": "Kimchi jjigae with rice",
"sundubu jjigae": "Sundubu jjigae with rice",
"tteokbokki": "Tteokbokki",
"jajangmyeon": "Jajangmyeon",
"samgyetang": "Samgyetang",
"budae jjigae": "Army stew (budae jjigae)",
"pajeon": "Seafood pajeon",
"tonkatsu": "Tonkatsu set",
"unadon": "Unagi don",
"karaage": "Chicken karaage",
"okonomiyaki": "Okonomiyaki",
"yakitori": "Yakitori",
"california roll": "California roll",
"zaru soba": "Cold soba",
"thai green curry": "Green curry chicken with rice",
"som tam": "Som tam",
"thai iced tea": "Thai milk tea",
"char siu rice": "Char siu rice",
"siu yuk": "Siew yuk rice",
"sweet and sour pork": "Sweet and sour pork with rice",
"kung pao chicken": "Kung pao chicken with rice",
"kolo mee": "Kolo mee / Kon lo mee",
"sarawak laksa": "Sarawak laksa",
"nasi kerabu": "Nasi kerabu with ayam percik",
"rendang": "Rendang daging with rice",
"butter chicken": "Butter chicken with naan",
"banana leaf rice": "Banana leaf rice with chicken",
"vada": "Vadai",
"barbecue chicken wings": "BBQ chicken wings",
"barbecue pork ribs": "BBQ pork ribs",
"lamb chops plate": "Lamb chop with fries",
"full english breakfast": "Big breakfast",
"avocado toast egg": "Avocado toast with egg",
"big mac": "Big Mac",
"chinese hot pot": "Hot pot, typical meal",
"otak-otak": "Otak-otak"
}
UA = {"User-Agent": "MacaronFoodModel/1.1 (personal, non-commercial food classifier; https://github.com/SiobhanGoh/macaron)"}

def api(params, tries=6):
    """Wikimedia asks bots to go slowly. Retry with a growing pause when it says no."""
    for k in range(tries):
        try:
            r = requests.get("https://commons.wikimedia.org/w/api.php", headers=UA, timeout=30, params=params)
            if r.status_code == 200 and r.headers.get("content-type", "").startswith("application/json"):
                return r.json()
        except Exception:
            pass
        time.sleep(3 * 2 ** k)
    raise RuntimeError("Wikimedia Commons kept refusing")

def commons(term, want=120):
    out, cont = [], {}
    while len(out) < want:
        r = api({"action": "query", "format": "json", "generator": "search", "gsrsearch": f"filetype:bitmap {term}",
                 "gsrnamespace": 6, "gsrlimit": 50, "prop": "imageinfo", "iiprop": "url|mime", "iiurlwidth": 512,
                 "maxlag": 5, **cont})
        for pg in r.get("query", {}).get("pages", {}).values():
            ii = (pg.get("imageinfo") or [{}])[0]
            if ii.get("mime") in ("image/jpeg", "image/png") and ii.get("thumburl"): out.append(ii["thumburl"])
        if "continue" not in r: break
        cont = r["continue"]; time.sleep(1.0)
    return out[:want]

# ---- optional: top up each dish with photos from a public image search ----
# These are other people's photos. They are used here only to teach the model and are never shown in the app
# or shared. Keep it that way: do not publish the downloaded folder. Set WEB_PHOTOS = False to skip this.
WEB_PHOTOS = True
WEB_PER_DISH = 100

def web_photos(term, want):
    q = term + " food"
    try:
        try: from ddgs import DDGS
        except ImportError: from duckduckgo_search import DDGS
        for k in range(3):
            try: return [r["image"] for r in DDGS().images(q, max_results=want) if r.get("image")]
            except Exception as e: err = e; time.sleep(6 * (k + 1))
        raise err
    except Exception as e:
        print("  image search unavailable for", term, "-", str(e)[:80]); return []

from concurrent.futures import ThreadPoolExecutor
def grab(job):
    u, fn = job
    try:
        if not os.path.exists(fn):
            im = Image.open(io.BytesIO(requests.get(u, headers=UA, timeout=20).content)).convert("RGB")
            im.thumbnail((512, 512)); im.save(fn, quality=88)
        return fn
    except Exception: return None

slug = lambda t: "cm_" + re.sub(r"[^a-z0-9]+", "_", t.lower()).strip("_")
MAP_CM, CM_FILES = {}, {}
for term, food in COMMONS.items():
    lab = slug(term); d = f"/tmp/commons/{lab}"; os.makedirs(d, exist_ok=True); files = []   # /tmp is kept out of the saved outputs, so no photo is ever published
    try: urls = commons(term)
    except Exception as e: print("skipped", term, e); continue
    jobs = [(u, f"{d}/{i}.jpg") for i, u in enumerate(urls)]
    if WEB_PHOTOS: jobs += [(u, f"{d}/w{j}.jpg") for j, u in enumerate(web_photos(term, WEB_PER_DISH))]; time.sleep(1.0)
    time.sleep(2.0)
    with ThreadPoolExecutor(4) as pool: files = [f for f in pool.map(grab, jobs) if f]
    if len(files) >= 25: MAP_CM[lab] = food; CM_FILES[lab] = files
    print(f"{term}: {len(files)} photos" + ("" if len(files) >= 25 else "  (too few, left out)"))

# ---- dishes from the restaurants in the big malls (1 Utama, Sunway Pyramid, KL) ----
# Many shop dishes look the same in a photo (every shop's xiao long bao looks alike), so each shop dish
# teaches the shared "look" dish. The app then lists the shop versions after a scan.
# Photos come from the same public image search as above, stay in /tmp and are never published.
SHOP_LOOKS = {
"Absolute Thai pad thai with prawns": "Pad thai",
"Absolute Thai green curry chicken with rice": "Green curry chicken with rice",
"Absolute Thai tom yum goong": "Tom yum soup with rice",
"Absolute Thai basil chicken rice with fried egg": "Chicken rice, steamed",
"Absolute Thai pineapple fried rice": "Nasi goreng kampung",
"Absolute Thai mango sticky rice": "Mango sticky rice",
"Thai milk tea, iced": "Bubble milk tea",
"AK Noodles pan mee, dry": "Pan mee, soup",
"AK Noodles pan mee, soup": "Pan mee, soup",
"AK Noodles chili pan mee": "Pan mee, soup",
"AK Noodles wantan mee": "Wantan mee, dry",
"ANTE kurobuta pork chop": "Hainanese chicken chop",
"ANTE pork belly aglio olio": "Spaghetti carbonara",
"ANTE carbonara with bacon": "Spaghetti carbonara",
"ANTE pork burger": "Cheeseburger",
"Ayam pongteh with rice": "Green curry chicken with rice",
"Nyonya laksa": "Curry laksa",
"Nyonya curry kapitan chicken with rice": "Green curry chicken with rice",
"Nyonya chap chye with rice": "Stir-fried greens",
"Pai tee": "Kuih, 2 pieces",
"Cendol, Nyonya style": "Cendol",
"BananaBro nasi lemak ayam goreng berempah": "Nasi lemak biasa",
"BananaBro nasi lemak rendang daging": "Nasi lemak biasa",
"BananaBro nasi lemak sotong sambal": "Nasi lemak biasa",
"BananaBro nasi lemak biasa": "Nasi lemak biasa",
"Banglo 289 nasi goreng kampung": "Nasi goreng kampung",
"Banglo 289 ayam penyet with rice": "Nasi ayam penyet",
"Banglo 289 mee goreng mamak": "Mee goreng mamak",
"Bar B Q Plaza pork set, per person": "Korean BBQ, typical meal",
"Bar B Q Plaza chicken set, per person": "Korean BBQ, typical meal",
"Bar B Q Plaza seafood set, per person": "Korean BBQ, typical meal",
"Bar B Q Plaza jasmine rice": "White rice",
"Bask Bear iced latte": "Latte, iced",
"Bask Bear Spanish latte, iced": "Latte, iced",
"Bask Bear Coconut latte, iced": "Latte, iced",
"Bask Bear Hazelnut latte, iced": "Latte, iced",
"Bask Bear matcha latte, iced": "Matcha latte, iced",
"bhc Bburinkle chicken": "Korean fried chicken, soy garlic",
"bhc Matchoking chicken": "Korean fried chicken, soy garlic",
"bhc Hot Fried chicken": "Korean fried chicken, soy garlic",
"bhc cheese balls": "Chicken nuggets",
"Black Canyon tom yum fried rice": "Nasi goreng kampung",
"Black Canyon pad kra pao chicken rice": "Chicken rice, steamed",
"Black Canyon green curry with rice": "Green curry chicken with rice",
"Black Canyon iced cappuccino": "Chocolate milkshake",
"BMS Organics brown rice nasi lemak": "Nasi lemak biasa",
"BMS Organics vegetarian laksa": "Curry laksa",
"BMS Organics quinoa salad": "Chicken caesar salad",
"BMS Organics alkaline juice": "Soft drink",
"Bungkus Kaw Kaw nasi lemak ayam": "Nasi lemak biasa",
"Bungkus Kaw Kaw nasi lemak sotong": "Nasi lemak biasa",
"Bungkus Kaw Kaw teh tarik": "Teh tarik",
"Canton-i wantan noodles": "Wantan mee, dry",
"Canton-i century egg and pork congee": "Chicken porridge",
"Canton-i fish slice congee": "Chicken porridge",
"Canton-i beef brisket noodles": "Pho",
"Canton-i Hong Kong milk tea": "Bubble milk tea",
"Canton-i char siew rice": "Char siu rice",
"Champignons croissant": "Butter croissant",
"Champignons almond croissant": "Butter croissant",
"Champignons fruit tart": "Fruit tart",
"Champignons chocolate cake": "Chocolate cake",
"Chili's Oldtimer with cheese burger": "Cheeseburger",
"Chili's Southwestern eggrolls": "Pisang goreng",
"Chili's chicken fajitas": "Bagel with cream cheese",
"Chili's baby back ribs, half rack": "BBQ pork ribs",
"Chili's chicken crispers": "Chicken nuggets",
"Chili's Molten Chocolate Cake": "Chocolate cake",
"Chili's Presidente margarita, non-alcoholic": "Soft drink",
"Vegetarian economy rice, 3 dishes": "Economy rice: 1 meat, 2 veg",
"Vegetarian curry mee": "Curry laksa",
"Vegetarian fried bee hoon": "Char kway teow / Kway teow goreng",
"Cili Kampung nasi ayam bakar": "Ikan bakar with rice and ulam",
"Cili Kampung asam pedas ikan with rice": "Steamed fish with rice and greens",
"Cili Kampung daging masak hitam with rice": "Rendang daging with rice",
"Cili Kampung sayur campur": "Stir-fried greens",
"Cong Caphe coconut coffee smoothie": "Chocolate milkshake",
"Cong Caphe Vietnamese iced coffee with condensed milk": "Kuih, 2 pieces",
"Cong Caphe egg coffee": "Latte, iced",
"Cong Caphe banh mi": "Bagel with cream cheese",
"Dave's ribeye steak": "Steak with vegetables",
"Dave's chicken chop": "Hainanese chicken chop",
"Dave's fish and chips": "Fish and chips",
"Dave's lamb shank": "Steak with vegetables",
"De Irrfan's nasi kerabu ayam": "Nasi kerabu with ayam percik",
"De Irrfan's laksa Johor": "Curry laksa",
"De Irrfan's roti john": "Roti canai kosong",
"Dragon-i Shanghai xiao long bao": "Xiao long bao",
"Dragon-i la mian in spicy soup": "Fish slice soup with tofu and tomato",
"Dragon-i dan dan noodles": "Fish slice soup with tofu and tomato",
"Dragon-i braised pork belly with rice": "Sweet and sour pork with rice",
"Dragon-i hot and sour soup": "Minced pork soup",
"Dragon-i deep-fried dumplings": "Pan-fried pork dumplings",
"Malatang, typical bowl": "Fish slice soup with tofu and tomato",
"Japanese curry rice (Dua)": "Chicken katsu curry rice",
"Empire Sushi salmon nigiri": "Sushi plates",
"Empire Sushi California roll": "Sushi plates",
"Empire Sushi salmon mentai roll": "Sushi plates",
"Empire Sushi chicken teriyaki don": "Sushi plates",
"Esquire Kitchen golden pumpkin noodles": "Wantan mee, dry",
"Esquire Kitchen salted fish fried rice": "Nasi goreng kampung",
"Esquire Kitchen sweet and sour fish with rice": "Sweet and sour pork with rice",
"Esquire Kitchen hokkien mee": "Hokkien mee (KL)",
"Fei Fan hot pot, per person": "Hot pot, typical meal",
"Fish & Co fish and chips": "Fish and chips",
"Fish & Co Fish in a Pan": "Fish and chips",
"Fish & Co seafood platter for one": "Fish and chips",
"Fish & Co New York fish and chips": "Fish and chips",
"Fish & Co clam chowder": "Minced pork soup",
"Fong Lye braised pork rice": "Sweet and sour pork with rice",
"Fong Lye Three-cup chicken set": "Sweet and sour pork with rice",
"Fong Lye beef noodles": "Boat noodles",
"Fong Lye pork chop rice": "Hainanese chicken chop",
"Food court chicken rice": "Chicken rice, steamed",
"Food court nasi campur, 2 dishes": "Economy rice: 1 meat, 2 veg",
"Food court char kuey teow": "Char kway teow / Kway teow goreng",
"GIGI iced latte": "Latte, iced",
"GIGI butterscotch latte, iced": "Latte, iced",
"GIGI mocha, iced": "Mocha, iced",
"GO Unique nasi lemak ayam": "Nasi lemak biasa",
"GO Unique mee goreng": "Mee goreng mamak",
"Haidilao tomato broth hot pot, per person": "Hot pot, typical meal",
"Haidilao mala broth hot pot, per person": "Fish slice soup with tofu and tomato",
"Haidilao hand-pulled noodles": "Fish slice soup with tofu and tomato",
"Hansang bibimbap": "Bibimbap",
"Hansang kimchi jjigae": "Army stew (budae jjigae)",
"Hansang bulgogi set": "Bibimbap",
"Hansang japchae": "Bibimbap",
"Hokkaido Sushi salmon sashimi": "Salmon sashimi and rice",
"Hokkaido Sushi chirashi don": "Sushi plates",
"Hokkaido Sushi unagi don": "Sushi plates",
"Kim Gary baked cheese rice with chicken chop": "Hainanese chicken chop",
"Kim Gary pineapple bun with butter": "Pineapple bun with butter",
"Kim Gary French toast": "French toast",
"Kim Gary Hong Kong milk tea, iced": "Bubble milk tea",
"Kim Gary yin yong (coffee and tea)": "Bubble milk tea",
"Hotpot Kitchen, per person": "Hot pot, typical meal",
"House of Pok pork noodles, soup": "Wantan mee, dry",
"House of Pok char siew rice": "Char siu rice",
"HWC flat white": "Latte, iced",
"HWC iced mocha": "Mocha, iced",
"HWC chicken pie": "Apple pie",
"I Love Yoo! you tiao": "Pisang goreng",
"I Love Yoo! soy milk": "Mango lassi",
"I Love Yoo! you tiao with kaya": "Pisang goreng",
"India Gate chicken biryani": "Nasi briyani ayam",
"India Gate butter chicken with naan": "Butter chicken with naan",
"India Gate garlic naan": "Tandoori chicken with naan",
"India Gate tandoori chicken": "Tandoori chicken with naan",
"India Gate palak paneer": "Butter chicken with naan",
"India Gate mango lassi": "Mango lassi",
"Ippudo Shiromaru Classic": "Ramen, tonkotsu",
"Ippudo Akamaru Modern": "Ramen, tonkotsu",
"Ippudo Karaka Spicy ramen": "Ramen, tonkotsu",
"Ippudo chicken ramen": "Ramen, tonkotsu",
"Ippudo pork bun": "Char siu bao",
"Ippudo gyoza": "Pan-fried pork dumplings",
"Jardin earl grey milk tea": "Bubble milk tea",
"Jardin iced latte": "Latte, iced",
"Jatujak boat noodles": "Boat noodles",
"Jatujak pad kra pao with egg": "Pad kra pao with rice and egg",
"Jatujak moo ping skewers": "Satay ayam",
"Jatujak mango sticky rice": "Mango sticky rice",
"Jipinhe scone, plain": "Scone with jam and cream",
"Jipinhe chocolate scone": "Scone with jam and cream",
"Johnny's tom yum steamboat, per person": "Hot pot, typical meal",
"Johnny's Thai fried rice": "Nasi goreng kampung",
"Johnny's chicken chop": "Hainanese chicken chop",
"K Fry Yangnyeom chicken": "Korean fried chicken, soy garlic",
"K Fry soy garlic chicken": "Korean fried chicken, soy garlic",
"K Fry Spicy Black Garlic Chicken": "Korean fried chicken, soy garlic",
"K Fry kimchi fried rice": "Nasi goreng kampung",
"K Fry rabokki": "Tteokbokki",
"K Fry Berry-Matcha Mochi Bingsoo": "Injeolmi bingsu",
"KAKI 5 pan mee": "Pan mee, soup",
"KAKI 5 curry mee": "Curry laksa",
"Kampung Dining nasi kerabu": "Nasi kerabu with ayam percik",
"Kampung Dining nasi lemak ayam rendang": "Nasi lemak biasa",
"Kampung Dining asam laksa": "Asam laksa",
"Kenangan Signature coffee latte, iced": "Latte, iced",
"Kenangan Avocado coffee": "Latte, iced",
"Kenangan Black Aren latte": "Latte, iced",
"Ko Hyang dolsot bibimbap": "Bibimbap",
"Ko Hyang sundubu jjigae": "Army stew (budae jjigae)",
"Ko Hyang dakgalbi": "Korean BBQ, typical meal",
"Kyochon Soy Garlic wings": "Korean fried chicken, soy garlic",
"Kyochon Red Original wings": "Korean fried chicken, soy garlic",
"Kyochon Honey Original drumsticks": "Korean fried chicken, soy garlic",
"Kyochon Salsal chicken strips": "Chicken nuggets",
"Laem Charoen deep-fried seabass with fish sauce": "Steamed fish with rice and greens",
"Laem Charoen steamed seabass with lime and chili": "Steamed fish with rice and greens",
"Laem Charoen crab fried rice": "Nasi goreng kampung",
"Laem Charoen tom yum seafood": "Tom yum soup with rice",
"Little Cravings nasi lemak with fried chicken": "Nasi lemak biasa",
"Little Cravings char kuey teow": "Char kway teow / Kway teow goreng",
"Little Cravings Penang assam laksa": "Asam laksa",
"Little Nyonya nasi lemak Nyonya": "Nasi lemak biasa",
"Little Nyonya Nyonya laksa": "Curry laksa",
"Little Nyonya ayam buah keluak with rice": "Green curry chicken with rice",
"Long Jing Dongpo pork with rice": "Sweet and sour pork with rice",
"Long Jing green tea chicken": "Bubble milk tea",
"Long Jing tofu with minced pork": "Stir-fried greens",
"LOVNS banh mi": "Bagel with cream cheese",
"LOVNS iced latte": "Latte, iced",
"Luretta margherita pizza": "Pizza",
"Luretta truffle mushroom pasta": "Spaghetti carbonara",
"Luretta tiramisu": "Tiramisu",
"Madam Kwan's nasi lemak": "Nasi lemak biasa",
"Madam Kwan's nasi bojari": "Nasi briyani ayam",
"Madam Kwan's char kuey teow": "Char kway teow / Kway teow goreng",
"Madam Kwan's Nasi Ayam Gepuk": "Nasi ayam penyet",
"Madam Kwan's assam laksa": "Asam laksa",
"Madam Kwan's cendol": "Cendol",
"Makii Makii onigiri, tuna mayo": "Sushi plates",
"Makii Makii salmon hand roll": "Sushi plates",
"Marco margherita pizza": "Pizza",
"Marco carbonara": "Spaghetti carbonara",
"Marco tiramisu": "Tiramisu",
"Mee tarik beef soup": "Minced pork soup",
"Mee tarik fried with chicken": "Mee goreng mamak",
"Mee tarik dry with minced beef": "Wantan mee, dry",
"Melaka chicken rice balls": "Chicken rice, steamed",
"Melaka Nyonya asam pedas fish with rice": "Steamed fish with rice and greens",
"Melaka cendol with gula Melaka": "Cendol",
"Menya Musashi tsukemen": "Ramen, tonkotsu",
"Menya Musashi black ramen": "Ramen, tonkotsu",
"Menya Musashi red spicy ramen": "Ramen, tonkotsu",
"MODU Korean corn dog": "Pisang goreng",
"MODU tteokbokki": "Tteokbokki",
"Monster Curry pork katsu curry": "Chicken katsu curry rice",
"Monster Curry chicken katsu curry": "Green curry chicken with rice",
"Monster Curry omu curry rice": "Chicken katsu curry rice",
"Mr Fish fish head noodle soup": "Fish slice soup with tofu and tomato",
"Mr Fish sliced fish noodle soup": "Fish slice soup with tofu and tomato",
"Mr Fish seafood noodles, milky broth": "Fish slice soup with tofu and tomato",
"Mr Tuk Tuk pad thai": "Pad thai",
"Mr Tuk Tuk tom yum fried rice": "Nasi goreng kampung",
"Mr Tuk Tuk Thai iced tea": "Bubble milk tea",
"NakNak bulgogi kimbap": "Sushi plates",
"NakNak ramyeon with egg": "Ramen, shoyu",
"Nam Heong white coffee, iced": "Latte, iced",
"Nam Heong egg tart": "Egg tart",
"Nam Heong Ipoh chicken hor fun": "Ipoh hor fun",
"Nam Heong curry mee": "Curry laksa",
"Nippori matcha latte": "Matcha latte, iced",
"Nippori matcha soft serve": "Ice cream",
"Noodle Shack chili pan mee": "Pan mee, soup",
"Noodle Shack dry wantan noodles": "Wantan mee, dry",
"Nyonya Colors kuih lapis": "Kuih, 2 pieces",
"Nyonya Colors onde-onde": "Kuih, 2 pieces",
"Nyonya Colors nasi lemak": "Nasi lemak biasa",
"Nyonya Colors curry puff": "Curry puff",
"Onigirazu, chicken katsu": "Sushi plates",
"Onigirazu, salmon and egg": "Sushi plates",
"Oversea char siew": "Char siu rice",
"Oversea yang chow fried rice": "Nasi goreng kampung",
"Oversea dim sum, 3 baskets": "Dim sum breakfast",
"Paris Baguette croissant": "Butter croissant",
"Paris Baguette strawberry cream cake": "Chocolate cake",
"Paris Baguette ham and cheese sandwich": "Cheeseburger",
"Paris Baguette soboro bread": "Char siu bao",
"Park's Bagels plain bagel with cream cheese": "Bagel with cream cheese",
"Park's Bagels smoked salmon bagel": "Bagel with cream cheese",
"Park's Bagels egg and cheese bagel": "Bagel with cream cheese",
"Pasar Eatai char kuey teow": "Char kway teow / Kway teow goreng",
"Pasar Eatai Hokkien mee": "Hokkien mee (KL)",
"Pasar Eatai satay, 6 sticks": "Satay ayam",
"Penang chendul": "Cendol",
"Penang chendul with red beans and pulut": "Cendol",
"Penang ais kacang": "Ais kacang (ABC)",
"Petite Maison pain au chocolat": "Butter croissant",
"Petite Maison sourdough sandwich": "Cheeseburger",
"Pho Vietz beef pho": "Pho",
"Pho Vietz chicken pho": "Pho",
"Pho Vietz fresh spring rolls": "Pisang goreng",
"Pho Vietz fried spring rolls": "Pisang goreng",
"Pho Vietz bun bo hue": "Pho",
"Pho Vietz Vietnamese drip coffee": "Pho",
"Porridge Time chicken porridge": "Chicken porridge",
"Porridge Time fish porridge": "Chicken porridge",
"Porridge Time pork porridge with century egg": "Chicken porridge",
"Putien Heng Hwa fried bee hoon": "Char kway teow / Kway teow goreng",
"Pu Tien braised pork belly with buns": "Char siu bao",
"Putien Lor mee": "Lor mee / Loh mee",
"Putien stir-fried yam": "Stir-fried greens",
"Pao Xiang bak kut teh set": "Bak kut teh with rice",
"Pao Xiang dry bak kut teh": "Bak kut teh with rice",
"Peranakan Place ayam pongteh with rice": "Green curry chicken with rice",
"Peranakan Place babi pongteh with rice": "Green curry chicken with rice",
"Rich Kopitiam kaya toast set": "Kaya toast set",
"Rich Kopitiam curry mee": "Curry laksa",
"Rich Kopitiam nasi lemak": "Nasi lemak biasa",
"Robataya robatayaki set": "Korean BBQ, typical meal",
"Robataya salmon teriyaki set": "Chicken teriyaki don",
"Rocku yakiniku beef set, per person": "Korean BBQ, typical meal",
"Samtai nasi lemak": "Nasi lemak biasa",
"Samtai kaya butter toast": "Kaya toast set",
"Samtai Hainan chicken chop rice": "Hainanese chicken chop",
"Se'Indonesia nasi ayam penyet": "Nasi ayam penyet",
"Se'Indonesia bakso": "Minced pork soup",
"Se'Indonesia gado-gado": "Som tam",
"Sekinchan fish and chips": "Fish and chips",
"Sekinchan grilled fish rice": "Ikan bakar with rice and ulam",
"Shabu-Yo shabu shabu, per person": "Hot pot, typical meal",
"Shake Shack ShackBurger single": "Cheeseburger",
"Shake Shack Double ShackBurger": "Cheeseburger",
"Shake Shack Chick'n Shack": "Cheeseburger",
"Shake Shack crinkle cut fries": "French fries",
"Shake Shack cheese fries": "French fries",
"Shake Shack vanilla shake": "Chocolate milkshake",
"Jjajangmyeon": "Jajangmyeon",
"Budae jjigae, per person": "Army stew (budae jjigae)",
"Jjamppong": "Jajangmyeon",
"Tangsuyuk (sweet and sour pork)": "Sweet and sour pork with rice",
"Since 1939 claypot chicken rice": "Claypot chicken rice",
"Since 1939 Hokkien mee": "Hokkien mee (KL)",
"Soup Restaurant Samsui ginger chicken": "Minced pork soup",
"Soup Restaurant double-boiled soup of the day": "Minced pork soup",
"Starbucks caffe latte, grande": "Latte, iced",
"Starbucks caramel macchiato, grande": "Latte, iced",
"Starbucks java chip Frappuccino, grande": "Chocolate milkshake",
"Starbucks matcha latte, grande": "Matcha latte, iced",
"Starbucks butter croissant": "Butter croissant",
"Strangers iced latte": "Latte, iced",
"Strangers brunch plate": "Big breakfast",
"Stuff'd kebab wrap, chicken": "Bagel with cream cheese",
"Stuff'd burrito": "Bagel with cream cheese",
"Sushi Zanmai salmon sashimi": "Salmon sashimi and rice",
"Sushi Zanmai salmon nigiri": "Sushi plates",
"Sushi Zanmai chawanmushi": "Sushi plates",
"Sushi Zanmai unagi don": "Sushi plates",
"Sushi Zanmai salmon don": "Sushi plates",
"Sushiro salmon nigiri": "Sushi plates",
"Sushiro thick cut salmon belly": "Salmon sashimi and rice",
"Sushiro grilled eel nigiri": "Sushi plates",
"Sushiro tuna nigiri": "Sushi plates",
"Sushiro typical meal": "Sushi plates",
"Sushiya salmon don": "Sushi plates",
"Sushiya chicken katsu don": "Sushi plates",
"Tai Er sauerkraut fish, per person": "Fish slice soup with tofu and tomato",
"Tai Ho Jiak lu rou fan": "Sweet and sour pork with rice",
"Tai Ho Jiak Taiwanese fried chicken": "Korean fried chicken, soy garlic",
"Tanyu grilled fish, per person": "Steamed fish with rice and greens",
"Teh Tarik Club teh tarik": "Teh tarik",
"Teh Tarik Club roti canai with curry": "Roti canai kosong",
"Teh Tarik Nation teh tarik": "Teh tarik",
"Teh Tarik Nation nasi lemak": "Nasi lemak biasa",
"Teochew lor mee": "Lor mee / Loh mee",
"Teochew braised duck rice": "Roast duck rice",
"Teppanyaki chicken set": "Chicken teriyaki don",
"Teppanyaki salmon set": "Chicken teriyaki don",
"TGI Fridays Jack Daniel's glazed chicken": "Doughnut",
"TGI Fridays loaded potato skins": "French fries",
"TGI Fridays Fridays cheeseburger": "Cheeseburger",
"TGI Fridays sesame jack chicken strips": "Chicken nuggets",
"Thai chicken rice (khao man gai)": "Chicken rice, steamed",
"Thai Hou Sek tom yum noodles": "Pho",
"Thai Hou Sek basil pork rice": "Pad kra pao with rice and egg",
"Coffee Bean Ice Blended, mocha, regular": "Chocolate milkshake",
"Coffee Bean vanilla latte, iced, regular": "Latte, iced",
"Coffee Bean chicken pie": "Apple pie",
"Daily Greens grain bowl with chicken": "Chicken caesar salad",
"Daily Greens salmon salad": "Salmon sashimi and rice",
"Fish Bowl poke bowl, salmon": "Salmon sashimi and rice",
"Fish Bowl poke bowl, tuna": "Salmon sashimi and rice",
"Tim Hortons Iced Capp": "Chocolate milkshake",
"Tim Hortons Timbits, chocolate glazed": "Doughnut",
"Tim Hortons double double coffee": "Latte, iced",
"Tim Hortons Farmer's breakfast wrap": "Bagel with cream cheese",
"Ma Maison rosu katsu set": "Tonkatsu set",
"Ma Maison hire katsu set": "Tonkatsu set",
"Ma Maison katsu curry": "Chicken katsu curry rice",
"Tous les Jours walnut pound cake": "Chocolate cake",
"Tous les Jours sausage bread": "Char siu bao",
"Tous les Jours fresh cream cake": "Chocolate cake",
"Wang Donkas pork cutlet": "Tonkatsu set",
"Wang Donkas cheese donkas": "Tonkatsu set",
"Wanhoi wantan noodles": "Wantan mee, dry",
"Wanhoi prawn wantan soup": "Fried wantan",
"Whisk flat white": "Latte, iced",
"Whisk big breakfast": "Big breakfast",
"Whisk eggs Benedict": "Big breakfast",
"Wok Hey egg fried rice with chicken": "Nasi goreng kampung",
"Wok Hey black pepper beef fried rice": "Nasi goreng kampung",
"Wok Hey fried udon": "Char kway teow / Kway teow goreng",
"Wong Kok baked pork chop rice": "Baked pork chop rice",
"Wong Kok satay beef noodles": "Boat noodles",
"Wong Kok pineapple bun with butter": "Pineapple bun with butter",
"Wong Kok milk tea, iced": "Bubble milk tea",
"Wonjo Buanjib pork belly BBQ set, per person": "Korean BBQ, typical meal",
"WPS Cafe spaghetti aglio olio": "Spaghetti carbonara",
"WPS Cafe iced latte": "Latte, iced",
"Pork and chive dumplings, boiled": "Pan-fried pork dumplings",
"Hot and sour soup": "Minced pork soup",
"Yakiniku Kuro wagyu set, per person": "Steak with vegetables",
"Yat Yat Seng wantan mee": "Wantan mee, dry",
"Yat Yat Seng roast pork rice": "Char siu rice",
"Yuan Room dim sum, 3 baskets": "Dim sum breakfast",
"Yuan Room roast duck": "Roast duck rice",
"Yunnan crossing-the-bridge noodles": "Pho",
"Yunnan spicy rice noodles": "Pho",
"Yun Nans steam pot chicken soup": "Hot pot, typical meal",
"Yun Nans rice noodles with minced pork": "Pho",
"Zok bamboo noodles with char siu": "Wantan mee, dry",
"ZUS Spanish latte, iced": "Latte, iced",
"ZUS butter croissant": "Butter croissant",
"Come True cafe brunch pasta": "Spaghetti carbonara",
"Asador grilled steak, per person": "Korean BBQ, typical meal",
"Kumo Bento chicken karaage bento": "Chicken teriyaki don",
"Tsukiji Sushi salmon don": "Sushi plates",
"Moo Yogurt frozen yogurt": "Mango lassi",
"Sandwich Box chicken sandwich": "Cheeseburger",
"Bread History sausage bun": "Char siu bao",
"Tengyu wagyu yakiniku set": "Steak with vegetables",
"Lee Xiao Ming malatang": "Fish slice soup with tofu and tomato",
"Oh Cha matcha latte": "Matcha latte, iced",
"Onsemiro Korean BBQ, per person": "Korean BBQ, typical meal",
"Tenkai tendon": "Chicken teriyaki don",
"Dodo Korea bibimbap": "Bibimbap",
"Dodo Korea kimchi fried rice": "Nasi goreng kampung",
"Family pan mee, soup": "Pan mee, soup",
"Sizzling hot plate chicken": "Hainanese chicken chop",
"Ma Yi Lan Lanzhou beef noodles": "Boat noodles",
"Farm House Kitchen economy rice": "Economy rice: 1 meat, 2 veg",
"Mama Yap Hakka noodles": "Wantan mee, dry",
"Cauli & Rice grain bowl": "Chicken caesar salad",
"Kim Lian Kee Hokkien mee": "Hokkien mee (KL)",
"Soong Kee beef ball noodles, dry": "Wantan mee, dry",
"Duck egg char kuey teow": "Char kway teow / Kway teow goreng",
"Klang bak kut teh": "Bak kut teh with rice",
"Hon Kee fish porridge": "Chicken porridge",
"Imbi pork noodles, soup": "Wantan mee, dry",
"The Bean nasi lemak": "Nasi lemak biasa",
"Masala King mutton biryani": "Nasi briyani ayam",
"Tutti Frutti frozen yogurt": "Mango lassi",
"Grand Harbour dim sum, per person": "Dim sum breakfast",
"Al-Amar Express chicken shawarma wrap": "Bagel with cream cheese",
"FWF hot pot, per person": "Hot pot, typical meal",
"Mengs Kee claypot chicken rice": "Claypot chicken rice",
"Corks Out steak with fries": "Steak with vegetables",
"Luk Yu dim sum, per person": "Dim sum breakfast",
"Shook! grilled wagyu": "Steak with vegetables",
"Enak KL nasi lemak with rendang": "Nasi lemak biasa",
"DIN xiao long bao": "Xiao long bao",
"DIN pork chop fried rice": "Nasi goreng kampung",
"Little Penang curry mee": "Curry laksa",
"Little Penang char kway teow": "Char kway teow / Kway teow goreng",
"Little Penang nasi lemak": "Nasi lemak biasa",
"Little Penang asam laksa": "Asam laksa",
"Bumbu Desa ayam penyet with rice": "Nasi ayam penyet",
"Bumbu Desa nasi campur": "Economy rice: 1 meat, 2 veg",
"Aseana nasi lemak": "Nasi lemak biasa",
"Chakri Palace pineapple fried rice": "Nasi goreng kampung",
"Chakri Palace green curry": "Green curry chicken with rice",
"Para Thai tom yum noodles": "Pho",
"Rén Peking duck, per person": "Roast duck rice",
"Rén la mian with minced pork": "Fish slice soup with tofu and tomato",
"Ki Ki dry noodles with scallion oil": "Wantan mee, dry",
"Koryo-Won bulgogi set": "Bibimbap",
"Koryo-Won Korean BBQ, per person": "Korean BBQ, typical meal",
"Sakana sushi set": "Sushi plates",
"Chinoz pasta carbonara": "Spaghetti carbonara",
"Delirium Belgian mussels with fries": "Butter prawns",
"Maria's Signature Spanish paella": "Nasi goreng kampung",
"Melur & Thyme nasi lemak": "Nasi lemak biasa",
"Open House nasi lemak with ayam goreng": "Nasi lemak biasa",
"O'Briens chicken sandwich": "Cheeseburger",
"Acme burger": "Cheeseburger",
"Acme iced latte": "Latte, iced",
"Bape Cafe latte": "Latte, iced",
"Eight Ounce iced latte": "Latte, iced",
"Bisou cupcake": "Chocolate cake",
"Hazukido croissant": "Butter croissant",
"Hui Lau Shan mango pomelo sago": "Tau fu fah, less syrup",
"Salon Du Chocolat hot chocolate": "Chocolate bar",
"Yomie's purple rice yogurt": "Mango lassi",
"B& iced latte": "Latte, iced",
"Ahh Yum nasi bunga telang": "Nasi lemak biasa",
"Food court wantan mee": "Wantan mee, dry",
"Eatomo chicken teriyaki bento": "Chicken teriyaki don",
"Fuel Shack Jumbo Smoothie Bowl": "Chocolate milkshake",
"Fuel Shack Big Shack burger": "Cheeseburger",
"JJ chili pan mee": "Pan mee, soup",
"Kagura chicken paitan ramen": "Ramen, tonkotsu",
"Ken-Chan katsu curry rice": "Chicken katsu curry rice",
"Nathan's Famous hot dog": "Pretzel, original",
"Nathan's Famous cheese fries": "French fries",
"Milano pizza, 2 slices": "Pizza",
"Milano spaghetti bolognese": "Spaghetti carbonara",
"Saravanaa Bhavan masala dosa": "Thosai with dhal and chutney",
"Saravanaa Bhavan South Indian meals (banana leaf)": "Nasi briyani ayam",
"Saravanaa Bhavan idli": "Thosai with dhal and chutney",
"Shihlin XXL crispy chicken": "Korean fried chicken, soy garlic",
"Shihlin oyster mee sua": "Bee hoon soup / Mee hoon soup",
"Shihlin popcorn chicken": "Chicken nuggets",
"Sizzles sizzling chicken chop": "Hainanese chicken chop",
"Warung Prima nasi kunyit with chicken": "Nasi lemak biasa",
"Hameed nasi kandar": "Nasi briyani ayam",
"Big Pan pan mee": "Pan mee, soup",
"Pizza Hut Super Supreme pan pizza": "Pizza",
"Pizza Hut Hawaiian Chicken pan pizza": "Pizza",
"Pizza Hut pepperoni pan pizza": "Pizza",
"Pizza Hut Chicken Supreme thin crust": "Pizza",
"Pizza Hut cheesy stuffed crust pizza": "Pizza",
"Pizza Hut chicken wings": "Korean fried chicken, soy garlic",
"Pizza Hut garlic bread": "Pizza",
"Pizza Hut cheesy baked spaghetti, bolognese": "Spaghetti carbonara",
"Pizza Hut mushroom soup": "Pizza",
"Domino's Chicken Pepperoni pizza, hand tossed": "Pizza",
"Domino's Domino's Deluxe pizza, hand tossed": "Pizza",
"Domino's Hawaiian Paradise pizza": "Pizza",
"Carl's Jr Famous Star with cheese": "Cheeseburger",
"Carl's Jr Western Bacon Cheeseburger": "Cheeseburger",
"Carl's Jr Guacamole Bacon Burger": "Cheeseburger",
"Carl's Jr crispy chicken sandwich": "Cheeseburger",
"Carl's Jr onion rings": "French fries",
"Marrybrown fried chicken, 2 pieces": "KFC Original Recipe, thigh",
"Marrybrown MB burger": "Cheeseburger",
"Marrybrown nasi lemak MB": "Nasi lemak biasa",
"Marrybrown chicken porridge": "Chicken porridge",
"Jollibee Chickenjoy, 1 piece with rice": "KFC Original Recipe, thigh",
"Jollibee Jolly Spaghetti": "Spaghetti bolognese",
"Jollibee Yumburger": "Cheeseburger",
"Jollibee peach mango pie": "Apple pie",
"Dunkin' glazed donut": "Doughnut",
"Dunkin' Boston Kreme donut": "Doughnut",
"Dunkin' chocolate frosted donut": "Doughnut",
"Dunkin' Munchkins": "Doughnut",
"Krispy Kreme Original Glazed": "Doughnut",
"Krispy Kreme chocolate iced glazed": "Doughnut",
"Krispy Kreme filled doughnut": "Doughnut",
"Cinnabon MiniBon": "Cinnabon Classic Roll",
"Cinnabon Chocobon": "Cinnabon Classic Roll",
"Beard Papa chocolate cream puff": "Beard Papa original cream puff",
"Beard Papa eclair": "Beard Papa original cream puff",
"Famous Amos chocolate chip cookies": "Chocolate chip cookie",
"Ben's Cookies milk chocolate cookie": "Chocolate chip cookie",
"Haagen-Dazs single scoop": "Ice cream",
"Haagen-Dazs fondue for one": "Ice cream",
"Lindt Lindor truffles": "Chocolate bar",
"Lindt hot chocolate": "Chocolate bar",
"Chateraise strawberry shortcake": "Chocolate cake",
"Chateraise cream puff": "Beard Papa original cream puff",
"Komugi cheese tart": "Egg tart",
"Komugi curry bread": "Char siu bao",
"Mille crepe, original": "Chocolate cake",
"Mille crepe, matcha": "Chocolate cake",
"Garrett CaramelCrisp popcorn": "Popcorn, caramel",
"Garrett Chicago Mix popcorn": "Popcorn, caramel",
"Happy Potato tornado potato": "French fries",
"Happy Potato cheese potato balls": "French fries",
"Hot & Roll chicken roll": "Bagel with cream cheese",
"Crispy popiah": "Pisang goreng",
"Bibi's popiah, fresh": "Popiah, fresh",
"LFC fried chicken, 2 pieces": "KFC Original Recipe, thigh",
"Boost Juice Mango Magic": "Soft drink",
"Boost Juice Berry Crush": "Soft drink",
"Juice Works Mango Tango": "Soft drink",
"Juice Works watermelon juice": "Soft drink",
"HEYTEA cheese grape": "Bubble milk tea",
"HEYTEA jasmine milk tea": "Bubble milk tea",
"ChaTraMue Thai tea": "Bubble milk tea",
"ChaTraMue Thai green tea": "Bubble milk tea",
"ShuYi grass jelly milk tea": "Bubble milk tea",
"Oat Story oat latte": "Latte, iced",
"Claypot white coffee": "Latte, iced",
"Frozen yogurt with toppings": "Mango lassi",
"Yolé frozen yogurt": "Mango lassi",
"Snowflake taro balls dessert": "Cendol",
"Sugirl durian mango dessert": "Cendol",
"Shantea grass jelly dessert": "Cendol",
"Pobbyice bingsu": "Injeolmi bingsu",
"Herbal Farmer herbal jelly": "Big breakfast",
"Roasted Gong pear drink": "Soft drink",
"Paradise Dynasty 8-flavour xiao long bao": "Xiao long bao",
"Paradise Dynasty la mian with braised pork": "Fish slice soup with tofu and tomato",
"Paradise Dynasty dan dan noodles": "Fish slice soup with tofu and tomato",
"Paradise Dynasty fried rice with shrimp": "Nasi goreng kampung",
"Grand Taipei braised pork rice": "Sweet and sour pork with rice",
"Grand Taipei beef noodle soup": "Minced pork soup",
"Grand Taipei popcorn chicken": "Chicken nuggets",
"Amarin green curry chicken": "Green curry chicken with rice",
"Amarin tom yum goong": "Tom yum soup with rice",
"Amarin pad thai": "Pad thai",
"Tanba yakiniku set, per person": "Korean BBQ, typical meal",
"Palsaik 8-flavour pork belly set, per person": "Korean BBQ, typical meal",
"Palsaik kimchi stew": "Army stew (budae jjigae)",
"BBQ Town grill set, per person": "Korean BBQ, typical meal",
"Xiao Long Kan mala hot pot, per person": "Fish slice soup with tofu and tomato",
"Mala mini hotpot": "Fish slice soup with tofu and tomato",
"Fei Fei Crab spicy crab pot, per person": "Butter prawns",
"Peng Chu Hunan chili pork": "Steamed fish with rice and greens",
"Peng Chu steamed fish with chili": "Steamed fish with rice and greens",
"Souper Tang double-boiled chicken soup": "Minced pork soup",
"MyIPOH Ipoh chicken hor fun": "Ipoh hor fun",
"MyIPOH bean sprout chicken": "Chicken rice, steamed",
"MyIPOH white coffee": "Latte, iced",
"NALE nasi lemak ayam goreng": "Nasi lemak biasa",
"NALE nasi lemak rendang daging": "Nasi lemak biasa",
"Nasi Lemak Shop nasi lemak ayam berempah": "Nasi lemak biasa",
"Nasi kandar with fried chicken and curry": "Nasi briyani ayam",
"Kayu roti canai with dhal": "Roti canai kosong",
"Kayu mee goreng mamak": "Mee goreng mamak",
"Ali Muthu nasi lemak": "Nasi lemak biasa",
"Ali Muthu roti bakar set": "Kaya toast set",
"Ayam gepuk with rice": "Nasi ayam penyet",
"Nasi Arab mandi with chicken": "Nasi briyani ayam",
"Lalezar mixed grill kebab": "Satay ayam",
"Lalezar chicken shawarma wrap": "Bagel with cream cheese",
"Kiku-Zakura salmon teriyaki set": "Chicken teriyaki don",
"Kiku-Zakura tempura udon": "Udon in soup",
"Kamakura chicken katsu don": "Tonkatsu set",
"Bankara ramen": "Ramen, tonkotsu",
"Bankara tsukemen": "Ramen, tonkotsu",
"MyeongDong topokki": "Tteokbokki",
"MyeongDong Korean fried chicken": "Korean fried chicken, soy garlic",
"MyeongDong ramyeon": "Ramen, shoyu",
"Halla Holla Korean corn dog": "Pisang goreng",
"Sin Man Kee baked pork chop rice": "Baked pork chop rice",
"Sin Man Kee pineapple bun with butter": "Pineapple bun with butter",
"Sin Man Kee Hong Kong milk tea": "Bubble milk tea",
"Mak's Chee wonton noodle soup": "Fried wantan",
"Mak's Chee beef brisket noodles": "Pho",
"Omega pork noodles, soup": "Wantan mee, dry",
"Go Noodle signature mixed pork noodle soup": "Minced pork soup",
"Good Chen chicken rice": "Chicken rice, steamed",
"Lo Hong Ka chicken rice": "Chicken rice, steamed",
"Kwan Kee claypot rice": "Claypot chicken rice",
"Oriental Kopi kaya butter toast": "Kaya toast set",
"Oriental Kopi white coffee": "Latte, iced",
"Oriental Kopi egg tart": "Egg tart",
"Travelling Duck roast duck rice": "Roast duck rice",
"Dao kampung chicken rice": "Chicken rice, steamed",
"Antipodean big breakfast": "Big breakfast",
"Antipodean flat white": "Latte, iced",
"Ben's fish and chips": "Fish and chips",
"Ben's chicken pie": "Apple pie",
"The Barn fried chicken": "KFC Original Recipe, thigh",
"LingLing LongLong fusion rice bowl": "Economy rice: 1 meat, 2 veg",
"Christine's bun": "Char siu bao",
"Yin's sourdough sandwich": "Cheeseburger",
"Founders Bakery bun": "Char siu bao",
"Lapisan kek lapis": "Chocolate cake",
"Just Good Coffee iced latte": "Latte, iced",
"Nononsense iced latte": "Latte, iced",
"Kyo ha matcha latte": "Matcha latte, iced",
"Kyo ha matcha parfait": "Matcha latte, iced",
"dipndip chocolate crepe": "Waffle with ice cream",
"dipndip waffle with chocolate": "Waffle with ice cream",
"Creme de la creme fruit tart": "Fruit tart",
"Sukiya gyudon, regular": "Udon in soup",
"Sukiya cheese gyudon": "Udon in soup",
"Sukiya curry rice": "Chicken katsu curry rice",
"Jungle House chicken rice": "Chicken rice, steamed",
"B.O.D steamed bun with meat": "Char siu bao",
"Omiyage Japanese cheese tart": "Egg tart",
"Shae.co bagel sandwich": "Cheeseburger",
"Rightside burger": "Cheeseburger",
"Apollo rice set with 2 dishes": "Economy rice: 1 meat, 2 veg",
"Food hall mixed rice": "Economy rice: 1 meat, 2 veg",
"Squeezby fresh juice": "Soft drink",
"Red Box set lunch, chicken chop rice": "Hainanese chicken chop",
"Ke Ren Lai Hakka lei cha rice": "Economy rice: 1 meat, 2 veg",
"Ichiban tonkotsu ramen": "Ramen, tonkotsu",
"Mr Dakgalbi cheese dakgalbi, per person": "Korean BBQ, typical meal",
"Nong Geng Ji pepper fried pork": "Economy rice: 1 meat, 2 veg",
"Five Guys cheeseburger": "Cheeseburger",
"Five Guys little cheeseburger": "Cheeseburger",
"Five Guys bacon cheeseburger": "Cheeseburger",
"Five Guys Cajun fries, little": "French fries",
"Five Guys milkshake": "Chocolate milkshake",
"Gong Cha milk tea with pearls": "Bubble milk tea",
"Gong Cha milk foam green tea": "Bubble milk tea",
"Gong Cha brown sugar fresh milk with pearls": "Bubble milk tea",
"Gong Cha mango yakult": "Bubble milk tea",
"Luckin coconut latte, iced": "Latte, iced",
"Luckin Americano, iced": "Kopi C",
"Luckin velvet latte": "Latte, iced",
"Coolblog original milk tea with pearls": "Bubble milk tea",
"Coolblog ice blended chocolate": "Chocolate bar",
"Chicha San Chen Dong Ding oolong milk tea": "Bubble milk tea",
"Arabica Spanish latte, iced": "Latte, iced",
"Arabica cafe latte": "Latte, iced",
"SF Coffee cafe mocha": "Mocha, iced",
"SF Coffee chicken pie": "Apple pie",
"illy cappuccino": "Latte, iced",
"TWG tea, plain": "Kopi C",
"TWG macarons": "Chocolate chip cookie",
"TWG scone with jam and tea jelly": "Scone with jam and cream",
"Dome chicken pie": "Apple pie",
"Dome Dome burger": "Cheeseburger",
"Dome iced mocha": "Mocha, iced",
"PAUL croissant": "Butter croissant",
"PAUL croque monsieur": "Bagel with cream cheese",
"PAUL eclair": "Beard Papa original cream puff",
"Rotiboy coffee bun": "Char siu bao",
"Cold Stone Like It ice cream creation": "Ice cream",
"Gindaco takoyaki, original": "Takoyaki",
"Gindaco takoyaki, cheese mentai": "Takoyaki",
"Potato Corner flavoured fries, regular": "French fries",
"Tokyo Milk Cheese Factory cheese cookies": "Chocolate chip cookie",
"Royce Nama chocolate": "Chocolate bar",
"Godiva chocolixir": "Chocolate milkshake",
"Godiva soft serve cone": "Ice cream",
"Venchi gelato, 2 flavours in a cup": "Ice cream",
"Cioccolati Italiani gelato cone": "Ice cream",
"Harriston hot chocolate": "Chocolate bar",
"Melvados creme brulee cheesecake": "Cheesecake",
"Winky Walnut walnut cakes": "Chocolate chip cookie",
"Croissant taiyaki, custard": "Butter croissant",
"Honey Ring donut": "Doughnut",
"adough donut": "Doughnut",
"Cookiecrumbs cookie": "Chocolate chip cookie",
"Ning Ji hand-pounded lemon tea": "Bubble milk tea",
"Lam Heung Ling fragrant lemon tea": "Bubble milk tea",
"O'Cha matcha latte": "Matcha latte, iced",
"108 Matcha Saro matcha soft serve": "Ice cream",
"Matcha Hero matcha latte": "Matcha latte, iced",
"Loka iced latte": "Latte, iced",
"Hub Coffee iced latte": "Latte, iced",
"Ralph's iced latte": "Latte, iced",
"Cafe Kitsune iced latte": "Latte, iced",
"Cafe Kitsune fox shortbread": "Chocolate chip cookie",
"fRost n Bites frozen yogurt": "Mango lassi",
"Chinese egg roll biscuits": "Chocolate chip cookie",
"Eureka popcorn": "Popcorn, caramel",
"Leekor Leekor drink": "Bubble milk tea",
"Mama Kimbap tuna mayo kimbap": "Sushi plates",
"Mama Kimbap bulgogi kimbap": "Sushi plates",
"Oloiya Korean snack": "Tteokbokki",
"Tan Ngan Lo herbal tea": "Soft drink",
"Taimal herbal tea": "Soft drink",
"Nespresso espresso": "Kopi C",
"Putien sweet and sour pork with lychee": "Sweet and sour pork with rice",
"Putien Bian Rou soup (meat dumplings)": "Pan-fried pork dumplings",
"Kampachi bento set": "Chicken teriyaki don",
"Kampachi chirashi don": "Salmon sashimi and rice",
"Kampachi gyu teriyaki set": "Chicken teriyaki don",
"Uncle Roger egg fried rice": "Nasi goreng kampung",
"Uncle Roger fried chicken egg fried rice": "Nasi goreng kampung",
"Santouka shio ramen": "Ramen, tonkotsu",
"Santouka toroniku ramen": "Ramen, tonkotsu",
"Aori tonkotsu ramen": "Ramen, tonkotsu",
"Ichi Zen sashimi set": "Salmon sashimi and rice",
"Maisen kurobuta tonkatsu set": "Tonkatsu set",
"Maisen katsu sando": "Tonkatsu set",
"Donkai katsu don": "Tonkatsu set",
"Takeno udon set": "Udon in soup",
"Shojikiya beef sukiyaki set": "Hot pot, typical meal",
"Yamazo chicken teriyaki don": "Chicken teriyaki don",
"Yu Yan dim sum lunch, per person": "Dim sum breakfast",
"Dolly Dim Sum har gow": "Dim sum breakfast",
"Dolly Dim Sum siu mai": "Dim sum breakfast",
"Dolly Dim Sum char siu bun": "Char siu bao",
"Dolly Dim Sum chee cheong fun with prawns": "Dim sum breakfast",
"Dai Cha Dim dim sum, per person": "Dim sum breakfast",
"Canton Court roast duck": "Roast duck rice",
"Grand Imperial roast pork and duck rice": "Roast duck rice",
"Xu's Cuisine Hunan chili fish": "Steamed fish with rice and greens",
"Siwawa Guizhou wraps": "Bagel with cream cheese",
"Wuliguan hot pot, per person": "Hot pot, typical meal",
"Yang Guo Fu mala tang": "Hot pot, typical meal",
"Siong Tong Gai steamed chicken rice": "Chicken rice, steamed",
"JYNNS steamed chicken rice": "Chicken rice, steamed",
"On Lee wonton noodles": "Fried wantan",
"Hot Crush pork noodles": "Wantan mee, dry",
"Yee3 chicken rice": "Chicken rice, steamed",
"Yuen Kee pork and chive dumplings, boiled": "Pan-fried pork dumplings",
"Lanzhou beef lamian": "Pho",
"Samy & Min bak kut teh": "Bak kut teh with rice",
"Dirty duck with rice": "Roast duck rice",
"De.Wan nasi lemak with rendang": "Nasi lemak biasa",
"De.Wan ikan bakar set": "Ikan bakar with rice and ulam",
"Malaya Heritage nasi lemak": "Nasi lemak biasa",
"Serai nasi lemak with ayam berempah": "Nasi lemak biasa",
"Serai mee goreng": "Mee goreng mamak",
"Grandmama's nyonya fried rice": "Nasi goreng kampung",
"Grandmama's asam laksa": "Asam laksa",
"Celadon green curry chicken": "Green curry chicken with rice",
"Celadon tom yum goong": "Tom yum soup with rice",
"ParaThai pad kra pao chicken": "Pad kra pao with rice and egg",
"ParaThai mango sticky rice": "Mango sticky rice",
"Al Halabi mixed grill": "Satay ayam",
"Al-Amar chicken shawarma plate": "Bagel with cream cheese",
"Michelangelo's spaghetti carbonara": "Spaghetti carbonara",
"Michelangelo's margherita pizza": "Pizza",
"13DE Marzo pizza": "Pizza",
"La Boca beef tacos": "Bagel with cream cheese",
"KGB classic beef burger": "Cheeseburger",
"Weissbrau pork knuckle": "Steak with vegetables",
"Arthur's fish and chips": "Fish and chips",
"Two Sons mussels in white wine": "Butter prawns",
"O'Galito Portuguese grilled chicken": "Grilled chicken chop with salad",
"Ocean Twe12ve seafood pasta": "Spaghetti carbonara",
"Da On galbi set": "Korean BBQ, typical meal",
"Tomacado salad bowl": "Chicken caesar salad",
"LAC protein bowl": "Economy rice: 1 meat, 2 veg",
"Okonomi by Tokyo Don okonomiyaki": "Takoyaki",
"Okonomi by Tokyo Don salmon don": "Salmon sashimi and rice",
"Cucur udang": "Pisang goreng",
"A Decade nasi lemak ayam goreng": "Nasi lemak biasa",
"A Decade mee goreng mamak": "Mee goreng mamak",
"A Decade roti bakar with kaya": "Kaya toast set",
"ABC One chicken chop rice": "Hainanese chicken chop",
"ABC One fried rice": "Nasi goreng kampung",
"Bone soup rice noodles": "Pho",
"Spicy bone soup rice noodles": "Pho",
"Auntea Jenny Hong Kong lemon tea": "Bubble milk tea",
"Auntea Jenny milk tea with boba": "Bubble milk tea",
"Auntea Jenny mango pomelo sago": "Tau fu fah, less syrup",
"Ah Cheng Penang assam laksa": "Asam laksa",
"Ah Cheng curry laksa": "Curry laksa",
"Ah Cheng Ipoh hor fun": "Ipoh hor fun",
"An Viet pho bo": "Pho",
"An Viet bun cha": "Pho",
"An Viet goi cuon (fresh spring rolls)": "Pisang goreng",
"An Viet Vietnamese iced coffee": "Latte, iced",
"Andaman grilled seafood platter, per person": "Fish and chips",
"Andaman tom yum seafood soup": "Tom yum soup with rice",
"Ayam penyet with rice": "Nasi ayam penyet",
"Ayam bakar with rice": "Ikan bakar with rice and ulam",
"Lele penyet with rice": "Nasi ayam penyet",
"Bebek goreng with rice": "Nasi ayam penyet",
"Baskin Robbins single scoop, cup": "Ice cream",
"Baskin Robbins double scoop in a waffle cone": "Waffle with ice cream",
"Baskin Robbins ice cream cake": "Chocolate cake",
"Genki Sushi salmon nigiri": "Sushi plates",
"Genki Sushi typical meal": "Sushi plates",
"Genki Sushi salmon aburi with mayo": "Sushi plates",
"Sushi Jiro typical meal": "Sushi plates",
"Nanyang Cafe kaya butter toast set": "Kaya toast set",
"Nanyang Cafe curry mee": "Curry laksa",
"Nanyang Cafe dry chili pan mee": "Pan mee, soup",
"Taiwan spicy beef noodles": "Boat noodles",
"Taiwan braised pork rice": "Sweet and sour pork with rice",
"MET oolong milk tea": "Bubble milk tea",
"MET fruit tea": "Bubble milk tea",
"MET matcha latte": "Matcha latte, iced",
"Beutea brewed milk tea": "Bubble milk tea",
"Beutea cheese foam tea": "Bubble milk tea",
"Itsu salmon teriyaki bento": "Chicken teriyaki don",
"Itsu chicken katsu don": "Tonkatsu set",
"Kinori salmon nori taco": "Sushi plates",
"Kinori spicy tuna nori taco": "Sushi plates",
"Jordan HK baked pork chop rice": "Baked pork chop rice",
"Jordan HK satay beef noodles": "Boat noodles",
"Jordan HK French toast": "French toast",
"Jordan HK milk tea, iced": "Bubble milk tea",
"Gordon Ramsay beef Wellington": "Steak with vegetables",
"Gordon Ramsay sticky toffee pudding": "Chocolate cake",
"Gordon Ramsay Street Pizza margherita slice": "Pizza",
"Gordon Ramsay Street Pizza pepperoni slice": "Pizza",
"Thai Thai tom yum goong": "Tom yum soup with rice",
"Thai Thai pineapple fried rice": "Nasi goreng kampung",
"Thai Thai green curry chicken": "Green curry chicken with rice",
"Rakuzen bento set": "Chicken teriyaki don",
"Rakuzen chirashi don": "Salmon sashimi and rice",
"Din Tai Fung xiao long bao": "Xiao long bao",
"Din Tai Fung chicken xiao long bao": "Xiao long bao",
"Din Tai Fung egg fried rice with shrimp": "Nasi goreng kampung",
"Din Tai Fung braised beef noodle soup": "Minced pork soup",
"Din Tai Fung spicy wontons": "Fried wantan",
"Din Tai Fung stir-fried string beans": "Stir-fried greens",
"Dandy grilled chicken plate": "Hainanese chicken chop",
"Dandy beef burger": "Cheeseburger",
"Simple Life brown rice set": "Economy rice: 1 meat, 2 veg",
"Simple Life vegetarian laksa": "Curry laksa",
"Black Tap All-American burger": "Cheeseburger",
"Black Tap CrazyShake, cookies and cream": "Chocolate milkshake",
"Black Tap buffalo wings": "Korean fried chicken, soy garlic",
"Pokok big breakfast": "Big breakfast",
"Pokok iced latte": "Latte, iced",
"Pokok truffle fries": "French fries",
"Mantra chicken biryani": "Nasi briyani ayam",
"Mantra butter chicken with naan": "Butter chicken with naan",
"Mantra masala dosa": "Thosai with dhal and chutney",
"Verrona Hills croissant": "Butter croissant",
"Verrona Hills basque cheesecake": "Cheesecake",
"The Coach iced latte": "Latte, iced",
"Taiwanese popcorn chicken": "Chicken nuggets",
"Taiwanese XXL fried chicken": "Korean fried chicken, soy garlic",
"Walking Hotpot, per person": "Hot pot, typical meal",
"Johor mini steamboat, per person": "Hot pot, typical meal",
"Mr Bean soya milk": "Mango lassi",
"Mr Bean soya pancake with peanut": "Waffle with ice cream",
"Hikiniku To Come hamburg steak set": "Steak with vegetables",
"Le Bourne hot chocolate": "Chocolate bar",
"Le Bourne chocolate pralines": "Chocolate bar",
"DubuYo sundubu jjigae": "Army stew (budae jjigae)",
"DubuYo bibimbap": "Bibimbap",
"DubuYo topokki": "Tteokbokki",
"Koong Woh Tong herbal jelly": "Tau fu fah, less syrup",
"Koong Woh Tong herbal tea": "Soft drink",
"Chez Choux cream puff": "Beard Papa original cream puff",
"Beryl's chocolate lava cake": "Chocolate cake",
"Beryl's hot chocolate": "Chocolate bar",
"Miga beef bulgogi set": "Bibimbap",
"Wok Thai pad kra pao chicken": "Pad kra pao with rice and egg",
"Wok Thai pad see ew": "Char kway teow / Kway teow goreng",
"Torii teppanyaki chicken set": "Chicken teriyaki don",
"BreadStory sausage bun": "Char siu bao",
"BreadStory cream bun": "Char siu bao",
"Ramen Matsuri chicken shoyu ramen": "Ramen, tonkotsu",
"Ramen Matsuri spicy miso ramen": "Ramen, tonkotsu",
"Santan nasi lemak": "Nasi lemak biasa",
"Santan Pak Nasser's nasi lemak": "Nasi lemak biasa",
"Chizu cheese foam tea": "Bubble milk tea",
"Village roast duck rice": "Roast duck rice",
"Village roast duck, quarter": "Roast duck rice",
"Aftermeal waffle with ice cream": "Waffle with ice cream",
"Whopper Jr": "Cheeseburger",
"Double Whopper with cheese": "Cheeseburger",
"Mushroom Swiss beef burger": "Cheeseburger",
"Tendercrisp chicken burger": "Cheeseburger",
"Burger King chicken royale": "Cheeseburger",
"Burger King fish burger": "Cheeseburger",
"Burger King onion rings": "Cheeseburger",
"Burger King fries": "Cheeseburger",
"Burger King nuggets": "Chicken nuggets",
"Burger King sundae, chocolate": "Cheeseburger",
"6-inch Subway Melt": "Bagel with cream cheese",
"6-inch Italian B.M.T. (chicken ham, pepperoni, salami)": "Bagel with cream cheese",
"6-inch Chicken Slice sub": "Bagel with cream cheese",
"6-inch Steak & Cheese sub": "Steak with vegetables",
"6-inch Spicy Italian sub": "Bagel with cream cheese",
"6-inch egg mayo sub": "Bagel with cream cheese",
"6-inch seafood sensation sub": "Bagel with cream cheese",
"Subway salad, roasted chicken": "Chicken caesar salad",
"Subway chocolate chip cookie": "Chocolate chip cookie",
"Nando's half chicken": "Grilled chicken chop with salad",
"Nando's quarter chicken breast with 2 regular sides": "Grilled chicken chop with salad",
"Nando's chicken butterfly": "Grilled chicken chop with salad",
"Nando's chicken wrap": "Bagel with cream cheese",
"Nando's chicken burger": "Cheeseburger",
"Nando's peri-peri chips": "French fries",
"Nando's spicy rice": "Stir-fried greens",
"Nando's macho peas": "Stir-fried greens",
"Nando's corn on the cob": "Stir-fried greens",
"Texas Chicken spicy chicken, 2 pieces": "KFC Original Recipe, thigh",
"Texas Chicken tenders": "Chicken nuggets",
"Texas Chicken sandwich": "Cheeseburger",
"Texas Chicken honey butter rice bowl": "Korean fried chicken, soy garlic",
"Texas Chicken mashed potato": "Stir-fried greens",
"Texas Chicken cajun fries": "French fries",
"4Fingers wings, soy garlic": "Korean fried chicken, soy garlic",
"4Fingers wings, hot and spicy": "Korean fried chicken, soy garlic",
"4Fingers drumsticks, soy garlic": "Korean fried chicken, soy garlic",
"4Fingers crispy chicken burger": "Cheeseburger",
"4Fingers seaweed fries": "French fries",
"A&W Mozza burger": "Cheeseburger",
"A&W Teen burger": "Cheeseburger",
"A&W chicken, 2 pieces": "KFC Original Recipe, thigh",
"A&W waffle with ice cream": "Waffle with ice cream",
"A&W curly fries": "French fries",
"A&W root beer": "Soft drink",
"Auntie Anne's pretzel, almond": "Pretzel, original",
"Auntie Anne's pretzel, sour cream and onion": "Pretzel, original",
"Auntie Anne's pretzel dogs": "Pretzel, original",
"Auntie Anne's pretzel bites, original": "Pretzel, original",
"Auntie Anne's lemonade": "Soft drink",
"Secret Recipe Marble Cheese cake": "Cheesecake",
"Secret Recipe Chocolate Indulgence": "Chocolate cake",
"Secret Recipe Durian cake": "Chocolate cake",
"Secret Recipe tom yam fried rice": "Nasi goreng kampung",
"Secret Recipe chicken lasagna": "Spaghetti carbonara",
"Secret Recipe nasi lemak with chicken rendang": "Nasi lemak biasa",
"Secret Recipe fish and chips": "Fish and chips",
"PappaRich nasi lemak with curry chicken": "Nasi lemak biasa",
"PappaRich curry laksa": "Curry laksa",
"PappaRich wantan mee": "Wantan mee, dry",
"PappaRich roti canai with curry": "Roti canai kosong",
"PappaRich chicken chop rice": "Hainanese chicken chop",
"PappaRich char kway teow": "Char kway teow / Kway teow goreng",
"PappaRich white coffee, iced": "Latte, iced",
"Kenny Rogers quarter chicken with 2 sides": "Grilled chicken chop with salad",
"Kenny Rogers half chicken with 3 sides": "Grilled chicken chop with salad",
"Kenny Rogers muffin": "Scone with jam and cream",
"Kenny Rogers mac and cheese": "Spaghetti carbonara",
"Boat Noodle beef noodles": "Boat noodles",
"Boat Noodle tom yam noodles": "Tom yum soup with rice",
"Boat Noodle pad thai": "Pad thai",
"Boat Noodle Thai milk tea": "Bubble milk tea",
"Boat Noodle mango sticky rice": "Mango sticky rice",
"Longjing xiao long bao": "Xiao long bao",
"Longjing shrimp fried rice": "Nasi goreng kampung",
"Chakri massaman chicken curry": "Green curry chicken with rice",
"Chakri basil minced pork rice": "Pad kra pao with rice and egg",
"Cili Kampung ikan keli berlado": "Nasi ayam penyet",
"Absolute Thai tom yum seafood": "Tom yum soup with rice",
"Absolute Thai basil chicken rice": "Chicken rice, steamed",
"Chica Bonita chicken tacos": "Bagel with cream cheese",
"Chica Bonita beef burrito": "Bagel with cream cheese",
"Chica Bonita nachos with cheese": "French fries",
"Robataya grilled skewers, per person": "Satay ayam",
"Robataya grilled saba set": "Steamed fish with rice and greens",
"Matcha Eight matcha latte": "Matcha latte, iced",
"Matcha Eight matcha cake": "Matcha latte, iced",
"Teochew braised duck, per person": "Roast duck rice",
"Teochew steamed pomfret, per person": "Steamed fish with rice and greens",
"Teochew porridge with sides": "Chicken porridge",
"Elementary fish and chips": "Fish and chips",
"Elementary beef pie": "Apple pie",
"Las Meninas seafood paella, per person": "Nasi goreng kampung",
"Quivo Mediterranean grilled chicken": "Grilled chicken chop with salad",
"a'rest iced latte": "Latte, iced",
"Reborn iced latte": "Latte, iced",
"36 Degree jalapeno bagel": "Bagel with cream cheese",
"All About Chew chocolate chip cookie": "Chocolate chip cookie",
"AiFOKATO affogato": "Ice cream",
"Ticco milk tea": "Bubble milk tea",
"Nak Nak boneless thigh, soy garlic": "Korean fried chicken, soy garlic",
"Nak Nak boneless thigh, gochujang": "Korean fried chicken, soy garlic",
"Nak Nak beef burger": "Cheeseburger",
"Nak Nak loaded fries": "French fries",
"Fong Lye three cup chicken rice": "Chicken rice, steamed",
"Fong Lye beef noodle soup": "Minced pork soup",
"Rooster& chicken rice": "Chicken rice, steamed",
"Meriah platter": "Latte, iced",
"Meriah banana fritters": "Pisang goreng",
"Meriah chilled mango sago": "Tau fu fah, less syrup",
"DayOne katsu curry pork chop omelette rice": "Chicken katsu curry rice",
"DayOne salmon don": "Salmon sashimi and rice",
"DayOne teriyaki chicken rice": "Chicken rice, steamed",
"Kanbe kuro ramen": "Ramen, tonkotsu",
"Steamed chicken rice set": "Chicken rice, steamed",
"Hotcakes with syrup": "Pancakes with syrup",
"Apple pie (McD)": "Apple pie",
"Roti boom Milo": "Iced chocolate",
"Cappuccino": "Latte",
"Flat white": "Latte",
"Latte with oat milk, iced": "Latte, iced",
"Caramel macchiato, iced": "Latte, iced",
"Gula melaka latte, iced": "Latte, iced",
"Americano, iced": "Latte, iced",
"Espresso": "Latte",
"Kopi O": "Latte",
"Kopi C kosong": "Latte",
"Teh C": "Teh tarik",
"Teh O": "Teh tarik",
"Teh O kosong": "Teh tarik",
"Teh tarik kurang manis": "Teh tarik",
"Milo ais kurang manis": "Iced chocolate",
"Milo dinosaur": "Iced chocolate",
"Bubble milk tea, 0% sugar": "Bubble milk tea",
"Bubble milk tea, 25% sugar": "Bubble milk tea",
"Bubble milk tea, 100% sugar": "Bubble milk tea",
"Bubble milk tea, large": "Bubble milk tea",
"Matcha milk tea with pearls": "Bubble milk tea",
"Chocolate milk tea with pearls": "Bubble milk tea",
"Fruit tea with pearls": "Bubble milk tea",
"Popcorn chicken": "Popcorn, caramel",
"KitKat, 2 fingers": "Cadbury Dairy Milk, whole bar",
"KitKat, 4 fingers": "Cadbury Dairy Milk, whole bar",
"KitKat Chunky": "Cadbury Dairy Milk, whole bar",
"KitKat Chunky Peanut Butter": "Cadbury Dairy Milk, whole bar",
"KitKat Dark": "Cadbury Dairy Milk, whole bar",
"KitKat Matcha": "Cadbury Dairy Milk, whole bar",
"KitKat Strawberry": "Cadbury Dairy Milk, whole bar",
"KitKat White": "Cadbury Dairy Milk, whole bar",
"Cadbury Dairy Milk": "Cadbury Dairy Milk, whole bar",
"Cadbury Dairy Milk Fruit & Nut": "Cadbury Dairy Milk, whole bar",
"Cadbury Dairy Milk Roast Almond": "Cadbury Dairy Milk, whole bar",
"Cadbury Dairy Milk Hazelnut": "Cadbury Dairy Milk, whole bar",
"Cadbury Dairy Milk Black Forest": "Cadbury Dairy Milk, whole bar",
"Cadbury Dairy Milk Caramello": "Cadbury Dairy Milk, whole bar",
"Cadbury Dairy Milk Oreo": "Cadbury Dairy Milk, whole bar",
"Cadbury Dairy Milk, mini bar": "Cadbury Dairy Milk, whole bar",
"Tim Tam Dark Chocolate": "Soft drink, zero sugar",
"Pocky Chocolate": "Ritz crackers",
"Pocky Strawberry": "Ritz crackers",
"Pocky Matcha": "Ritz crackers",
"Pocky Cookies & Cream": "Ritz crackers",
"Pocky Banana": "Ritz crackers",
"Pocky Almond Crush": "Ritz crackers",
"Oreo Original": "Ritz crackers",
"Oreo Strawberry Creme": "Ritz crackers",
"Oreo Chocolate Creme": "Ritz crackers",
"Oreo Peanut Butter": "Ritz crackers",
"Oreo Blueberry Ice Cream": "Ritz crackers",
"Oreo Thins": "Ritz crackers",
"Oreo Mini": "Ritz crackers",
"Ferrero Rocher": "Cadbury Dairy Milk, whole bar",
"Kinder Bueno": "Cadbury Dairy Milk, whole bar",
"Kinder Bueno White": "Cadbury Dairy Milk, whole bar",
"Kinder Joy": "Cadbury Dairy Milk, whole bar",
"Kinder Chocolate": "Cadbury Dairy Milk, whole bar",
"Toblerone": "Cadbury Dairy Milk, whole bar",
"Snickers": "Cadbury Dairy Milk, whole bar",
"M&M's Milk Chocolate": "Cadbury Dairy Milk, whole bar",
"M&M's Peanut": "Cadbury Dairy Milk, whole bar",
"Hello Panda Chocolate": "Soft drink, zero sugar",
"Yan Yan Chocolate": "Soft drink, zero sugar",
"Koala's March Chocolate": "Soft drink, zero sugar",
"Lotte Choco Pie": "Ritz crackers",
"Chipsmore Double Chocolate": "Soft drink, zero sugar",
"Julie's Love Letters": "Ritz crackers",
"Hup Seng Cream Crackers": "Ritz crackers",
"Munchy's Oat Krunch Chocolate": "Soft drink, zero sugar",
"Munchy's Oat Krunch Dark Chocolate": "Soft drink, zero sugar",
"Apollo Chocolate Layer Cake": "Ritz crackers",
"Apollo Pandan Layer Cake": "Ritz crackers",
"Mamee Monster": "Potato chips",
"Twisties Cheese": "Potato chips",
"Twisties Chicken": "Potato chips",
"Mister Potato Original": "Potato chips",
"Mister Potato Hot & Spicy": "Potato chips",
"Mister Potato Barbecue": "Potato chips",
"Super Ring Cheese": "Potato chips",
"Pringles Original": "Potato chips",
"Pringles Sour Cream & Onion": "Potato chips",
"Pringles Barbecue": "Potato chips",
"Pringles Cheesy Cheese": "Potato chips",
"Pringles Hot & Spicy": "Potato chips",
"Mamee Monster Barbecue": "Potato chips",
"Mamee Monster Hot & Spicy": "Potato chips",
"Mamee Monster, big pack": "Potato chips",
"Mamee Snek Ku": "Potato chips",
"Twisties Barbecue Curry": "Potato chips",
"Twisties Tomato": "Potato chips",
"Twisties, small pack": "Potato chips",
"Cheezels": "Potato chips",
"Tora corn snack": "Potato chips",
"Jacker potato chips": "Potato chips",
"Choki Choki": "Cadbury Dairy Milk, whole bar",
"Beryl's chocolate": "Cadbury Dairy Milk, whole bar",
"Yin's sourdough loaf slice": "Butter cake",
"Nasi goreng kampung, home-style": "Nasi goreng kampung",
"Tom yum seafood soup with rice": "Tom yum soup with rice",
"Spaghetti bolognese, home-style": "Spaghetti bolognese",
"Ikan kembung bakar with ulam and rice": "Ikan bakar with rice and ulam",
"Chicken soup noodles": "Chicken noodle soup",
"Boiled eggs": "Boiled egg",
"Plain yogurt": "Flavoured yogurt cup",
"Greek yogurt": "Flavoured yogurt cup",
"Wholemeal crackers with cheese": "Ritz crackers",
"Teh O ais kurang manis": "Teh tarik",
"Teh C kosong": "Teh tarik",
"Kopi (condensed milk)": "Latte",
"Kopi O kosong": "Latte",
"Milo kosong": "Iced chocolate",
"Milk tea, 0% sugar, no pearls": "Bubble milk tea",
"Jasmine green milk tea, less sugar": "Bubble milk tea",
"Spanish latte": "Latte",
"White coffee": "Latte",
"100 Plus": "Soft drink, zero sugar",
"Nasi lemak ayam goreng (Village Park)": "Nasi lemak ayam goreng",
"Lontong (Village Park)": "Lontong",
"Nasi lemak ayam goreng (Nasi Lemak Bumbung)": "Nasi lemak ayam goreng",
"Claypot chicken rice (Uncle Mani's)": "Claypot chicken rice",
"Hainanese chicken chop (Yut Kee)": "Hainanese chicken chop",
"Yakult Ace Light": "Yakult Ace",
"Vitagen Less Sugar": "Yakult Ace",
"Chocolate milk, packet": "Soft drink, zero sugar",
"Whey protein shake, with water": "Caramel frappé",
"Whey protein shake, with low-fat milk": "Caramel frappé",
"Gold Standard Whey shake, with water": "Caramel frappé",
"Herbalife Formula 1 shake": "Caramel frappé",
"Dutch Lady +Protein milk": "Milo, packet drink",
"Barebells Protein Milkshake": "Caramel frappé",
"Soy milk, sweetened packet": "Milo, packet drink",
"Yogurt drink": "Flavoured yogurt cup",
"Chobani Greek yogurt, strawberry": "Flavoured yogurt cup",
"Chobani Greek yogurt, mango": "Flavoured yogurt cup",
"Chobani Greek yogurt, blueberry": "Flavoured yogurt cup",
"Chobani Greek yogurt, passion fruit": "Flavoured yogurt cup",
"Chobani Greek yogurt, natural": "Flavoured yogurt cup",
"Chobani Greek yogurt, natural light": "Flavoured yogurt cup",
"FamilyMart Sofuto, cone": "Chocolate sundae",
"FamilyMart Sofuto, cup": "Chocolate sundae",
"FamilyMart Sofuto parfait": "Chocolate sundae",
"FamilyMart fruit sando": "Fruit sando",
"FamilyMart chicken teriyaki don": "Chicken teriyaki don",
"FamilyMart spaghetti bolognese": "Spaghetti bolognese",
"FamilyMart spaghetti carbonara": "Spaghetti carbonara",
"FamilyMart fried chicken": "Fried chicken",
"FamilyMart chicken karaage": "Chicken karaage",
"FamilyMart daifuku mochi": "Mochi",
"FamilyMart roll cake": "White bread",
"FamilyMart iced matcha latte": "Matcha latte, iced",
"FamilyMart iced latte": "Latte, iced",
"7-Eleven spaghetti bolognese": "Spaghetti bolognese",
"7-Eleven hot dog": "Hot dog",
"7-Eleven Slurpee, large": "Soft drink, zero sugar",
"Gardenia Original Classic bread": "White bread",
"Gardenia wholemeal bread": "White bread",
"Gardenia Twiggies": "White bread",
"Gardenia Delicia butterscotch roll": "White bread",
"Massimo sandwich loaf": "Butter cake",
"Farm Fresh fresh milk": "Milo, packet drink",
"Farm Fresh chocolate milk": "Soft drink, zero sugar",
"Farm Fresh yogurt drink": "Milo, packet drink",
"Oatside oat milk, barista blend": "Milo, packet drink",
"Oatside oat milk, chocolate": "Soft drink, zero sugar",
"Teh O limau ais": "Teh tarik",
"Golden bubble milk tea": "Bubble milk tea",
"Tea macchiato": "Latte",
"Jasmine green milk tea, large, normal sugar": "Bubble milk tea",
"Da Hong Pao milk tea, less sugar": "Bubble milk tea",
"Peach oolong milk tea, less sugar": "Bubble milk tea",
"Vanilla milkshake": "Caramel frappé",
"Ais krim potong": "Ice cream cone, packaged",
"Chocolate-coated ice cream stick": "Soft drink, zero sugar",
"Ice cream cone, packaged": "Ice cream",
"Caramel pudding": "Mango pudding",
"Corn snack pack": "Potato chips",
"Egg custard bun (nai wong bao)": "Mango pudding",
"Chicken ham and cheese croissant": "Butter croissant",
"Filled doughnut": "Doughnut",
"Wholemeal bread": "White bread",
"Mochi cheesecake": "Cheesecake",
"Brown sugar boba milk (Tiger Sugar)": "Bubble milk tea",
"Rekindle Double Fromage": "Cheesecake",
"Rekindle Rose Lychee cake": "Strawberry shortcake",
"Rekindle mango panna cotta": "Mango pudding",
"Rekindle matcha panna cotta": "Mango pudding",
"Nadeje mille crepe, original": "Mille crepe, original",
"Nadeje mille crepe, green tea": "Mille crepe, original",
"Nadeje mille crepe, cheese": "Mille crepe, original",
"Nadeje mille crepe, tiramisu": "Mille crepe, original",
"Nadeje mille crepe, strawberry": "Mille crepe, original",
"Nadeje mille crepe, rum and raisin": "Mille crepe, original",
"Nadeje mille crepe, mango yogurt": "Mille crepe, original",
"Nadeje mille crepe, chocolate": "Mille crepe, original",
"Nadeje mille crepe, royal milk tea": "Mille crepe, original",
"Starbucks coffee burnt cheesecake": "Burnt cheesecake",
"Starbucks tiramisu": "Tiramisu",
"Starbucks classic chocolate cake": "Chocolate cake",
"Starbucks red velvet cake": "Red velvet cake",
"Starbucks New York cheesecake": "Cheesecake",
"Starbucks blueberry muffin": "Blueberry muffin",
"Starbucks chocolate muffin": "Blueberry muffin",
"Starbucks cinnamon roll": "Cinnamon roll",
"Coffee Bean Chicago cheesecake": "Cheesecake",
"Coffee Bean Royal chocolate cake": "Chocolate cake",
"Secret Recipe Chocolate Fudge cake": "Chocolate cake",
"Secret Recipe Moist Chocolate cake": "Chocolate cake",
"Secret Recipe White Choc Macadamia cake": "Chocolate cake",
"Secret Recipe Hazelnut Chocolate cake": "Chocolate cake",
"Secret Recipe Chocolate Banana cake": "Chocolate cake",
"Secret Recipe Chocolate Chip Walnut cake": "Chocolate cake",
"Secret Recipe Chocolate Brulee cake": "Chocolate cake",
"Secret Recipe Chocolate Odyssey cake": "Chocolate cake",
"Secret Recipe Caramel Creme Choc cake": "Chocolate cake",
"Secret Recipe Black Forest cake": "Strawberry shortcake",
"Secret Recipe Thai Milk Tea cake": "Bubble milk tea",
"Secret Recipe Dark Cookies and Cream cake": "Strawberry shortcake",
"Secret Recipe Carrot cake": "Carrot cake with cream cheese",
"Secret Recipe Confetti Rainbow cake": "Strawberry shortcake",
"Secret Recipe Decadent Tiramisu cake": "Tiramisu",
"Secret Recipe New York cheesecake": "Cheesecake",
"Secret Recipe Wild Blueberry cheesecake": "Cheesecake",
"Secret Recipe Salted Caramel Macadamia cheesecake": "Cheesecake",
"Secret Recipe Red Velvet cake": "Red velvet cake",
"Secret Recipe Burnt cheesecake": "Burnt cheesecake",
"Secret Recipe Choc Cheese Berries cake": "Chocolate cake",
"Secret Recipe Hokkaido Triple cheesecake": "Cheesecake",
"Secret Recipe Lotus Biscoff Cheese cake": "Cheesecake",
"Secret Recipe Royal Chocolate cake": "Chocolate cake",
"Secret Recipe Boston Chocolate cake": "Chocolate cake",
"Secret Recipe walnut brownie": "Brownie",
"Burnt cheesecake, matcha": "Burnt cheesecake",
"Burnt cheesecake, chocolate": "Burnt cheesecake",
"Japanese cotton cheesecake": "Cheesecake",
"Oreo cheesecake": "Cheesecake",
"Blueberry cheesecake": "Cheesecake",
"Fresh cream fruit cake": "Strawberry shortcake",
"Black forest cake": "Strawberry shortcake",
"Chocolate mousse cake": "Strawberry shortcake",
"Opera cake": "Chocolate cake",
"Mango mousse cake": "Strawberry shortcake",
"Yam cream cake": "Strawberry shortcake",
"Durian mille crepe": "Mille crepe, original",
"Mochi cake": "Mochi",
"Strawberry daifuku": "Mochi",
"Warabi mochi": "Mochi",
"Matcha roll cake": "Swiss roll",
"Orange chiffon cake": "Pandan chiffon cake",
"Banana cake": "Butter cake",
"Marble cake": "Butter cake",
"Fruit cake": "Butter cake",
"Cream horn": "Cream puff",
"Choux puff with craquelin": "Cream puff",
"Profiteroles with chocolate sauce": "Cream puff",
"Kouign-amann": "Butter croissant",
"Cube croissant, filled": "Butter croissant",
"Chocolate tart": "Fruit tart",
"Lemon meringue tart": "Fruit tart",
"Creme brulee": "Mango pudding",
"Panna cotta": "Mango pudding",
"Trifle": "Mango pudding",
"Bread and butter pudding": "Mango pudding",
"Sticky date pudding": "Mango pudding",
"Lamington": "Cupcake",
"Cake pop": "Cupcake",
"Levain-style cookie": "Chocolate chip cookie",
"White chocolate macadamia cookie": "Chocolate chip cookie",
"Korean mochi bread": "Mochi",
"Grapes, small handful": "Grapes, purple seedless",
"Durian": "Musang King durian",
"Lamb skewer": "Lamb skewers (BBQ)",
"Lamb fat skewer": "Lamb skewers (BBQ)",
"Beef skewer": "Lamb skewers (BBQ)",
"Beef tendon skewer": "Lamb skewers (BBQ)",
"Pork belly skewer": "Lamb skewers (BBQ)",
"Pork skewer": "Lamb skewers (BBQ)",
"Pork intestine skewer": "Lamb skewers (BBQ)",
"Pork kidney skewer": "Lamb skewers (BBQ)",
"Chicken skewer": "Lamb skewers (BBQ)",
"Chicken skin skewer": "Lamb skewers (BBQ)",
"Chicken heart skewer": "Lamb skewers (BBQ)",
"Chicken gizzard skewer": "Lamb skewers (BBQ)",
"Chicken cartilage skewer": "Lamb skewers (BBQ)",
"Chicken intestine skewer": "Lamb skewers (BBQ)",
"Duck tongue skewer": "Lamb skewers (BBQ)",
"Fish tofu skewer": "Lamb skewers (BBQ)",
"Bacon enoki roll skewer": "Lamb skewers (BBQ)",
"Grilled chives skewer": "Lamb skewers (BBQ)",
"Tofu skin skewer": "Lamb skewers (BBQ)",
"Potato slice skewer": "Lamb skewers (BBQ)",
"Lotus root skewer": "Lamb skewers (BBQ)",
"Eggplant skewer": "Lamb skewers (BBQ)",
"Satay ayam, 1 stick": "Satay ayam",
"Satay daging, 1 stick": "Satay ayam",
"Satay kambing, 1 stick": "Satay ayam",
"Satay perut, 1 stick": "Satay ayam",
"Satay hati, 1 stick": "Satay ayam",
"Pork satay, 1 stick": "Satay ayam",
"Satay daging": "Satay ayam",
"Satay kambing": "Satay ayam",
"Peanut sauce for satay": "Satay ayam",
"Nasi impit": "Satay ayam",
"Sate kambing (Indonesian)": "Satay ayam",
"Satay celup": "Satay ayam",
"Yakitori momo (thigh)": "Yakitori",
"Yakitori negima (thigh and leek)": "Yakitori",
"Tsukune (chicken meatball)": "Yakitori",
"Yakitori kawa (skin)": "Yakitori",
"Tebasaki (wing)": "Yakitori",
"Yakitori reba (liver)": "Yakitori",
"Yakitori hatsu (heart)": "Yakitori",
"Sunagimo (gizzard)": "Yakitori",
"Nankotsu (cartilage)": "Yakitori",
"Buta bara (pork belly)": "Yakitori",
"Bacon asparagus skewer": "Yakitori",
"Gyutan (beef tongue) skewer": "Yakitori",
"Shiitake skewer": "Yakitori",
"Shishito pepper skewer": "Yakitori",
"Korean lamb skewers (yang-kkochi)": "Lamb skewers (BBQ)",
"Moo ping, 1 skewer": "Moo ping with sticky rice",
"Gai ping, 1 skewer": "Moo ping with sticky rice",
"Look chin ping (meatball skewer)": "Moo ping with sticky rice",
"Lamb kebab, 1 skewer": "Lamb skewers (BBQ)",
"Beef shish kebab, 1 skewer": "Lamb skewers (BBQ)",
"Lok lok fish ball, 1 stick": "Lok lok",
"Lok lok sausage, 1 stick": "Lok lok",
"Lok lok crab stick, 1 stick": "Lok lok",
"Lok lok quail eggs, 1 stick": "Lok lok",
"Lok lok cockles, 1 stick": "Lok lok",
"Lok lok pork belly, 1 stick": "Lok lok",
"Lok lok fried wonton, 1 stick": "Lok lok",
"Lok lok vegetables, 1 stick": "Lok lok",
"Lok lok satay sauce": "Lok lok"
}
SHOP_EXTRA = {
"korean bbq samgyeopsal grill": "Korean BBQ, typical meal",
"yakiniku grill table": "Korean BBQ, typical meal",
"xiao long bao": "Xiao long bao",
"beef wellington": "Steak with vegetables",
"pad kra pao": "Pad kra pao with rice and egg",
"mango sticky rice": "Mango sticky rice",
"fried chicken wings korean": "Korean fried chicken, soy garlic",
"hong kong pineapple bun": "Pineapple bun with butter",
"baked pork chop rice hong kong": "Baked pork chop rice",
"unagi don": "Unagi don",
"pan fried dumplings": "Pan-fried pork dumplings",
"croissant": "Butter croissant",
"tiramisu": "Tiramisu",
"iced latte": "Latte, iced",
"big breakfast plate": "Big breakfast",
"ipoh hor fun": "Ipoh hor fun"
}
SHOP_EXTRA.update(V9_EXTRA)
SHOP_PER_DISH = 30
by_look = {}
for q, look in list(SHOP_LOOKS.items()) + list(SHOP_EXTRA.items()):
    by_look.setdefault(look, []).append(q)
if WEB_PHOTOS:
    for look, qs in by_look.items():
        lab = "sh_" + re.sub(r"[^a-z0-9]+", "_", look.lower()).strip("_"); d = f"/tmp/shops/{lab}"; os.makedirs(d, exist_ok=True)
        jobs = []
        for qi, q in enumerate(qs):
            jobs += [(u, f"{d}/{qi}_{j}.jpg") for j, u in enumerate(web_photos(q, V9_PER_QUERY if q in V9_EXTRA else SHOP_PER_DISH))]; time.sleep(1.5)
        with ThreadPoolExecutor(6) as pool: files = [f for f in pool.map(grab, jobs) if f]
        if len(files) >= 25: MAP_CM[lab] = look; CM_FILES[lab] = files
        print(f"shops, {look}: {len(files)} photos from {len(qs)} dishes" + ("" if len(files) >= 25 else "  (too few, left out)"))

# ---- photos of one base or one protein by itself (plain bee hoon, fried chicken pieces) ----
PO_TAGS = {}
if WEB_PHOTOS:
    for q, tg in PART_ONLY.items():
        lab = "po_" + re.sub(r"[^a-z0-9]+", "_", q.lower()).strip("_"); d = f"/tmp/parts/{lab}"; os.makedirs(d, exist_ok=True)
        jobs = [(u, f"{d}/{j}.jpg") for j, u in enumerate(web_photos(q, 80))]; time.sleep(1.5)
        with ThreadPoolExecutor(6) as pool: files = [f for f in pool.map(grab, jobs) if f]
        if len(files) >= 25: CM_FILES[lab] = files; PO_TAGS[lab] = tg
        print(f"parts, {q}: {len(files)} photos" + ("" if len(files) >= 25 else "  (too few, left out)"))

# ---- drop web photos that are not food (menus, logos, shop fronts, people) ----
if CLEAN_WEB and CM_FILES:
    import open_clip
    cm, _, cpre = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k", device=DEVICE); cm.eval()
    tok = open_clip.get_tokenizer("ViT-B-32")
    prompts = ["a photo of food", "a photo of a dish of food", "a photo of a drink",
               "a restaurant menu", "a photo of text", "a logo", "a photo of a person", "a shop front", "a screenshot", "a cartoon"]
    with torch.no_grad():
        tf_ = cm.encode_text(tok(prompts).to(DEVICE)); tf_ /= tf_.norm(dim=-1, keepdim=True)
    def foodness(fns):
        out = []
        for k in range(0, len(fns), 128):
            ims = []
            for fn in fns[k:k + 128]:
                try: ims.append(cpre(Image.open(fn).convert("RGB")))
                except Exception: ims.append(torch.zeros(3, 224, 224))
            with torch.no_grad(), torch.autocast(device_type=DEVICE, enabled=DEVICE == "cuda"):
                f = cm.encode_image(torch.stack(ims).to(DEVICE)); f = f / f.norm(dim=-1, keepdim=True)
                pr = (100 * f @ tf_.T).softmax(-1)[:, :3].sum(-1)
            out += pr.float().cpu().tolist()
        return out
    before = sum(len(v) for v in CM_FILES.values())
    for lab in list(CM_FILES):
        fs = CM_FILES[lab]; sc = foodness(fs); keep = [f for f, p in zip(fs, sc) if p >= 0.5]
        if len(keep) >= 25: CM_FILES[lab] = keep
        else: CM_FILES.pop(lab); MAP_CM.pop(lab, None); print("left out after cleaning:", lab)
    print(f"web photos kept after cleaning: {sum(len(v) for v in CM_FILES.values())} of {before}")
    del cm; torch.cuda.empty_cache()

PO_FILES = {k: CM_FILES.pop(k) for k in list(CM_FILES) if k.startswith("po_")}   # parts-only photos are not a dish class

CLASSES = sorted(set(MAP_MY.values()) | set(MAP_F101.values()) | set(MAP_IN.values()) | set(MAP_CM.values()))   # one class per dish in the app, so the same dish from two sources is one class
IDX = {c: i for i, c in enumerate(CLASSES)}
print(len(CLASSES), "classes")

PART_IDX = {p: i for i, p in enumerate(PART_NAMES)}
PO_LABS = sorted(PO_FILES)
def part_vec(ts):
    v = torch.zeros(len(PART_NAMES))
    for t in ts: v[PART_IDX[t]] = 1.0
    return v
PT = torch.stack([part_vec(TAGS_OF.get(c, [])) for c in CLASSES] + [part_vec(PO_TAGS[l]) for l in PO_LABS])   # row y = the parts of label y
print(sum(1 for c in CLASSES if TAGS_OF.get(c)), "dishes have parts,", len(PO_LABS), "parts-only photo sets")

class Files(Dataset):
    def __init__(self, rows, tf): self.rows, self.tf = rows, tf
    def __len__(self): return len(self.rows)
    def __getitem__(self, k):
        fn, y = self.rows[k]
        return self.tf(Image.open(fn).convert("RGB")), y

cm_train, cm_val = [], []
for lab, files in CM_FILES.items():
    random.Random(SEED).shuffle(files); nv = min(80, max(5, len(files) // 5))
    y = IDX[MAP_CM[lab]]; cm_val += [(fn, y) for fn in files[:nv]]; cm_train += [(fn, y) for fn in files[nv:nv + PER_CLASS]]   # capped so big groups don't drown the rest
print(len(cm_train), "extra training photos from Wikimedia Commons")


def pick(ds, names, mapping, cap, seed=SEED):
    """Row indices for the wanted classes, at most `cap` per class."""
    labels = ds["label"]                                # reads the label column only, no image decoding
    by = {}
    for i, l in enumerate(labels):
        n = names[l]
        if n in mapping: by.setdefault(n, []).append(i)
    rng = random.Random(seed); out = []
    for n, idx in by.items():
        rng.shuffle(idx); out += [(i, IDX[mapping[n]]) for i in idx[:cap]]
    return out

class Photos(Dataset):
    def __init__(self, ds, rows, tf): self.ds, self.rows, self.tf = ds, rows, tf
    def __len__(self): return len(self.rows)
    def __getitem__(self, k):
        for step in range(20):   # a few photos in the public datasets are broken files: skip to the next one
            i, y = self.rows[(k + step) % len(self.rows)]
            try: return self.tf(self.ds[i]["image"].convert("RGB")), y
            except Exception: continue
        raise RuntimeError("too many unreadable photos in a row")

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def square(img):
    """Centre square crop then resize to 224. The app prepares photos exactly this way."""
    w, h = img.size; m = min(w, h)
    return img.crop(((w - m) // 2, (h - m) // 2, (w + m) // 2, (h + m) // 2)).resize((224, 224), Image.BILINEAR)

train_tf = T.Compose([T.RandomResizedCrop(224, scale=(0.45, 1.0)), T.RandomHorizontalFlip(), T.TrivialAugmentWide(),
                      T.ToTensor(), T.Normalize(MEAN, STD), T.RandomErasing(p=0.25)])
test_tf = T.Compose([T.Lambda(square), T.ToTensor(), T.Normalize(MEAN, STD)])

train_ds = ConcatDataset([Photos(my["train"], pick(my["train"], my_names, MAP_MY, PER_CLASS), train_tf),
                          Photos(f101["train"], pick(f101["train"], f_names, MAP_F101, PER_CLASS), train_tf)]
                         + ([Photos(india["train"], pick(india["train"], in_names, MAP_IN, PER_CLASS), train_tf)] if india else [])
                         + ([Files(cm_train, train_tf)] if cm_train else []))
val_ds = ConcatDataset([Photos(my["test"], pick(my["test"], my_names, MAP_MY, 150), test_tf),
                        Photos(f101["validation"], pick(f101["validation"], f_names, MAP_F101, 150), test_tf)]
                       + ([Files(cm_val, test_tf)] if cm_val else []))
po_train, po_val = [], []
for j, lab in enumerate(PO_LABS):
    files = PO_FILES[lab]; random.Random(SEED).shuffle(files); nv = min(20, max(5, len(files) // 5)); y = len(CLASSES) + j   # labels past the dishes = parts only
    po_val += [(fn, y) for fn in files[:nv]]; po_train += [(fn, y) for fn in files[nv:nv + PER_CLASS]]
if po_train:
    train_ds = ConcatDataset([train_ds, Files(po_train, train_tf)]); val_ds = ConcatDataset([val_ds, Files(po_val, test_tf)])
print(len(train_ds), "training photos,", len(val_ds), "test photos")

train_dl = DataLoader(train_ds, BATCH, shuffle=True, num_workers=4, drop_last=True, pin_memory=True)
val_dl = DataLoader(val_ds, BATCH, shuffle=False, num_workers=4, pin_memory=True)

## Train

First only the new final layer is trained, then the whole network at a lower learning rate.

In [ ]:
class TwoAnswers(nn.Module):
    """One network, two answers: which dish (one of CLASSES) and which parts (any of PART_NAMES)."""
    def __init__(self, pretrained=True):
        super().__init__()
        self.body = timm.create_model(ARCH, pretrained=pretrained, num_classes=0)
        self.body.eval()
        with torch.no_grad(): n = self.body(torch.zeros(2, 3, 224, 224)).shape[1]   # size of the features the two answers read
        self.body.train()
        self.dish = nn.Linear(n, len(CLASSES)); self.parts = nn.Linear(n, len(PART_NAMES))
    def forward(self, x):
        f = self.body(x); return self.dish(f), self.parts(f)

model = TwoAnswers().to(DEVICE)
HEAD = {n for n, _ in model.named_parameters() if n.startswith(("dish.", "parts."))}
ema = timm.utils.ModelEmaV3(model, decay=0.999)
dish_loss = nn.CrossEntropyLoss(label_smoothing=0.1, ignore_index=-100)
parts_loss = nn.BCEWithLogitsLoss()
scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")
PTD = PT.to(DEVICE); NC = len(CLASSES)

def losses(out, y):
    d, p = out; yd = torch.where(y < NC, y, torch.full_like(y, -100))
    l = PARTS_WEIGHT * parts_loss(p.float(), PTD[y])          # total = dish loss + 0.5 x parts loss
    if (yd >= 0).any(): l = l + dish_loss(d.float(), yd)        # parts-only photos have no dish answer
    return l

TOP3 = [0.0]; PARTS_SCORE = [0.0]
def evaluate(net=None):
    net = net or ema.module; net.eval(); right = np.zeros(NC); seen = np.zeros(NC); top3 = 0; tp = fp = fn = 0
    with torch.no_grad():
        for x, y in val_dl:
            d, p = net(x.to(DEVICE)); d = d.float().cpu(); p = torch.sigmoid(p.float().cpu())
            truth = PT[y]; guess = (p >= 0.5).float()
            tp += (guess * truth).sum().item(); fp += (guess * (1 - truth)).sum().item(); fn += ((1 - guess) * truth).sum().item()
            m = y < NC
            if m.any():
                dd, yy = d[m], y[m]; pr = dd.argmax(1); t3 = dd.topk(3, 1).indices
                top3 += (t3 == yy[:, None]).any(1).sum().item()
                for a, b in zip(pr.tolist(), yy.tolist()): seen[b] += 1; right[b] += a == b
    TOP3[0] = top3 / max(seen.sum(), 1); PARTS_SCORE[0] = 2 * tp / max(2 * tp + fp + fn, 1)   # F1 over all parts
    return right.sum() / max(seen.sum(), 1), right / np.maximum(seen, 1)

def run(epochs, lr, head_only):
    for n, p in model.named_parameters(): p.requires_grad = (not head_only) or n in HEAD
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=0.02)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_dl))
    for ep in range(epochs):
        model.train(); total = 0
        for x, y in train_dl:
            x, y = x.to(DEVICE), y.to(DEVICE); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, enabled=DEVICE == "cuda"):
                out = model(x)
            loss = losses(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); ema.update(model); total += loss.item()
        acc, _ = evaluate()
        print(f"epoch {ep + 1}/{epochs}  loss {total / len(train_dl):.3f}  dish accuracy {acc:.1%}  top-3 {TOP3[0]:.1%}  parts score {PARTS_SCORE[0]:.1%}")

run(EPOCHS_HEAD, 2e-3, head_only=True)
run(EPOCHS_ALL, 6e-4, head_only=False)

acc, per = evaluate()
print(f"\nFinal dish accuracy: {acc:.1%}   top-3: {TOP3[0]:.1%}   parts score: {PARTS_SCORE[0]:.1%}\n")
for c, a in sorted(zip(CLASSES, per), key=lambda t: t[1]): print(f"{a:6.1%}  {c}")


## Save the two files for the app

In [ ]:
import onnx, onnxruntime as ort

model = ema.module
model.eval().cpu()

class ForApp(nn.Module):
    """What the app runs: dish scores first (the app reads the first output), then parts as 0..1 chances."""
    def __init__(self, m): super().__init__(); self.m = m
    def forward(self, x):
        d, p = self.m(x); return d, torch.sigmoid(p)

app_model = ForApp(model).eval()
dummy = torch.zeros(1, 3, 224, 224)
kw = dict(input_names=["input"], output_names=["logits", "parts"], opset_version=17)
try: torch.onnx.export(app_model, dummy, "food.onnx", dynamo=False, **kw)
except TypeError: torch.onnx.export(app_model, dummy, "food.onnx", **kw)
onnx.checker.check_model(onnx.load("food.onnx"))

import re
json.dump([{"label": re.sub(r"[^a-z0-9]+", "_", c.lower()).strip("_"), "food": c} for c in CLASSES], open("labels.json", "w"), indent=1)
json.dump({"parts": PART_NAMES}, open("parts.json", "w"), indent=1)

# check the saved model gives the same answers as the trained one
sess = ort.InferenceSession("food.onnx")
x, y = next(iter(val_dl))
a, ap = sess.run(None, {"input": x[:1].numpy()})
b, bp = app_model(x[:1]); b, bp = b.detach().numpy(), bp.detach().numpy()
print("largest difference between saved and trained model:", float(max(np.abs(a - b).max(), np.abs(ap - bp).max())))

import os
print(f"food.onnx: {os.path.getsize('food.onnx') / 1e6:.1f} MB.  labels.json: {len(CLASSES)} classes.  parts.json: {len(PART_NAMES)} parts.")
print("Download food.onnx, labels.json and parts.json from the Output panel.")


## Adding your own dishes later

To teach the model a dish that is not in the list:

1. Collect at least 80 to 100 of your own photos of it, taken the way you normally would.
2. Add them as another dataset class, add a line to `MAP` with the exact dish name from the app's food list, and run the notebook again.

Only use photos you took yourself or that are published under a licence that allows reuse.